<a href="https://colab.research.google.com/github/TasminaTasin75/COMP6012-Assignment3/blob/main/assignment3_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

from google.colab import drive
drive.mount("/content/drive")

import os

PROJECT_DIR = "/content/drive/MyDrive/COMP6012_Assignment3"
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")
LOGS_DIR = os.path.join(PROJECT_DIR, "logs")

for directory in [PROJECT_DIR, CHECKPOINT_DIR, RESULTS_DIR, LOGS_DIR]:
    os.makedirs(directory, exist_ok=True)

print("Google Drive connected.")
print("Project directory:", PROJECT_DIR)
print("Checkpoint directory:", CHECKPOINT_DIR)
print("Results directory:", RESULTS_DIR)
print("Logs directory:", LOGS_DIR)

Mounted at /content/drive
Google Drive connected.
Project directory: /content/drive/MyDrive/COMP6012_Assignment3
Checkpoint directory: /content/drive/MyDrive/COMP6012_Assignment3/checkpoints
Results directory: /content/drive/MyDrive/COMP6012_Assignment3/results
Logs directory: /content/drive/MyDrive/COMP6012_Assignment3/logs


In [ ]:
%pip install -q transformers datasets peft accelerate evaluate scikit-learn matplotlib pandas sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00


Inspecting AG News Dataset

In [ ]:
from datasets import load_dataset

ag_news_ds = load_dataset("fancyzhx/ag_news")

print("Loaded AG News Dataset:")
print(ag_news_ds)

print("\nAvailable splits:")
for split_name, split_data in ag_news_ds.items():
    print(f"{split_name}: {len(split_data):,} rows")
    print("Columns:", split_data.column_names)
    print("Features:", split_data.features)

print("\nFirst training example:")
print(ag_news_ds["train"][0])

print("\nClass labels:")
print(ag_news_ds["train"].features["label"].names)

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Loaded AG News Dataset:
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

Available splits:
train: 120,000 rows
Columns: ['text', 'label']
Features: {'text': Value('string'), 'label': ClassLabel(names=['World', 'Sports', 'Business', 'Sci/Tech'])}
test: 7,600 rows
Columns: ['text', 'label']
Features: {'text': Value('string'), 'label': ClassLabel(names=['World', 'Sports', 'Business', 'Sci/Tech'])}

First training example:
{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}

Class labels:
['World', 'Sports', 'Business', 'Sci/Tech']


In [ ]:

from collections import Counter

train_labels = Counter(ag_news_ds["train"]["label"])

print("AG News training examples:", f'{len(ag_news_ds["train"]):,}')
print("Training examples per label:")

label_names = ag_news_ds["train"].features["label"].names
for label_id, count in sorted(train_labels.items()):
    print(f"{label_id} — {label_names[label_id]}: {count:,}")

print("\nValidation/test split sizes:")
for split_name in ag_news_ds.keys():
    if split_name != "train":
        print(f"{split_name}: {len(ag_news_ds[split_name]):,}")

AG News training examples: 120,000
Training examples per label:
0 — World: 30,000
1 — Sports: 30,000
2 — Business: 30,000
3 — Sci/Tech: 30,000

Validation/test split sizes:
test: 7,600


**AG News Classification**
(a) Preparing train, test data

In [ ]:

from datasets import DatasetDict

# Using the full official training and test splits.
ag_news_prepared_ds = DatasetDict({
    "train": ag_news_ds["train"],
    "test": ag_news_ds["test"]
})

print(ag_news_prepared_ds)

print("\nFinal split sizes:")
for split_name, split_data in ag_news_prepared_ds.items():
    print(f"{split_name}: {len(split_data):,}")

print("\nLabel names:")
print(ag_news_prepared_ds["train"].features["label"].names)

print("\nTraining label distribution:")
print(ag_news_prepared_ds["train"].unique("label"))

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

Final split sizes:
train: 120,000
test: 7,600

Label names:
['World', 'Sports', 'Business', 'Sci/Tech']

Training label distribution:
[2, 3, 1, 0]


Loading Qwen for classification  

In [ ]:
import torch
import peft
import transformers
from transformers import AutoModelForSequenceClassification

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

qwen_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    torch_dtype=torch.float16,
    ignore_mismatched_sizes=True,
)

print("Model loaded:", type(qwen_model).__name__)
print("Number of labels:", qwen_model.config.num_labels)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: Qwen2ForSequenceClassification
Number of labels: 4


Moving Qwen to GPU

In [ ]:

import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Stop here and check the Colab runtime.")

qwen_model = qwen_model.to("cuda") #Qwen moving to GPU here
qwen_model.eval()

print("Model device:", next(qwen_model.parameters()).device)
print(f"GPU allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
print(f"GPU reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
print(f"GPU total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

Model device: cuda:0
GPU allocated: 1.86 GB
GPU reserved: 1.94 GB
GPU total: 14.56 GB


Attaching LoRA and inspect trainable parameters

In [ ]:

from peft import LoraConfig, TaskType, get_peft_model
RANK = 8
LORA_ALPHA = 16
DROPOUT = 0.1
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=DROPOUT,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    modules_to_save=["score"],
)

qwen_lora_model = get_peft_model(qwen_model, lora_config)

qwen_lora_model.print_trainable_parameters()


trainable params: 544,256 || all params: 494,580,608 || trainable%: 0.1100


In [ ]:

from transformers import AutoTokenizer
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
qwen_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

MAX_LENGTH = 256

def tokenize_ag_news(examples):
    return qwen_tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

qwen_train_ds = ag_news_prepared_ds["train"].map(
    tokenize_ag_news,
    batched=True,
    desc="Tokenizing full AG News training set",
)

qwen_test_ds = ag_news_prepared_ds["test"].map(
    tokenize_ag_news,
    batched=True,
    desc="Tokenizing full AG News test set",
)

In [ ]:

# Keeping the model inputs and labels only.
qwen_train_model = qwen_train_ds.select_columns(
    ["input_ids", "attention_mask", "label"]
)

qwen_test_model = qwen_test_ds.select_columns(
    ["input_ids", "attention_mask", "label"]
)

# Rename labels for Hugging Face model training.
qwen_train_model = qwen_train_model.rename_column(
    "label", "labels"
)

qwen_test_model = qwen_test_model.rename_column(
    "label", "labels"
)

# Return PyTorch tensors when retrieving examples.
qwen_train_model.set_format("torch")
qwen_test_model.set_format("torch")

In [ ]:
from torch.utils.data import DataLoader
import torch

BATCH_SIZE = 8

train_loader = DataLoader(
    qwen_train_model,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)

test_loader = DataLoader(
    qwen_test_model,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)


Creating a checkpoint folder and save the initial adapter

In [ ]:
import os

CHECKPOINT_DIR = "/content/drive/MyDrive/COMP6012_Assignment3/Qwen_LoRA"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

INITIAL_ADAPTER_DIR = os.path.join(CHECKPOINT_DIR, "initial_adapter")
qwen_lora_model.save_pretrained(INITIAL_ADAPTER_DIR)
qwen_tokenizer.save_pretrained(INITIAL_ADAPTER_DIR)

print("Initial adapter saved to:", INITIAL_ADAPTER_DIR)
print("Files:", os.listdir(INITIAL_ADAPTER_DIR))

Initial adapter saved to: /content/drive/MyDrive/COMP6012_Assignment3/Qwen_LoRA/initial_adapter
Files: ['tokenizer.json', 'README.md', 'chat_template.jinja', 'adapter_config.json', 'adapter_model.safetensors', 'tokenizer_config.json']


In [ ]:
qwen_lora_model.config.pad_token_id = qwen_tokenizer.pad_token_id

print("Model pad token ID:", qwen_lora_model.config.pad_token_id)
print("Tokenizer pad token:", qwen_tokenizer.pad_token)
print("Tokenizer pad token ID:", qwen_tokenizer.pad_token_id)
print("Model pad token ID:", qwen_lora_model.config.pad_token_id)

Model pad token ID: 151643
Tokenizer pad token: <|endoftext|>
Tokenizer pad token ID: 151643
Model pad token ID: 151643


Evaluating the baseline results of the model before training

In [ ]:

import os
import json
import torch
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

qwen_lora_model.eval()

all_predictions = []
all_labels = []

with torch.inference_mode():
    for batch in tqdm(test_loader, desc="Qwen baseline evaluation"):
        input_ids = batch["input_ids"].to("cuda")
        attention_mask = batch["attention_mask"].to("cuda")
        labels = batch["labels"].to("cuda")

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            outputs = qwen_lora_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        predictions = outputs.logits.argmax(dim=-1)

        all_predictions.extend(predictions.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

baseline_accuracy = accuracy_score(all_labels, all_predictions)
baseline_macro_f1 = f1_score(
    all_labels, all_predictions, average="macro", zero_division=0
)

print(f"Baseline accuracy: {baseline_accuracy:.4f}")
print(f"Baseline macro-F1: {baseline_macro_f1:.4f}")
print("\nClassification report:")
print(classification_report(
    all_labels,
    all_predictions,
    target_names=["World", "Sports", "Business", "Sci/Tech"],
    digits=4,
    zero_division=0
))

baseline_results = {
    "model": "Qwen2.5-0.5B-Instruct with untrained classification head",
    "evaluation": "untrained baseline",
    "test_samples": len(all_labels),
    "accuracy": baseline_accuracy,
    "macro_f1": baseline_macro_f1
}

results_path = os.path.join(CHECKPOINT_DIR, "qwen_baseline_results.json")
with open(results_path, "w") as f:
    json.dump(baseline_results, f, indent=2)

print("Baseline metrics saved to:", results_path)

Qwen baseline evaluation:   0%|          | 0/950 [00:00<?, ?it/s]

Baseline accuracy: 0.2384
Baseline macro-F1: 0.1070

Classification report:
              precision    recall  f1-score   support

       World     0.2509    0.9363    0.3958      1900
      Sports     0.0213    0.0042    0.0070      1900
    Business     0.3548    0.0058    0.0114      1900
    Sci/Tech     0.1359    0.0074    0.0140      1900

    accuracy                         0.2384      7600
   macro avg     0.1907    0.2384    0.1070      7600
weighted avg     0.1907    0.2384    0.1070      7600

Baseline metrics saved to: /content/drive/MyDrive/COMP6012_Assignment3/Qwen_LoRA/qwen_baseline_results.json


Fine-Tune Qwen with LoRA on the Full AG News Dataset

In [ ]:

import os
import json
import time
import random
import numpy as np
import torch

from torch.utils.data import DataLoader, Subset
from torch.optim import AdamW
from tqdm.auto import tqdm
from safetensors.torch import load_file
from peft.utils.save_and_load import set_peft_model_state_dict


# Setting Varriables

RUN_NAME = "qwen_lora_run1"
LEARNING_RATE = 1e-5
NUM_EPOCHS = 1
CHECKPOINT_EVERY = 500
MAX_GRAD_NORM = 1.0
SEED = 42

DEVICE = "cuda"

RUN_DIR = os.path.join(CHECKPOINT_DIR, RUN_NAME)
os.makedirs(RUN_DIR, exist_ok=True)

CHECKPOINTS_DIR = os.path.join(RUN_DIR, "checkpoints")
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)

POINTER_FILE = os.path.join(RUN_DIR, "latest_checkpoint.json")

assert torch.cuda.is_available(), "GPU is not available."
assert "qwen_lora_model" in globals()
assert "qwen_train_model_ds" in globals()
assert "train_loader" in globals()

BATCH_SIZE = train_loader.batch_size or 8
STEPS_PER_EPOCH = (len(qwen_train_model) + BATCH_SIZE - 1) // BATCH_SIZE
TOTAL_STEPS = STEPS_PER_EPOCH * NUM_EPOCHS

print("GPU:", torch.cuda.get_device_name(0))
print("Training examples:", len(qwen_train_model))
print("Batch size:", BATCH_SIZE)
print("Steps per epoch:", STEPS_PER_EPOCH)
print("Learning rate:", LEARNING_RATE)
print("Checkpoint folder:", RUN_DIR)


# Preparing the model and optimizer

qwen_lora_model.to(DEVICE)

for p in qwen_lora_model.parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.float()

trainable_params = [
    p for p in qwen_lora_model.parameters() if p.requires_grad
]

optimizer = AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    weight_decay=0.01
)

scaler = torch.amp.GradScaler("cuda")


#  Restoring progress if a checkpoint exists

global_step = 0
start_epoch = 0
batches_done = 0
loss_history = []
elapsed_before = 0.0
resumed = False

if os.path.exists(POINTER_FILE):
    with open(POINTER_FILE) as f:
        pointer = json.load(f)

    saved_dir = pointer["checkpoint_dir"]
    state_file = os.path.join(saved_dir, "training_state.pt")
    adapter_file = os.path.join(saved_dir, "adapter_model.safetensors")

    if os.path.exists(state_file) and os.path.exists(adapter_file):
        state = torch.load(
            state_file, map_location="cpu", weights_only=False
        )

        adapter_weights = load_file(adapter_file)
        set_peft_model_state_dict(
            qwen_lora_model,
            adapter_weights,
            adapter_name="default"
        )

        optimizer.load_state_dict(state["optimizer"])
        scaler.load_state_dict(state["scaler"])

        global_step = state["global_step"]
        start_epoch = state["next_epoch"]
        batches_done = state["batches_done"]
        loss_history = state["loss_history"]
        elapsed_before = state["elapsed_seconds"]

        # Restore random states for reproducibility.
        random.setstate(state["python_rng"])
        np.random.set_state(state["numpy_rng"])
        torch.set_rng_state(state["torch_rng"])
        torch.cuda.set_rng_state_all(state["cuda_rng"])

        resumed = True
        print("Resuming from step:", global_step)
        print("Epoch index:", start_epoch)
        print("Batches already completed in this epoch:", batches_done)
    else:
        print("Checkpoint is incomplete; starting from the initial model.")

if not resumed:
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)


# Saving a recoverable checkpoint

session_start = time.time()

def save_checkpoint(next_epoch, next_batch):
    checkpoint_name = (
        f"step_{global_step:06d}_"
        f"epoch_{next_epoch:02d}_batch_{next_batch:05d}"
    )
    checkpoint_dir = os.path.join(CHECKPOINTS_DIR, checkpoint_name)
    os.makedirs(checkpoint_dir, exist_ok=True)

    # Save the LoRA adapter and classification head.
    qwen_lora_model.save_pretrained(checkpoint_dir)
    qwen_tokenizer.save_pretrained(checkpoint_dir)

    state = {
        "global_step": global_step,
        "next_epoch": next_epoch,
        "batches_done": next_batch,
        "loss_history": loss_history,
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "elapsed_seconds": elapsed_before + time.time() - session_start,
        "python_rng": random.getstate(),
        "numpy_rng": np.random.get_state(),
        "torch_rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state_all(),
    }

    torch.save(state, os.path.join(checkpoint_dir, "training_state.pt"))

    # Update the pointer only after saving the checkpoint files.
    temp_file = POINTER_FILE + ".tmp"
    with open(temp_file, "w") as f:
        json.dump({"checkpoint_dir": checkpoint_dir}, f)
    os.replace(temp_file, POINTER_FILE)

    print(f"\nCheckpoint saved: step {global_step}")


# Model Training

qwen_lora_model.train()

for epoch in range(start_epoch, NUM_EPOCHS):

    # Generate the same shuffled order when resuming this epoch.
    generator = torch.Generator()
    generator.manual_seed(SEED + epoch)

    shuffled_indices = torch.randperm(
        len(qwen_train_model),
        generator=generator
    ).tolist()

    ordered_dataset = Subset(qwen_train_model, shuffled_indices)

    # Reuse the original collator so batch formatting stays consistent.
    epoch_loader = DataLoader(
        ordered_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=train_loader.collate_fn
    )

    skip = batches_done if epoch == start_epoch else 0

    progress = tqdm(
        enumerate(epoch_loader, start=1),
        total=len(epoch_loader),
        desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}"
    )

    for batch_number, batch in progress:

        # Skip batches already processed before the interruption.
        if batch_number <= skip:
            continue

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            outputs = qwen_lora_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss

        if not torch.isfinite(loss):
            print("\nNon-finite loss. Saving the previous recovery point.")
            save_checkpoint(epoch, batch_number - 1)
            raise RuntimeError("Training stopped because loss was non-finite.")

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        # Clip large gradients to improve training stability.
        torch.nn.utils.clip_grad_norm_(
            trainable_params,
            MAX_GRAD_NORM
        )

        # GradScaler skips the optimizer update if it detects overflow.
        scaler.step(optimizer)
        scaler.update()

        global_step += 1
        loss_value = loss.detach().float().item()
        loss_history.append(loss_value)

        recent_losses = loss_history[-100:]
        progress.set_postfix(
            loss=f"{loss_value:.4f}",
            avg100=f"{np.mean(recent_losses):.4f}"
        )

        if global_step % 100 == 0:
            progress.write(
                f"Step {global_step}/{TOTAL_STEPS} "
                f"| loss={loss_value:.4f}"
            )

        if global_step % CHECKPOINT_EVERY == 0:
            save_checkpoint(epoch, batch_number)

    # Save progress at the end of the epoch.
    save_checkpoint(epoch + 1, 0)
    batches_done = 0


# Saving the final summary
qwen_lora_model.eval()

summary = {
    "run_name": RUN_NAME,
    "epochs_requested": NUM_EPOCHS,
    "optimizer_steps_completed": global_step,
    "total_steps_expected": TOTAL_STEPS,
    "training_complete": global_step >= TOTAL_STEPS,
    "training_samples": len(qwen_train_model),
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "first_100_step_mean_loss": (
        float(np.mean(loss_history[:100])) if loss_history else None
    ),
    "last_100_step_mean_loss": (
        float(np.mean(loss_history[-100:])) if loss_history else None
    ),
    "total_elapsed_seconds": elapsed_before + time.time() - session_start,
}

with open(os.path.join(RUN_DIR, "training_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("\nTraining summary:")
print(json.dumps(summary, indent=2))

GPU: Tesla T4
Training examples: 120000
Batch size: 8
Steps per epoch: 15000
Learning rate: 1e-05
Checkpoint folder: /content/drive/MyDrive/COMP6012_Assignment3/Qwen_LoRA/qwen_lora_run1
Resuming from step: 15000
Epoch index: 1
Batches already completed in this epoch: 0

Training summary:
{
  "run_name": "qwen_lora_run1",
  "epochs_requested": 1,
  "optimizer_steps_completed": 15000,
  "total_steps_expected": 15000,
  "training_complete": true,
  "training_samples": 120000,
  "batch_size": 8,
  "learning_rate": 1e-05,
  "first_100_step_mean_loss": 5.198216907978058,
  "last_100_step_mean_loss": 0.3413862507045269,
  "total_elapsed_seconds": 4596.57471370697
}


Evaluating the trained Qwen model

In [ ]:

import os
import json
import numpy as np
import torch
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from tqdm.auto import tqdm

# Use the trained model for evaluation
qwen_lora_model.eval()

all_predictions = []
all_labels = []

total_test_loss = 0.0
total_examples = 0

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating Qwen"):

        # Move data to GPU
        batch = {
            key: value.to("cuda")
            for key, value in batch.items()
        }

        # Get predictions
        with torch.autocast("cuda", dtype=torch.float16):
            outputs = qwen_lora_model(**batch)

        # Calculate test loss
        batch_size = batch["labels"].size(0)
        total_test_loss += outputs.loss.item() * batch_size
        total_examples += batch_size

        # Select the category with the highest score
        predictions = torch.argmax(outputs.logits, dim=1)

        all_predictions.extend(predictions.cpu().tolist())
        all_labels.extend(batch["labels"].cpu().tolist())

# Calculate evaluation metrics
test_accuracy = accuracy_score(all_labels, all_predictions)
test_macro_f1 = f1_score(
    all_labels, all_predictions, average="macro"
)
average_test_loss = total_test_loss / total_examples

class_names = ["World", "Sports", "Business", "Sci/Tech"]

print("\n===== QWEN TEST RESULTS =====")
print(f"Test examples: {total_examples}")
print(f"Test loss: {average_test_loss:.4f}")
print(f"Accuracy: {test_accuracy:.4f} ({test_accuracy * 100:.2f}%)")
print(f"Macro-F1: {test_macro_f1:.4f}")

print("\n===== CLASSIFICATION REPORT =====")
print(classification_report(
    all_labels,
    all_predictions,
    labels=[0, 1, 2, 3],
    target_names=class_names,
    digits=4,
    zero_division=0
))

print("===== CONFUSION MATRIX =====")
print(confusion_matrix(
    all_labels,
    all_predictions,
    labels=[0, 1, 2, 3]
))

# Save evaluation results to Google Drive
results_dir = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "Qwen_LoRA/qwen_lora_run1"
)
os.makedirs(results_dir, exist_ok=True)

results = {
    "model": "Qwen2.5-0.5B-Instruct with LoRA",
    "test_examples": int(total_examples),
    "test_loss": float(average_test_loss),
    "accuracy": float(test_accuracy),
    "macro_f1": float(test_macro_f1),
    "classification_report": classification_report(
        all_labels,
        all_predictions,
        labels=[0, 1, 2, 3],
        target_names=class_names,
        output_dict=True,
        zero_division=0
    ),
    "confusion_matrix": confusion_matrix(
        all_labels,
        all_predictions,
        labels=[0, 1, 2, 3]
    ).tolist()
}

results_path = os.path.join(
    results_dir, "qwen_test_results.json"
)

with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"\nResults saved to: {results_path}")

Evaluating Qwen:   0%|          | 0/950 [00:00<?, ?it/s]


===== QWEN TEST RESULTS =====
Test examples: 7600
Test loss: 0.2940
Accuracy: 0.9251 (92.51%)
Macro-F1: 0.9251

===== CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

       World     0.9490    0.9211    0.9348      1900
      Sports     0.9740    0.9863    0.9801      1900
    Business     0.8895    0.8858    0.8877      1900
    Sci/Tech     0.8887    0.9074    0.8979      1900

    accuracy                         0.9251      7600
   macro avg     0.9253    0.9251    0.9251      7600
weighted avg     0.9253    0.9251    0.9251      7600

===== CONFUSION MATRIX =====
[[1750   33   69   48]
 [  12 1874    9    5]
 [  44   10 1683  163]
 [  38    7  131 1724]]

Results saved to: /content/drive/MyDrive/COMP6012_Assignment3/Qwen_LoRA/qwen_lora_run1/qwen_test_results.json


Trial 2 Qwen: Hyperparameter Change - LR 2e-5, batch 8, 1 epoch

In [ ]:

import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

BASE_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "Qwen_LoRA"
)
INITIAL_ADAPTER_DIR = os.path.join(BASE_DIR, "initial_adapter")

assert torch.cuda.is_available(), "GPU is not available."
assert os.path.isdir(INITIAL_ADAPTER_DIR), (
    "Initial adapter folder was not found."
)

trial2_tokenizer = AutoTokenizer.from_pretrained(
    INITIAL_ADAPTER_DIR
)

trial2_base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=4,
    torch_dtype=torch.float32,
    device_map="cpu"
)

trial2_base_model.config.pad_token_id = trial2_tokenizer.pad_token_id

trial2_model = PeftModel.from_pretrained(
    trial2_base_model,
    INITIAL_ADAPTER_DIR,
    is_trainable=True
)

trial2_model.config.pad_token_id = trial2_tokenizer.pad_token_id
trial2_model.to("cuda")

print("Fresh Trial 2 model loaded.")
print("Device:", next(trial2_model.parameters()).device)
print(
    "Trainable parameters:",
    f"{sum(p.numel() for p in trial2_model.parameters() if p.requires_grad):,}"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fresh Trial 2 model loaded.
Device: cuda:0
Trainable parameters: 544,256


Trail 2 : Qwen Model training

In [ ]:

import os
import json
import time
import random
import numpy as np
import torch

from torch.utils.data import DataLoader, Subset
from torch.optim import AdamW
from tqdm.auto import tqdm
from safetensors.torch import load_file
from peft.utils.save_and_load import set_peft_model_state_dict


# Setting Variables

RUN_NAME = "qwen_lora_trial2"
LEARNING_RATE = 2e-5
NUM_EPOCHS = 1
CHECKPOINT_EVERY = 500
MAX_GRAD_NORM = 1.0
SEED = 42

DEVICE = "cuda"

RUN_DIR = os.path.join(CHECKPOINT_DIR, RUN_NAME)
os.makedirs(RUN_DIR, exist_ok=True)

CHECKPOINTS_DIR = os.path.join(RUN_DIR, "checkpoints")
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)

POINTER_FILE = os.path.join(RUN_DIR, "latest_checkpoint.json")

assert torch.cuda.is_available(), "GPU is not available."
assert "trial2_model" in globals()
assert "qwen_train_model_ds" in globals()
assert "train_loader" in globals()
assert len(qwen_train_model) == 120000

BATCH_SIZE = train_loader.batch_size or 8
STEPS_PER_EPOCH = (
    len(qwen_train_model) + BATCH_SIZE - 1
) // BATCH_SIZE
TOTAL_STEPS = STEPS_PER_EPOCH * NUM_EPOCHS

print("GPU:", torch.cuda.get_device_name(0))
print("Training examples:", len(qwen_train_model))
print("Batch size:", BATCH_SIZE)
print("Steps per epoch:", STEPS_PER_EPOCH)
print("Learning rate:", LEARNING_RATE)
print("Checkpoint folder:", RUN_DIR)


# Preparing the model and optimizer

trial2_model.to(DEVICE)

for p in trial2_model.parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.float()

trainable_params = [
    p for p in trial2_model.parameters() if p.requires_grad
]

optimizer = AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    weight_decay=0.01
)

scaler = torch.amp.GradScaler("cuda")


# Restoring progress if a checkpoint exists

global_step = 0
start_epoch = 0
batches_done = 0
loss_history = []
elapsed_before = 0.0
resumed = False

if os.path.exists(POINTER_FILE):
    with open(POINTER_FILE) as f:
        pointer = json.load(f)

    saved_dir = pointer["checkpoint_dir"]
    state_file = os.path.join(saved_dir, "training_state.pt")
    adapter_file = os.path.join(saved_dir, "adapter_model.safetensors")

    if os.path.exists(state_file) and os.path.exists(adapter_file):
        state = torch.load(
            state_file, map_location="cpu", weights_only=False
        )

        adapter_weights = load_file(adapter_file)
        set_peft_model_state_dict(
            trial2_model,
            adapter_weights,
            adapter_name="default"
        )

        optimizer.load_state_dict(state["optimizer"])
        scaler.load_state_dict(state["scaler"])

        global_step = state["global_step"]
        start_epoch = state["next_epoch"]
        batches_done = state["batches_done"]
        loss_history = state["loss_history"]
        elapsed_before = state["elapsed_seconds"]

        random.setstate(state["python_rng"])
        np.random.set_state(state["numpy_rng"])
        torch.set_rng_state(state["torch_rng"])
        torch.cuda.set_rng_state_all(state["cuda_rng"])

        resumed = True
        print("Resuming from step:", global_step)
        print("Epoch index:", start_epoch)
        print("Batches already completed in this epoch:", batches_done)
    else:
        raise RuntimeError(
            "Trial 2 checkpoint is incomplete. Inspect it before proceeding."
        )

if not resumed:
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)


# Saving a recoverable checkpoint

session_start = time.time()

def save_checkpoint(next_epoch, next_batch):
    checkpoint_name = (
        f"step_{global_step:06d}_"
        f"epoch_{next_epoch:02d}_batch_{next_batch:05d}"
    )
    checkpoint_dir = os.path.join(CHECKPOINTS_DIR, checkpoint_name)
    os.makedirs(checkpoint_dir, exist_ok=True)

    # Save the LoRA adapter and classification head.
    trial2_model.save_pretrained(checkpoint_dir)
    trial2_tokenizer.save_pretrained(checkpoint_dir)

    state = {
        "global_step": global_step,
        "next_epoch": next_epoch,
        "batches_done": next_batch,
        "loss_history": loss_history,
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "elapsed_seconds": elapsed_before + time.time() - session_start,
        "python_rng": random.getstate(),
        "numpy_rng": np.random.get_state(),
        "torch_rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state_all(),
    }

    torch.save(state, os.path.join(checkpoint_dir, "training_state.pt"))

    # Update the pointer only after saving the checkpoint files.
    temp_file = POINTER_FILE + ".tmp"
    with open(temp_file, "w") as f:
        json.dump({"checkpoint_dir": checkpoint_dir}, f)
    os.replace(temp_file, POINTER_FILE)

    print(f"\nCheckpoint saved: step {global_step}")


# Model Training

trial2_model.train()

for epoch in range(start_epoch, NUM_EPOCHS):

    generator = torch.Generator()
    generator.manual_seed(SEED + epoch)

    shuffled_indices = torch.randperm(
        len(qwen_train_model),
        generator=generator
    ).tolist()

    ordered_dataset = Subset(qwen_train_model, shuffled_indices)

    epoch_loader = DataLoader(
        ordered_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=train_loader.collate_fn
    )

    skip = batches_done if epoch == start_epoch else 0

    progress = tqdm(
        enumerate(epoch_loader, start=1),
        total=len(epoch_loader),
        desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}"
    )

    for batch_number, batch in progress:

        if batch_number <= skip:
            continue

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            outputs = trial2_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss

        if not torch.isfinite(loss):
            print("\nNon-finite loss. Saving the previous recovery point.")
            save_checkpoint(epoch, batch_number - 1)
            raise RuntimeError("Training stopped because loss was non-finite.")

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            trainable_params,
            MAX_GRAD_NORM
        )

        scaler.step(optimizer)
        scaler.update()

        global_step += 1
        loss_value = loss.detach().float().item()
        loss_history.append(loss_value)

        recent_losses = loss_history[-100:]
        progress.set_postfix(
            loss=f"{loss_value:.4f}",
            avg100=f"{np.mean(recent_losses):.4f}"
        )

        if global_step % 100 == 0:
            progress.write(
                f"Step {global_step}/{TOTAL_STEPS} "
                f"| loss={loss_value:.4f}"
            )

        if global_step % CHECKPOINT_EVERY == 0:
            save_checkpoint(epoch, batch_number)

    save_checkpoint(epoch + 1, 0)
    batches_done = 0


# Saving the final summary

trial2_model.eval()

summary = {
    "run_name": RUN_NAME,
    "epochs_requested": NUM_EPOCHS,
    "optimizer_steps_completed": global_step,
    "total_steps_expected": TOTAL_STEPS,
    "training_complete": global_step >= TOTAL_STEPS,
    "training_samples": len(qwen_train_model),
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "first_100_step_mean_loss": (
        float(np.mean(loss_history[:100])) if loss_history else None
    ),
    "last_100_step_mean_loss": (
        float(np.mean(loss_history[-100:])) if loss_history else None
    ),
    "total_elapsed_seconds": elapsed_before + time.time() - session_start,
}

with open(os.path.join(RUN_DIR, "training_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("\nTraining summary:")
print(json.dumps(summary, indent=2))

GPU: Tesla T4
Training examples: 120000
Batch size: 8
Steps per epoch: 15000
Learning rate: 2e-05
Checkpoint folder: /content/drive/MyDrive/COMP6012_Assignment3/Qwen_LoRA/qwen_lora_trial2_lr2e5/checkpoints/qwen_lora_trial2


Epoch 1/1:   0%|          | 0/15000 [00:00<?, ?it/s]

Step 100/15000 | loss=2.3626
Step 200/15000 | loss=1.5757
Step 300/15000 | loss=1.4541
Step 400/15000 | loss=1.1881
Step 500/15000 | loss=1.2990

Checkpoint saved: step 500
Step 600/15000 | loss=0.4401
Step 700/15000 | loss=0.4780
Step 800/15000 | loss=0.2870
Step 900/15000 | loss=0.1743
Step 1000/15000 | loss=0.0607

Checkpoint saved: step 1000
Step 1100/15000 | loss=0.1069
Step 1200/15000 | loss=0.5587
Step 1300/15000 | loss=0.1236
Step 1400/15000 | loss=0.1015
Step 1500/15000 | loss=0.5780

Checkpoint saved: step 1500
Step 1600/15000 | loss=0.0265
Step 1700/15000 | loss=0.6494
Step 1800/15000 | loss=0.0037
Step 1900/15000 | loss=0.7300
Step 2000/15000 | loss=0.0212

Checkpoint saved: step 2000
Step 2100/15000 | loss=0.0969
Step 2200/15000 | loss=0.6527
Step 2300/15000 | loss=0.0272
Step 2400/15000 | loss=0.7935
Step 2500/15000 | loss=0.1563

Checkpoint saved: step 2500
Step 2600/15000 | loss=0.0047
Step 2700/15000 | loss=0.5675
Step 2800/15000 | loss=1.1264
Step 2900/15000 | loss=0.

Trial 2 Evaluation

In [ ]:

import os
import json
import numpy as np
import torch
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from tqdm.auto import tqdm

# Use the trained Trial 2 model for evaluation
trial2_model.eval()

all_predictions = []
all_labels = []

total_test_loss = 0.0
total_examples = 0

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating Qwen Trial 2"):

        # Move data to GPU
        batch = {
            key: value.to("cuda")
            for key, value in batch.items()
        }

        # Get predictions
        with torch.autocast("cuda", dtype=torch.float16):
            outputs = trial2_model(**batch)

        # Calculate test loss
        batch_size = batch["labels"].size(0)
        total_test_loss += outputs.loss.item() * batch_size
        total_examples += batch_size

        # Select the category with the highest score
        predictions = torch.argmax(outputs.logits, dim=1)

        all_predictions.extend(predictions.cpu().tolist())
        all_labels.extend(batch["labels"].cpu().tolist())

# Calculate evaluation metrics
test_accuracy = accuracy_score(all_labels, all_predictions)
test_macro_f1 = f1_score(
    all_labels, all_predictions, average="macro"
)
average_test_loss = total_test_loss / total_examples

class_names = ["World", "Sports", "Business", "Sci/Tech"]

print("\n===== QWEN TRIAL 2 TEST RESULTS =====")
print(f"Test examples: {total_examples}")
print(f"Test loss: {average_test_loss:.4f}")
print(f"Accuracy: {test_accuracy:.4f} ({test_accuracy * 100:.2f}%)")
print(f"Macro-F1: {test_macro_f1:.4f}")

print("\n===== CLASSIFICATION REPORT =====")
print(classification_report(
    all_labels,
    all_predictions,
    labels=[0, 1, 2, 3],
    target_names=class_names,
    digits=4,
    zero_division=0
))

print("===== CONFUSION MATRIX =====")
print(confusion_matrix(
    all_labels,
    all_predictions,
    labels=[0, 1, 2, 3]
))

# Save evaluation results to Google Drive
results_dir = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "Qwen_LoRA/qwen_lora_trial2_lr2e5"
)
os.makedirs(results_dir, exist_ok=True)

results = {
    "trial": 2,
    "model": "Qwen2.5-0.5B-Instruct with LoRA",
    "learning_rate": 2e-5,
    "batch_size": 8,
    "epochs": 1,
    "training_examples": 120000,
    "test_examples": int(total_examples),
    "test_loss": float(average_test_loss),
    "accuracy": float(test_accuracy),
    "macro_f1": float(test_macro_f1),
    "classification_report": classification_report(
        all_labels,
        all_predictions,
        labels=[0, 1, 2, 3],
        target_names=class_names,
        output_dict=True,
        zero_division=0
    ),
    "confusion_matrix": confusion_matrix(
        all_labels,
        all_predictions,
        labels=[0, 1, 2, 3]
    ).tolist()
}

results_path = os.path.join(
    results_dir, "qwen_test_results.json"
)

with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"\nResults saved to: {results_path}")

Evaluating Qwen Trial 2:   0%|          | 0/950 [00:00<?, ?it/s]


===== QWEN TRIAL 2 TEST RESULTS =====
Test examples: 7600
Test loss: 0.2858
Accuracy: 0.9300 (93.00%)
Macro-F1: 0.9301

===== CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

       World     0.9568    0.9205    0.9383      1900
      Sports     0.9776    0.9874    0.9825      1900
    Business     0.8833    0.9079    0.8954      1900
    Sci/Tech     0.9042    0.9042    0.9042      1900

    accuracy                         0.9300      7600
   macro avg     0.9305    0.9300    0.9301      7600
weighted avg     0.9305    0.9300    0.9301      7600

===== CONFUSION MATRIX =====
[[1749   26   70   55]
 [  11 1876   10    3]
 [  42    9 1725  124]
 [  26    8  148 1718]]

Results saved to: /content/drive/MyDrive/COMP6012_Assignment3/Qwen_LoRA/qwen_lora_trial2_lr2e5/qwen_test_results.json


TRIAL 3: Qwen LoRA Training
Learning rate: 1e-5 | Batch size: 8 | Epochs: 2

In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

BASE_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "Qwen_LoRA"
)
INITIAL_ADAPTER_DIR = os.path.join(BASE_DIR, "initial_adapter")

assert torch.cuda.is_available(), "GPU is not available."
assert os.path.isdir(INITIAL_ADAPTER_DIR), (
    "Initial adapter folder was not found."
)

trial3_tokenizer = AutoTokenizer.from_pretrained(
    INITIAL_ADAPTER_DIR
)

trial3_base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=4,
    torch_dtype=torch.float32,
    device_map="cpu"
)

trial3_base_model.config.pad_token_id = trial3_tokenizer.pad_token_id

trial3_model = PeftModel.from_pretrained(
    trial3_base_model,
    INITIAL_ADAPTER_DIR,
    is_trainable=True
)

trial3_model.config.pad_token_id = trial3_tokenizer.pad_token_id
trial3_model.to("cuda")

print("Fresh Trial 2 model loaded.")
print("Device:", next(trial3_model.parameters()).device)
print(
    "Trainable parameters:",
    f"{sum(p.numel() for p in trial3_model.parameters() if p.requires_grad):,}"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fresh Trial 2 model loaded.
Device: cuda:0
Trainable parameters: 544,256


Trail 3 Qwen Training

In [ ]:


import os
import json
import time
import random
import numpy as np
import torch

from torch.utils.data import DataLoader, Subset
from torch.optim import AdamW
from tqdm.auto import tqdm
from safetensors.torch import load_file
from peft import set_peft_model_state_dict


# Settings


RUN_NAME = "qwen_lora_trial3"
LEARNING_RATE = 1e-5
NUM_EPOCHS = 2
CHECKPOINT_EVERY = 500
MAX_GRAD_NORM = 1.0
SEED = 42
DEVICE = "cuda"

CHECKPOINT_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/Qwen_LoRA"
)

RUN_DIR = os.path.join(CHECKPOINT_DIR, RUN_NAME)
CHECKPOINTS_DIR = os.path.join(RUN_DIR, "checkpoints")
POINTER_FILE = os.path.join(RUN_DIR, "latest_checkpoint.json")
SUMMARY_FILE = os.path.join(RUN_DIR, "training_summary.json")

os.makedirs(CHECKPOINTS_DIR, exist_ok=True)


# Verifying Setup


assert torch.cuda.is_available(), "GPU is not available."
assert "trial3_model" in globals(), "trial3_model is missing."
assert "trial3_tokenizer" in globals(), "trial3_tokenizer is missing."
assert "qwen_train_model" in globals(), "Training dataset is missing."
assert "qwen_train_model_ds" in globals(), "Tokenized training dataset is missing."
assert "train_loader" in globals(), "train_loader is missing."

assert len(qwen_train_model) == 120000, (
    f"Expected 120000 training samples, got {len(qwen_train_model)}."
)

print("GPU:", torch.cuda.get_device_name(0))
print("Training examples:", len(qwen_train_model))


# Calculating Training Steps

BATCH_SIZE = train_loader.batch_size or 8
STEPS_PER_EPOCH = (
    len(qwen_train_model) + BATCH_SIZE - 1
) // BATCH_SIZE

TOTAL_STEPS_EXPECTED = STEPS_PER_EPOCH * NUM_EPOCHS

print("Batch size:", BATCH_SIZE)
print("Steps per epoch:", STEPS_PER_EPOCH)
print("Total expected optimizer steps:", TOTAL_STEPS_EXPECTED)
print("Learning rate:", LEARNING_RATE)
print("Epochs:", NUM_EPOCHS)
print("Checkpoint folder:", CHECKPOINTS_DIR)


# Model and Optimizer


trial3_model.to(DEVICE)

for param in trial3_model.parameters():
    if param.requires_grad:
        param.data = param.data.float()

trainable_params = [
    param for param in trial3_model.parameters()
    if param.requires_grad
]

assert len(trainable_params) > 0, "No trainable parameters found."

optimizer = AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    weight_decay=0.01
)

scaler = torch.amp.GradScaler("cuda")


# Restore Previous Progress, If Available


global_step = 0
start_epoch = 0
batches_done = 0
loss_history = []
elapsed_before = 0.0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if os.path.isfile(POINTER_FILE):
    with open(POINTER_FILE) as f:
        pointer = json.load(f)

    checkpoint_path = pointer["checkpoint_dir"]
    state_path = os.path.join(
        checkpoint_path, "training_state.pt"
    )
    adapter_path = os.path.join(
        checkpoint_path, "adapter_model.safetensors"
    )

    assert os.path.isfile(state_path), (
        f"Training state is missing: {state_path}"
    )
    assert os.path.isfile(adapter_path), (
        f"Adapter weights are missing: {adapter_path}"
    )

    state = torch.load(
        state_path,
        map_location="cpu",
        weights_only=False
    )

    set_peft_model_state_dict(
        trial3_model,
        load_file(adapter_path)
    )

    optimizer.load_state_dict(state["optimizer_state_dict"])
    scaler.load_state_dict(state["scaler_state_dict"])

    global_step = state["global_step"]
    start_epoch = state["epoch_index"]
    batches_done = state["batches_done"]
    loss_history = state["loss_history"]
    elapsed_before = state["elapsed_seconds"]

    random.setstate(state["python_rng_state"])
    np.random.set_state(state["numpy_rng_state"])
    torch.set_rng_state(state["torch_rng_state"])
    torch.cuda.set_rng_state_all(state["cuda_rng_state_all"])

    print("Resuming from:", checkpoint_path)
    print("Restored optimizer steps:", global_step)
    print("Restored epoch index:", start_epoch + 1)

else:
    print("Starting Trial 3 from its initialized model.")


# Checkpoint Function


def save_checkpoint(epoch_index, completed_batches, tag="periodic"):
    checkpoint_name = (
        f"{tag}_step_{global_step:06d}"
        f"_epoch_{epoch_index + 1:02d}"
    )

    checkpoint_path = os.path.join(
        CHECKPOINTS_DIR, checkpoint_name
    )
    os.makedirs(checkpoint_path, exist_ok=True)

    trial3_model.save_pretrained(checkpoint_path)
    trial3_tokenizer.save_pretrained(checkpoint_path)

    state = {
        "global_step": global_step,
        "epoch_index": epoch_index,
        "batches_done": completed_batches,
        "loss_history": loss_history,
        "elapsed_seconds": elapsed_before + (
            time.time() - training_start_time
        ),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "python_rng_state": random.getstate(),
        "numpy_rng_state": np.random.get_state(),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state_all": torch.cuda.get_rng_state_all()
    }

    torch.save(
        state,
        os.path.join(checkpoint_path, "training_state.pt")
    )

    pointer = {
        "global_step": global_step,
        "checkpoint_dir": checkpoint_path,
        "epoch_index": epoch_index,
        "batches_done": completed_batches
    }

    temp_pointer = POINTER_FILE + ".tmp"

    with open(temp_pointer, "w") as f:
        json.dump(pointer, f, indent=2)

    os.replace(temp_pointer, POINTER_FILE)


# Training Loop


training_start_time = time.time()
trial3_model.train()

for epoch_index in range(start_epoch, NUM_EPOCHS):

    # Deterministic shuffle for each epoch
    generator = torch.Generator()
    generator.manual_seed(SEED + epoch_index)

    indices = torch.randperm(
        len(qwen_train_model),
        generator=generator
    ).tolist()

    shuffled_dataset = Subset(qwen_train_model, indices)

    epoch_loader = DataLoader(
        shuffled_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=train_loader.collate_fn
    )

    skip_batches = (
        batches_done if epoch_index == start_epoch else 0
    )

    progress = tqdm(
        enumerate(epoch_loader),
        total=len(epoch_loader),
        desc=f"Trial 3 | Epoch {epoch_index + 1}/{NUM_EPOCHS}"
    )

    for batch_index, batch in progress:

        if batch_index < skip_batches:
            continue

        batch = {
            key: value.to(DEVICE)
            for key, value in batch.items()
        }

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast("cuda", dtype=torch.float16):
            outputs = trial3_model(**batch)
            loss = outputs.loss

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss at optimizer step {global_step + 1}"
            )

        loss_history.append(float(loss.detach().cpu()))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            trainable_params,
            MAX_GRAD_NORM
        )

        scaler.step(optimizer)
        scaler.update()

        global_step += 1

        progress.set_postfix(
            loss=f"{loss.item():.4f}",
            step=f"{global_step}/{TOTAL_STEPS_EXPECTED}"
        )

        completed_batches = batch_index + 1

        if global_step % CHECKPOINT_EVERY == 0:
            save_checkpoint(
                epoch_index,
                completed_batches
            )

    batches_done = 0


# Final Checkpoint and Training Summary


save_checkpoint(
    NUM_EPOCHS - 1,
    STEPS_PER_EPOCH,
    tag="final"
)

total_elapsed = elapsed_before + (
    time.time() - training_start_time
)

training_complete = global_step >= TOTAL_STEPS_EXPECTED

summary = {
    "run_name": RUN_NAME,
    "trial": 3,
    "epochs_requested": NUM_EPOCHS,
    "optimizer_steps_completed": global_step,
    "total_steps_expected": TOTAL_STEPS_EXPECTED,
    "training_complete": training_complete,
    "training_samples": len(qwen_train_model),
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "first_100_step_mean_loss": (
        float(np.mean(loss_history[:100]))
        if loss_history else None
    ),
    "last_100_step_mean_loss": (
        float(np.mean(loss_history[-100:]))
        if loss_history else None
    ),
    "total_elapsed_seconds": total_elapsed
}

with open(SUMMARY_FILE, "w") as f:
    json.dump(summary, f, indent=2)

print("\n===== TRIAL 3 TRAINING SUMMARY =====")
print(json.dumps(summary, indent=2))
print("\nSummary saved to:", SUMMARY_FILE)
print("Latest checkpoint pointer:", POINTER_FILE)

GPU: Tesla T4
Training examples: 120000
Batch size: 8
Steps per epoch: 15000
Total expected optimizer steps: 30000
Learning rate: 1e-05
Epochs: 2
Checkpoint folder: /content/drive/MyDrive/COMP6012_Assignment3/Qwen_LoRA/qwen_lora_trial3/checkpoints
Starting Trial 3 from its initialized model.


Trial 3 | Epoch 1/2:   0%|          | 0/15000 [00:00<?, ?it/s]

Trial 3 | Epoch 2/2:   0%|          | 0/15000 [00:00<?, ?it/s]


===== TRIAL 3 TRAINING SUMMARY =====
{
  "run_name": "qwen_lora_trial3",
  "trial": 3,
  "epochs_requested": 2,
  "optimizer_steps_completed": 30000,
  "total_steps_expected": 30000,
  "training_complete": true,
  "training_samples": 120000,
  "batch_size": 8,
  "learning_rate": 1e-05,
  "first_100_step_mean_loss": 5.323847124576568,
  "last_100_step_mean_loss": 0.25682972818613053,
  "total_elapsed_seconds": 9518.905596017838
}

Summary saved to: /content/drive/MyDrive/COMP6012_Assignment3/Qwen_LoRA/qwen_lora_trial3/training_summary.json
Latest checkpoint pointer: /content/drive/MyDrive/COMP6012_Assignment3/Qwen_LoRA/qwen_lora_trial3/latest_checkpoint.json


In [ ]:

import os
import json
import numpy as np
import torch
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from tqdm.auto import tqdm

# Use the trained Trial 3 model for evaluation
trial3_model.eval()

all_predictions = []
all_labels = []

total_test_loss = 0.0
total_examples = 0

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating Qwen Trial 3"):

        # Move data to GPU
        batch = {
            key: value.to("cuda")
            for key, value in batch.items()
        }

        # Get predictions
        with torch.autocast("cuda", dtype=torch.float16):
            outputs = trial3_model(**batch)

        # Calculate test loss
        batch_size = batch["labels"].size(0)
        total_test_loss += outputs.loss.item() * batch_size
        total_examples += batch_size

        # Select the category with the highest score
        predictions = torch.argmax(outputs.logits, dim=1)

        all_predictions.extend(predictions.cpu().tolist())
        all_labels.extend(batch["labels"].cpu().tolist())

# Calculate evaluation metrics
test_accuracy = accuracy_score(all_labels, all_predictions)
test_macro_f1 = f1_score(
    all_labels, all_predictions, average="macro"
)
average_test_loss = total_test_loss / total_examples

class_names = ["World", "Sports", "Business", "Sci/Tech"]

print("\n===== QWEN TRIAL 3 TEST RESULTS =====")
print(f"Test examples: {total_examples}")
print(f"Test loss: {average_test_loss:.4f}")
print(f"Accuracy: {test_accuracy:.4f} ({test_accuracy * 100:.2f}%)")
print(f"Macro-F1: {test_macro_f1:.4f}")

print("\n===== CLASSIFICATION REPORT =====")
print(classification_report(
    all_labels,
    all_predictions,
    labels=[0, 1, 2, 3],
    target_names=class_names,
    digits=4,
    zero_division=0
))

print("===== CONFUSION MATRIX =====")
print(confusion_matrix(
    all_labels,
    all_predictions,
    labels=[0, 1, 2, 3]
))

# Save evaluation results to Google Drive
results_dir = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "Qwen_LoRA/qwen_lora_trial3"
)
os.makedirs(results_dir, exist_ok=True)

results = {
    "trial": 3,
    "model": "Qwen2.5-0.5B-Instruct with LoRA",
    "learning_rate": 1e-5,
    "batch_size": 8,
    "epochs": 2,
    "training_examples": 120000,
    "test_examples": int(total_examples),
    "test_loss": float(average_test_loss),
    "accuracy": float(test_accuracy),
    "macro_f1": float(test_macro_f1),
    "classification_report": classification_report(
        all_labels,
        all_predictions,
        labels=[0, 1, 2, 3],
        target_names=class_names,
        output_dict=True,
        zero_division=0
    ),
    "confusion_matrix": confusion_matrix(
        all_labels,
        all_predictions,
        labels=[0, 1, 2, 3]
    ).tolist()
}

results_path = os.path.join(
    results_dir, "qwen_test_results.json"
)

with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"\nResults saved to: {results_path}")

Evaluating Qwen Trial 3:   0%|          | 0/950 [00:00<?, ?it/s]


===== QWEN TRIAL 3 TEST RESULTS =====
Test examples: 7600
Test loss: 0.2893
Accuracy: 0.9334 (93.34%)
Macro-F1: 0.9334

===== CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

       World     0.9494    0.9374    0.9433      1900
      Sports     0.9827    0.9879    0.9853      1900
    Business     0.8989    0.8984    0.8987      1900
    Sci/Tech     0.9029    0.9100    0.9064      1900

    accuracy                         0.9334      7600
   macro avg     0.9335    0.9334    0.9334      7600
weighted avg     0.9335    0.9334    0.9334      7600

===== CONFUSION MATRIX =====
[[1781   16   59   44]
 [  11 1877    9    3]
 [  45    9 1707  139]
 [  39    8  124 1729]]

Results saved to: /content/drive/MyDrive/COMP6012_Assignment3/Qwen_LoRA/qwen_lora_trial3/qwen_test_results.json


Loading the DistilBERT Model

In [ ]:

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

DISTILBERT_MODEL_NAME = "distilbert/distilbert-base-uncased"

distilbert_tokenizer = AutoTokenizer.from_pretrained(
    DISTILBERT_MODEL_NAME
)

distilbert_model = AutoModelForSequenceClassification.from_pretrained(
    DISTILBERT_MODEL_NAME,
    num_labels=4,
    id2label={
        0: "World",
        1: "Sports",
        2: "Business",
        3: "Sci/Tech"
    },
    label2id={
        "World": 0,
        "Sports": 1,
        "Business": 2,
        "Sci/Tech": 3
    }
)

print("Model:", DISTILBERT_MODEL_NAME)
print("GPU available:", torch.cuda.is_available())
print("Number of labels:", distilbert_model.config.num_labels)
print("Tokenizer vocabulary size:", len(distilbert_tokenizer))
print(
    "Model parameters:",
    f"{sum(p.numel() for p in distilbert_model.parameters()):,}"
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model: distilbert/distilbert-base-uncased
GPU available: True
Number of labels: 4
Tokenizer vocabulary size: 30522
Model parameters: 66,956,548


Tokenizing AG News for DistilBERT

In [ ]:

from datasets import DatasetDict

MAX_LENGTH = 256

distilbert_tokenized_ds = DatasetDict({
    split: ag_news_prepared_ds[split].map(
        lambda batch: distilbert_tokenizer(
            batch["text"],
            truncation=True,
            max_length=MAX_LENGTH
        ),
        batched=True,
        desc=f"Tokenizing DistilBERT {split} data"
    )
    for split in ["train", "test"]
})

# Keep only model inputs and labels
distilbert_tokenized_ds = distilbert_tokenized_ds.rename_column(
    "label", "labels"
)

print("Train examples:", len(distilbert_tokenized_ds["train"]))
print("Test examples:", len(distilbert_tokenized_ds["test"]))
print("Columns:", distilbert_tokenized_ds["train"].column_names)
print("First example keys:", distilbert_tokenized_ds["train"][0].keys())

Train examples: 120000
Test examples: 7600
Columns: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask']
First example keys: dict_keys(['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'])


Evaluating the baseline for DistilBERT

In [ ]:

import torch
import numpy as np
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

# Dynamic padding and model-compatible input fields
distilbert_collator = DataCollatorWithPadding(
    tokenizer=distilbert_tokenizer,
    return_tensors="pt"
)

distilbert_test_eval_ds = distilbert_tokenized_ds["test"].remove_columns(
    ["text", "token_type_ids"]
)

distilbert_test_loader = DataLoader(
    distilbert_test_eval_ds,
    batch_size=32,
    shuffle=False,
    collate_fn=distilbert_collator,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

distilbert_model = distilbert_model.to("cuda")
distilbert_model.eval()

all_predictions = []
all_labels = []
total_eval_loss = 0.0
total_examples = 0

device = next(distilbert_model.parameters()).device

with torch.no_grad():
    for batch in tqdm(distilbert_test_loader, desc="Evaluating DistilBERT baseline"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = distilbert_model(**batch)

        batch_size = batch["labels"].size(0)
        total_eval_loss += outputs.loss.item() * batch_size
        total_examples += batch_size

        predictions = torch.argmax(outputs.logits, dim=-1)
        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(batch["labels"].cpu().numpy())

distilbert_baseline_loss = total_eval_loss / total_examples
distilbert_baseline_accuracy = accuracy_score(all_labels, all_predictions)
distilbert_baseline_macro_f1 = f1_score(
    all_labels, all_predictions, average="macro"
)

print(f"Baseline test loss: {distilbert_baseline_loss:.4f}")
print(f"Baseline test accuracy: {distilbert_baseline_accuracy:.4f}")
print(f"Baseline test macro-F1: {distilbert_baseline_macro_f1:.4f}")
print("\nPer-class results:")
print(classification_report(
    all_labels,
    all_predictions,
    labels=[0, 1, 2, 3],
    target_names=["World", "Sports", "Business", "Sci/Tech"],
    digits=4,
    zero_division=0
))

Evaluating DistilBERT baseline:   0%|          | 0/238 [00:00<?, ?it/s]

Baseline test loss: 1.3972
Baseline test accuracy: 0.2530
Baseline test macro-F1: 0.1070

Per-class results:
              precision    recall  f1-score   support

       World     0.0000    0.0000    0.0000      1900
      Sports     0.9259    0.0132    0.0259      1900
    Business     0.2515    0.9989    0.4019      1900
    Sci/Tech     0.0000    0.0000    0.0000      1900

    accuracy                         0.2530      7600
   macro avg     0.2944    0.2530    0.1070      7600
weighted avg     0.2944    0.2530    0.1070      7600



Setup DistilBERT model with LoRA

In [ ]:

import os
import torch
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, TaskType, get_peft_model

# Trial 1 settings
DISTILBERT_NAME = "distilbert/distilbert-base-uncased"
DISTILBERT_OUTPUT_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "DistilBERT_LoRA/distilbert_lora_trial1_lr2e5"
)

os.makedirs(DISTILBERT_OUTPUT_DIR, exist_ok=True)

# Reload a fresh base model; do not wrap the existing PEFT model
distilbert_base_model = AutoModelForSequenceClassification.from_pretrained(
    DISTILBERT_NAME,
    num_labels=4,
    id2label={
        0: "World",
        1: "Sports",
        2: "Business",
        3: "Sci/Tech",
    },
    label2id={
        "World": 0,
        "Sports": 1,
        "Business": 2,
        "Sci/Tech": 3,
    },
)

# Apply LoRA once to the fresh base model
distilbert_lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
    modules_to_save=["pre_classifier", "classifier"],
    bias="none",
)

distilbert_lora_model = get_peft_model(
    distilbert_base_model,
    distilbert_lora_config,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
distilbert_lora_model.to(device)

print("\nDistilBERT LoRA setup ready.")
distilbert_lora_model.print_trainable_parameters()
print("Device:", device)
print("Training examples:", len(distilbert_tokenized_ds["train"]))
print("Test examples:", len(distilbert_tokenized_ds["test"]))
print("Output directory:", DISTILBERT_OUTPUT_DIR)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



DistilBERT LoRA setup ready.
trainable params: 741,124 || all params: 67,697,672 || trainable%: 1.0948
Device: cuda
Training examples: 120000
Test examples: 7600
Output directory: /content/drive/MyDrive/COMP6012_Assignment3/DistilBERT_LoRA/distilbert_lora_trial1_lr2e5


Prepare DistilBERT LoRA Trial 1 training

Batch size: 8

Learning rate: 2e-05

Epochs: 1




In [ ]:

import torch
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

# Trial 1 configuration
DISTILBERT_BATCH_SIZE = 8
DISTILBERT_LR = 2e-5
DISTILBERT_EPOCHS = 1
DISTILBERT_WEIGHT_DECAY = 0.01
DISTILBERT_MAX_GRAD_NORM = 1.0
DISTILBERT_CHECKPOINT_EVERY = 500

# Dynamic padding; remove text and DistilBERT-unused token_type_ids
distilbert_train_loader_ds = (
    distilbert_tokenized_ds["train"]
    .remove_columns(["text", "token_type_ids"])
)

distilbert_test_loader_ds = (
    distilbert_tokenized_ds["test"]
    .remove_columns(["text", "token_type_ids"])
)

distilbert_collator = DataCollatorWithPadding(
    tokenizer=distilbert_tokenizer,
    return_tensors="pt"
)

distilbert_train_loader = DataLoader(
    distilbert_train_loader_ds,
    batch_size=DISTILBERT_BATCH_SIZE,
    shuffle=True,
    collate_fn=distilbert_collator,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

distilbert_test_loader = DataLoader(
    distilbert_test_loader_ds,
    batch_size=32,
    shuffle=False,
    collate_fn=distilbert_collator,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

# Optimise trainable LoRA parameters and the saved classification head
distilbert_optimizer = torch.optim.AdamW(
    [p for p in distilbert_lora_model.parameters() if p.requires_grad],
    lr=DISTILBERT_LR,
    weight_decay=DISTILBERT_WEIGHT_DECAY
)

DISTILBERT_TOTAL_STEPS = (
    len(distilbert_train_loader) * DISTILBERT_EPOCHS
)

print("Trial 1 training preparation complete.")
print("Training examples:", len(distilbert_train_loader_ds))
print("Test examples:", len(distilbert_test_loader_ds))
print("Training batches per epoch:", len(distilbert_train_loader))
print("Total training steps:", DISTILBERT_TOTAL_STEPS)
print("Batch size:", DISTILBERT_BATCH_SIZE)
print("Learning rate:", DISTILBERT_LR)
print("Epochs:", DISTILBERT_EPOCHS)
print("Checkpoint interval:", DISTILBERT_CHECKPOINT_EVERY)
print("Device:", next(distilbert_lora_model.parameters()).device)

Trial 1 training preparation complete.
Training examples: 120000
Test examples: 7600
Training batches per epoch: 15000
Total training steps: 15000
Batch size: 8
Learning rate: 2e-05
Epochs: 1
Checkpoint interval: 500
Device: cuda:0


Training DistilBERT + LoRA Trial 1

In [ ]:

import os
import json
import time
import torch
from tqdm.auto import tqdm
from peft import get_peft_model_state_dict

# Trial-specific paths
DISTILBERT_CHECKPOINT_DIR = os.path.join(
    DISTILBERT_OUTPUT_DIR, "checkpoints"
)
os.makedirs(DISTILBERT_CHECKPOINT_DIR, exist_ok=True)

DISTILBERT_LOG_PATH = os.path.join(
    DISTILBERT_OUTPUT_DIR, "training_log.json"
)

# Fresh run guard: do not silently overwrite previous training
if os.path.exists(DISTILBERT_LOG_PATH):
    raise FileExistsError(
        f"Training log already exists: {DISTILBERT_LOG_PATH}\n"
        "Stop here and inspect existing progress before starting again."
    )

distilbert_lora_model.train()
distilbert_device = next(distilbert_lora_model.parameters()).device

distilbert_step = 0
distilbert_running_loss = 0.0
distilbert_loss_window = 0.0
distilbert_window_steps = 0
distilbert_training_log = []
distilbert_start_time = time.time()

progress = tqdm(
    total=DISTILBERT_TOTAL_STEPS,
    desc="DistilBERT LoRA Trial 1"
)

for epoch in range(DISTILBERT_EPOCHS):
    for batch in distilbert_train_loader:
        batch = {
            key: value.to(distilbert_device)
            for key, value in batch.items()
        }

        distilbert_optimizer.zero_grad(set_to_none=True)

        outputs = distilbert_lora_model(**batch)
        loss = outputs.loss

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss at step {distilbert_step + 1}: {loss.item()}"
            )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            distilbert_lora_model.parameters(),
            DISTILBERT_MAX_GRAD_NORM
        )
        distilbert_optimizer.step()

        distilbert_step += 1
        loss_value = loss.item()
        distilbert_running_loss += loss_value
        distilbert_loss_window += loss_value
        distilbert_window_steps += 1

        progress.update(1)
        progress.set_postfix(loss=f"{loss_value:.4f}")

        # Log training loss every 100 steps
        if distilbert_step % 100 == 0:
            mean_loss = (
                distilbert_loss_window / distilbert_window_steps
            )
            distilbert_training_log.append({
                "step": distilbert_step,
                "epoch": epoch + 1,
                "mean_training_loss_100_steps": mean_loss,
                "elapsed_seconds": time.time() - distilbert_start_time
            })

            with open(DISTILBERT_LOG_PATH, "w") as f:
                json.dump(distilbert_training_log, f, indent=2)

            distilbert_loss_window = 0.0
            distilbert_window_steps = 0

        # Save adapter checkpoint every 500 steps
        if distilbert_step % DISTILBERT_CHECKPOINT_EVERY == 0:
            checkpoint_path = os.path.join(
                DISTILBERT_CHECKPOINT_DIR,
                f"checkpoint-{distilbert_step}"
            )
            os.makedirs(checkpoint_path, exist_ok=True)

            distilbert_lora_model.save_pretrained(checkpoint_path)
            distilbert_tokenizer.save_pretrained(checkpoint_path)

            # Save optimizer state for recovery
            torch.save(
                distilbert_optimizer.state_dict(),
                os.path.join(checkpoint_path, "optimizer.pt")
            )

            with open(os.path.join(checkpoint_path, "progress.json"), "w") as f:
                json.dump({
                    "step": distilbert_step,
                    "epoch": epoch,
                    "average_training_loss": (
                        distilbert_running_loss / distilbert_step
                    )
                }, f, indent=2)

progress.close()

# Save final adapter and tokenizer
distilbert_lora_model.save_pretrained(DISTILBERT_OUTPUT_DIR)
distilbert_tokenizer.save_pretrained(DISTILBERT_OUTPUT_DIR)

distilbert_training_summary = {
    "trial": 1,
    "learning_rate": DISTILBERT_LR,
    "epochs": DISTILBERT_EPOCHS,
    "batch_size": DISTILBERT_BATCH_SIZE,
    "training_examples": len(distilbert_train_loader_ds),
    "total_steps": distilbert_step,
    "average_training_loss": distilbert_running_loss / distilbert_step,
    "elapsed_seconds": time.time() - distilbert_start_time,
    "last_100_step_mean_loss": (
        distilbert_training_log[-1]["mean_training_loss_100_steps"]
        if distilbert_training_log else None
    )
}

with open(
    os.path.join(DISTILBERT_OUTPUT_DIR, "training_summary.json"), "w"
) as f:
    json.dump(distilbert_training_summary, f, indent=2)

print("\nTrial 1 training completed.")
print(json.dumps(distilbert_training_summary, indent=2))
print("Adapter saved to:", DISTILBERT_OUTPUT_DIR)
print("Checkpoints saved to:", DISTILBERT_CHECKPOINT_DIR)

DistilBERT LoRA Trial 1:   0%|          | 0/15000 [00:00<?, ?it/s]


Trial 1 training completed.
{
  "trial": 1,
  "learning_rate": 2e-05,
  "epochs": 1,
  "batch_size": 8,
  "training_examples": 120000,
  "total_steps": 15000,
  "average_training_loss": 0.3147667061840184,
  "elapsed_seconds": 720.4774484634399,
  "last_100_step_mean_loss": 0.2534594989940524
}
Adapter saved to: /content/drive/MyDrive/COMP6012_Assignment3/DistilBERT_LoRA/distilbert_lora_trial1_lr2e5
Checkpoints saved to: /content/drive/MyDrive/COMP6012_Assignment3/DistilBERT_LoRA/distilbert_lora_trial1_lr2e5/checkpoints


Evaluate Trial 1

In [ ]:

import os
import json
import torch
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from tqdm.auto import tqdm

# Evaluate the already-trained DistilBERT LoRA adapter
distilbert_lora_model.eval()
distilbert_device = next(distilbert_lora_model.parameters()).device

distilbert_all_labels = []
distilbert_all_predictions = []
distilbert_total_test_loss = 0.0
distilbert_total_test_examples = 0

with torch.no_grad():
    for batch in tqdm(distilbert_test_loader, desc="Evaluating DistilBERT Trial 1"):
        batch = {
            key: value.to(distilbert_device)
            for key, value in batch.items()
        }

        outputs = distilbert_lora_model(**batch)
        batch_size = batch["labels"].size(0)

        distilbert_total_test_loss += outputs.loss.item() * batch_size
        distilbert_total_test_examples += batch_size

        predictions = torch.argmax(outputs.logits, dim=-1)

        distilbert_all_labels.extend(batch["labels"].cpu().numpy().tolist())
        distilbert_all_predictions.extend(predictions.cpu().numpy().tolist())

distilbert_test_loss = (
    distilbert_total_test_loss / distilbert_total_test_examples
)
distilbert_accuracy = accuracy_score(
    distilbert_all_labels, distilbert_all_predictions
)
distilbert_macro_f1 = f1_score(
    distilbert_all_labels,
    distilbert_all_predictions,
    average="macro",
    zero_division=0
)

distilbert_class_names = ["World", "Sports", "Business", "Sci/Tech"]

distilbert_report = classification_report(
    distilbert_all_labels,
    distilbert_all_predictions,
    labels=[0, 1, 2, 3],
    target_names=distilbert_class_names,
    output_dict=True,
    zero_division=0
)

distilbert_trial1_results = {
    "model": "DistilBERT + LoRA",
    "trial": 1,
    "learning_rate": DISTILBERT_LR,
    "epochs": DISTILBERT_EPOCHS,
    "batch_size": DISTILBERT_BATCH_SIZE,
    "training_examples": len(distilbert_train_loader_ds),
    "test_examples": distilbert_total_test_examples,
    "test_loss": distilbert_test_loss,
    "accuracy": distilbert_accuracy,
    "macro_f1": distilbert_macro_f1,
    "classification_report": distilbert_report,
    "confusion_matrix": confusion_matrix(
        distilbert_all_labels,
        distilbert_all_predictions,
        labels=[0, 1, 2, 3]
    ).tolist()
}

distilbert_results_path = os.path.join(
    DISTILBERT_OUTPUT_DIR, "test_results.json"
)

with open(distilbert_results_path, "w") as f:
    json.dump(distilbert_trial1_results, f, indent=2)

print(f"Test loss: {distilbert_test_loss:.4f}")
print(f"Accuracy: {distilbert_accuracy:.4%}")
print(f"Macro-F1: {distilbert_macro_f1:.4f}")
print("\nPer-class evaluation:")
print(classification_report(
    distilbert_all_labels,
    distilbert_all_predictions,
    labels=[0, 1, 2, 3],
    target_names=distilbert_class_names,
    zero_division=0
))
print("Confusion matrix (rows=true, columns=predicted):")
print(np.array(distilbert_trial1_results["confusion_matrix"]))
print("\nResults saved to:", distilbert_results_path)

Evaluating DistilBERT Trial 1:   0%|          | 0/238 [00:00<?, ?it/s]

Test loss: 0.2726
Accuracy: 91.3158%
Macro-F1: 0.9130

Per-class evaluation:
              precision    recall  f1-score   support

       World       0.94      0.88      0.91      1900
      Sports       0.96      0.99      0.97      1900
    Business       0.89      0.86      0.87      1900
    Sci/Tech       0.86      0.93      0.89      1900

    accuracy                           0.91      7600
   macro avg       0.91      0.91      0.91      7600
weighted avg       0.91      0.91      0.91      7600

Confusion matrix (rows=true, columns=predicted):
[[1680   58   99   63]
 [   8 1874   14    4]
 [  49   10 1628  213]
 [  46    7   89 1758]]

Results saved to: /content/drive/MyDrive/COMP6012_Assignment3/DistilBERT_LoRA/distilbert_lora_trial1_lr2e5/test_results.json


Prepare DistilBERT LoRA Trial 2 training

Batch size: 8

Learning rate: 2e-04

Epochs: 1

In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

# Trial 2 configuration
DISTILBERT_BATCH_SIZE = 8
DISTILBERT_LR = 2e-4
DISTILBERT_EPOCHS = 1
DISTILBERT_WEIGHT_DECAY = 0.01
DISTILBERT_MAX_GRAD_NORM = 1.0
DISTILBERT_CHECKPOINT_EVERY = 500

# Dynamic padding; remove text and DistilBERT-unused token_type_ids
distilbert_train_loader_ds = (
    distilbert_tokenized_ds["train"]
    .remove_columns(["text", "token_type_ids"])
)

distilbert_test_loader_ds = (
    distilbert_tokenized_ds["test"]
    .remove_columns(["text", "token_type_ids"])
)

distilbert_collator = DataCollatorWithPadding(
    tokenizer=distilbert_tokenizer,
    return_tensors="pt"
)

distilbert_train_loader = DataLoader(
    distilbert_train_loader_ds,
    batch_size=DISTILBERT_BATCH_SIZE,
    shuffle=True,
    collate_fn=distilbert_collator,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

distilbert_test_loader = DataLoader(
    distilbert_test_loader_ds,
    batch_size=32,
    shuffle=False,
    collate_fn=distilbert_collator,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

# Optimise trainable LoRA parameters and the saved classification head
distilbert_optimizer = torch.optim.AdamW(
    [p for p in distilbert_lora_model.parameters() if p.requires_grad],
    lr=DISTILBERT_LR,
    weight_decay=DISTILBERT_WEIGHT_DECAY
)

DISTILBERT_TOTAL_STEPS = (
    len(distilbert_train_loader) * DISTILBERT_EPOCHS
)

print("Trial 2 training preparation complete.")
print("Training examples:", len(distilbert_train_loader_ds))
print("Test examples:", len(distilbert_test_loader_ds))
print("Training batches per epoch:", len(distilbert_train_loader))
print("Total training steps:", DISTILBERT_TOTAL_STEPS)
print("Batch size:", DISTILBERT_BATCH_SIZE)
print("Learning rate:", DISTILBERT_LR)
print("Epochs:", DISTILBERT_EPOCHS)
print("Checkpoint interval:", DISTILBERT_CHECKPOINT_EVERY)
print("Device:", next(distilbert_lora_model.parameters()).device)

Trial 2 training preparation complete.
Training examples: 120000
Test examples: 7600
Training batches per epoch: 15000
Total training steps: 15000
Batch size: 8
Learning rate: 0.0002
Epochs: 1
Checkpoint interval: 500
Device: cuda:0


Training DistilBERT + LoRA Trial 2

In [ ]:
import os
import json
import time
import torch
from tqdm.auto import tqdm

# Trial 2 output directory
DISTILBERT_OUTPUT_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "DistilBERT_LoRA/distilbert_lora_trial2_lr2e4"
)

DISTILBERT_CHECKPOINT_DIR = os.path.join(
    DISTILBERT_OUTPUT_DIR, "checkpoints"
)
os.makedirs(DISTILBERT_CHECKPOINT_DIR, exist_ok=True)

DISTILBERT_LOG_PATH = os.path.join(
    DISTILBERT_OUTPUT_DIR, "training_log.json"
)

# Training log
distilbert_training_log = {
    "trial": 2,
    "learning_rate": DISTILBERT_LR,
    "epochs": DISTILBERT_EPOCHS,
    "batch_size": DISTILBERT_BATCH_SIZE,
    "training_examples": len(distilbert_train_loader_ds),
    "total_steps": DISTILBERT_TOTAL_STEPS,
    "step_losses": [],
    "epoch_losses": [],
    "elapsed_seconds": None,
}

# Train
distilbert_lora_model.train()
distilbert_global_step = 0
distilbert_start_time = time.time()
distilbert_running_loss = 0.0

distilbert_progress = tqdm(
    total=DISTILBERT_TOTAL_STEPS,
    desc="DistilBERT LoRA Trial 2"
)

for epoch in range(DISTILBERT_EPOCHS):
    distilbert_epoch_loss = 0.0

    for batch in distilbert_train_loader:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        distilbert_optimizer.zero_grad(set_to_none=True)

        outputs = distilbert_lora_model(**batch)
        loss = outputs.loss

        if not torch.isfinite(loss):
            raise ValueError(
                f"Non-finite loss at step {distilbert_global_step + 1}: "
                f"{loss.item()}"
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            distilbert_lora_model.parameters(),
            DISTILBERT_MAX_GRAD_NORM
        )

        distilbert_optimizer.step()

        batch_loss = loss.item()
        distilbert_global_step += 1
        distilbert_epoch_loss += batch_loss
        distilbert_running_loss += batch_loss

        distilbert_training_log["step_losses"].append({
            "step": distilbert_global_step,
            "loss": batch_loss
        })

        distilbert_progress.update(1)
        distilbert_progress.set_postfix(
            loss=f"{batch_loss:.4f}",
            lr=f"{DISTILBERT_LR:.1e}"
        )

        # Save resumable training checkpoint every 500 steps
        if (
            distilbert_global_step
            % DISTILBERT_CHECKPOINT_EVERY == 0
        ):
            distilbert_checkpoint_path = os.path.join(
                DISTILBERT_CHECKPOINT_DIR,
                f"checkpoint-{distilbert_global_step}"
            )
            os.makedirs(distilbert_checkpoint_path, exist_ok=True)

            distilbert_lora_model.save_pretrained(
                distilbert_checkpoint_path
            )
            distilbert_tokenizer.save_pretrained(
                distilbert_checkpoint_path
            )
            torch.save(
                {
                    "optimizer_state_dict":
                        distilbert_optimizer.state_dict(),
                    "global_step": distilbert_global_step,
                    "epoch": epoch,
                    "running_loss": distilbert_running_loss,
                },
                os.path.join(
                    distilbert_checkpoint_path,
                    "training_state.pt"
                )
            )

            with open(DISTILBERT_LOG_PATH, "w") as f:
                json.dump(distilbert_training_log, f, indent=2)

    distilbert_average_epoch_loss = (
        distilbert_epoch_loss / len(distilbert_train_loader)
    )
    distilbert_training_log["epoch_losses"].append(
        distilbert_average_epoch_loss
    )

    print(
        f"Trial 2 | Epoch {epoch + 1}/{DISTILBERT_EPOCHS} "
        f"| Average training loss: "
        f"{distilbert_average_epoch_loss:.4f}"
    )

distilbert_progress.close()

# Save final adapter and tokenizer
distilbert_lora_model.save_pretrained(DISTILBERT_OUTPUT_DIR)
distilbert_tokenizer.save_pretrained(DISTILBERT_OUTPUT_DIR)

distilbert_elapsed_seconds = time.time() - distilbert_start_time

distilbert_training_log["average_training_loss"] = (
    distilbert_running_loss / distilbert_global_step
)
distilbert_training_log["elapsed_seconds"] = (
    distilbert_elapsed_seconds
)
distilbert_training_log["last_100_step_mean_loss"] = (
    sum(
        item["loss"]
        for item in distilbert_training_log["step_losses"][-100:]
    )
    / min(100, len(distilbert_training_log["step_losses"]))
)

with open(DISTILBERT_LOG_PATH, "w") as f:
    json.dump(distilbert_training_log, f, indent=2)

print("\nTrial 2 training complete.")
print(json.dumps({
    "trial": 2,
    "learning_rate": DISTILBERT_LR,
    "epochs": DISTILBERT_EPOCHS,
    "training_examples": len(distilbert_train_loader_ds),
    "total_steps": distilbert_global_step,
    "average_training_loss":
        distilbert_training_log["average_training_loss"],
    "last_100_step_mean_loss":
        distilbert_training_log["last_100_step_mean_loss"],
    "elapsed_seconds": distilbert_elapsed_seconds,
    "adapter_directory": DISTILBERT_OUTPUT_DIR,
    "training_log": DISTILBERT_LOG_PATH,
}, indent=2))

DistilBERT LoRA Trial 2:   0%|          | 0/15000 [00:00<?, ?it/s]

Trial 2 | Epoch 1/1 | Average training loss: 0.2450

Trial 2 training complete.
{
  "trial": 2,
  "learning_rate": 0.0002,
  "epochs": 1,
  "training_examples": 120000,
  "total_steps": 15000,
  "average_training_loss": 0.24498949335931489,
  "last_100_step_mean_loss": 0.30593760827439836,
  "elapsed_seconds": 713.7465059757233,
  "adapter_directory": "/content/drive/MyDrive/COMP6012_Assignment3/DistilBERT_LoRA/distilbert_lora_trial2_lr2e4",
  "training_log": "/content/drive/MyDrive/COMP6012_Assignment3/DistilBERT_LoRA/distilbert_lora_trial2_lr2e4/training_log.json"
}


Evaluate Trial 2

In [ ]:
import os
import json
import torch
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from tqdm.auto import tqdm

# Trial 2 evaluation paths
DISTILBERT_OUTPUT_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "DistilBERT_LoRA/distilbert_lora_trial2_lr2e4"
)

DISTILBERT_RESULTS_PATH = os.path.join(
    DISTILBERT_OUTPUT_DIR, "test_results.json"
)

distilbert_lora_model.eval()

distilbert_all_predictions = []
distilbert_all_labels = []
distilbert_total_test_loss = 0.0
distilbert_test_batches = 0

with torch.no_grad():
    for batch in tqdm(
        distilbert_test_loader,
        desc="Evaluating DistilBERT Trial 2"
    ):
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        outputs = distilbert_lora_model(**batch)

        distilbert_total_test_loss += outputs.loss.item()
        distilbert_test_batches += 1

        predictions = torch.argmax(outputs.logits, dim=-1)

        distilbert_all_predictions.extend(
            predictions.cpu().tolist()
        )
        distilbert_all_labels.extend(
            batch["labels"].cpu().tolist()
        )

distilbert_test_loss = (
    distilbert_total_test_loss / distilbert_test_batches
)
distilbert_accuracy = accuracy_score(
    distilbert_all_labels,
    distilbert_all_predictions
)
distilbert_macro_f1 = f1_score(
    distilbert_all_labels,
    distilbert_all_predictions,
    average="macro",
    zero_division=0
)

distilbert_class_report = classification_report(
    distilbert_all_labels,
    distilbert_all_predictions,
    labels=[0, 1, 2, 3],
    target_names=["World", "Sports", "Business", "Sci/Tech"],
    output_dict=True,
    zero_division=0
)

distilbert_confusion_matrix = confusion_matrix(
    distilbert_all_labels,
    distilbert_all_predictions,
    labels=[0, 1, 2, 3]
)

distilbert_test_results = {
    "trial": 2,
    "model": "DistilBERT LoRA",
    "learning_rate": DISTILBERT_LR,
    "epochs": DISTILBERT_EPOCHS,
    "test_examples": len(distilbert_all_labels),
    "test_loss": distilbert_test_loss,
    "accuracy": distilbert_accuracy,
    "macro_f1": distilbert_macro_f1,
    "classification_report": distilbert_class_report,
    "confusion_matrix": distilbert_confusion_matrix.tolist()
}

with open(DISTILBERT_RESULTS_PATH, "w") as f:
    json.dump(distilbert_test_results, f, indent=2)

print("DistilBERT Trial 2 test results")
print(f"Test loss: {distilbert_test_loss:.4f}")
print(f"Accuracy: {distilbert_accuracy:.4%}")
print(f"Macro-F1: {distilbert_macro_f1:.4f}")

print("\nPer-class classification report:")
print(classification_report(
    distilbert_all_labels,
    distilbert_all_predictions,
    labels=[0, 1, 2, 3],
    target_names=["World", "Sports", "Business", "Sci/Tech"],
    zero_division=0
))

print("Confusion matrix (rows=true, columns=predicted):")
print(distilbert_confusion_matrix)

print("\nResults saved to:", DISTILBERT_RESULTS_PATH)

Evaluating DistilBERT Trial 2:   0%|          | 0/238 [00:00<?, ?it/s]

DistilBERT Trial 2 test results
Test loss: 0.2290
Accuracy: 93.0921%
Macro-F1: 0.9310

Per-class classification report:
              precision    recall  f1-score   support

       World       0.94      0.93      0.94      1900
      Sports       0.98      0.98      0.98      1900
    Business       0.90      0.90      0.90      1900
    Sci/Tech       0.90      0.91      0.91      1900

    accuracy                           0.93      7600
   macro avg       0.93      0.93      0.93      7600
weighted avg       0.93      0.93      0.93      7600

Confusion matrix (rows=true, columns=predicted):
[[1764   21   61   54]
 [  24 1860   10    6]
 [  41   11 1715  133]
 [  41    9  114 1736]]

Results saved to: /content/drive/MyDrive/COMP6012_Assignment3/DistilBERT_LoRA/distilbert_lora_trial2_lr2e4/test_results.json


Prepare DistilBERT LoRA Trial 3 training

Batch size: 8

Learning rate: 2e-05

Epochs: 2

In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

# Trial 3 configuration
DISTILBERT_BATCH_SIZE = 8
DISTILBERT_LR = 2e-5
DISTILBERT_EPOCHS = 2
DISTILBERT_WEIGHT_DECAY = 0.01
DISTILBERT_MAX_GRAD_NORM = 1.0
DISTILBERT_CHECKPOINT_EVERY = 500

# Dynamic padding; remove text and DistilBERT-unused token_type_ids
distilbert_train_loader_ds = (
    distilbert_tokenized_ds["train"]
    .remove_columns(["text", "token_type_ids"])
)

distilbert_test_loader_ds = (
    distilbert_tokenized_ds["test"]
    .remove_columns(["text", "token_type_ids"])
)

distilbert_collator = DataCollatorWithPadding(
    tokenizer=distilbert_tokenizer,
    return_tensors="pt"
)

distilbert_train_loader = DataLoader(
    distilbert_train_loader_ds,
    batch_size=DISTILBERT_BATCH_SIZE,
    shuffle=True,
    collate_fn=distilbert_collator,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

distilbert_test_loader = DataLoader(
    distilbert_test_loader_ds,
    batch_size=32,
    shuffle=False,
    collate_fn=distilbert_collator,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

# Optimise trainable LoRA parameters and the saved classification head
distilbert_optimizer = torch.optim.AdamW(
    [p for p in distilbert_lora_model.parameters() if p.requires_grad],
    lr=DISTILBERT_LR,
    weight_decay=DISTILBERT_WEIGHT_DECAY
)

DISTILBERT_TOTAL_STEPS = (
    len(distilbert_train_loader) * DISTILBERT_EPOCHS
)

print("Trial 3 training preparation complete.")
print("Training examples:", len(distilbert_train_loader_ds))
print("Test examples:", len(distilbert_test_loader_ds))
print("Training batches per epoch:", len(distilbert_train_loader))
print("Total training steps:", DISTILBERT_TOTAL_STEPS)
print("Batch size:", DISTILBERT_BATCH_SIZE)
print("Learning rate:", DISTILBERT_LR)
print("Epochs:", DISTILBERT_EPOCHS)
print("Checkpoint interval:", DISTILBERT_CHECKPOINT_EVERY)
print("Device:", next(distilbert_lora_model.parameters()).device)

Trial 3 training preparation complete.
Training examples: 120000
Test examples: 7600
Training batches per epoch: 15000
Total training steps: 30000
Batch size: 8
Learning rate: 2e-05
Epochs: 2
Checkpoint interval: 500
Device: cuda:0


Training DistilBERT + LoRA Trial 3

In [ ]:
import os
import json
import time
import torch
from tqdm.auto import tqdm

# Trial 3 output directory
DISTILBERT_OUTPUT_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "DistilBERT_LoRA/distilbert_lora_trial3_lr2e5_2epochs"
)

DISTILBERT_CHECKPOINT_DIR = os.path.join(
    DISTILBERT_OUTPUT_DIR, "checkpoints"
)
os.makedirs(DISTILBERT_CHECKPOINT_DIR, exist_ok=True)

DISTILBERT_LOG_PATH = os.path.join(
    DISTILBERT_OUTPUT_DIR, "training_log.json"
)

# Training log
distilbert_training_log = {
    "trial": 3,
    "learning_rate": DISTILBERT_LR,
    "epochs": DISTILBERT_EPOCHS,
    "batch_size": DISTILBERT_BATCH_SIZE,
    "training_examples": len(distilbert_train_loader_ds),
    "total_steps": DISTILBERT_TOTAL_STEPS,
    "step_losses": [],
    "epoch_losses": [],
    "elapsed_seconds": None,
}

# Train
distilbert_lora_model.train()
distilbert_global_step = 0
distilbert_start_time = time.time()
distilbert_running_loss = 0.0

distilbert_progress = tqdm(
    total=DISTILBERT_TOTAL_STEPS,
    desc="DistilBERT LoRA Trial 3"
)

for epoch in range(DISTILBERT_EPOCHS):
    distilbert_epoch_loss = 0.0

    for batch in distilbert_train_loader:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        distilbert_optimizer.zero_grad(set_to_none=True)

        outputs = distilbert_lora_model(**batch)
        loss = outputs.loss

        if not torch.isfinite(loss):
            raise ValueError(
                f"Non-finite loss at step {distilbert_global_step + 1}: "
                f"{loss.item()}"
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            distilbert_lora_model.parameters(),
            DISTILBERT_MAX_GRAD_NORM
        )

        distilbert_optimizer.step()

        batch_loss = loss.item()
        distilbert_global_step += 1
        distilbert_epoch_loss += batch_loss
        distilbert_running_loss += batch_loss

        distilbert_training_log["step_losses"].append({
            "step": distilbert_global_step,
            "loss": batch_loss
        })

        distilbert_progress.update(1)
        distilbert_progress.set_postfix(
            loss=f"{batch_loss:.4f}",
            lr=f"{DISTILBERT_LR:.1e}"
        )

        # Save checkpoint every 500 steps
        if (
            distilbert_global_step
            % DISTILBERT_CHECKPOINT_EVERY == 0
        ):
            distilbert_checkpoint_path = os.path.join(
                DISTILBERT_CHECKPOINT_DIR,
                f"checkpoint-{distilbert_global_step}"
            )
            os.makedirs(distilbert_checkpoint_path, exist_ok=True)

            distilbert_lora_model.save_pretrained(
                distilbert_checkpoint_path
            )
            distilbert_tokenizer.save_pretrained(
                distilbert_checkpoint_path
            )

            torch.save(
                {
                    "optimizer_state_dict":
                        distilbert_optimizer.state_dict(),
                    "global_step": distilbert_global_step,
                    "epoch": epoch,
                    "running_loss": distilbert_running_loss,
                },
                os.path.join(
                    distilbert_checkpoint_path,
                    "training_state.pt"
                )
            )

            with open(DISTILBERT_LOG_PATH, "w") as f:
                json.dump(distilbert_training_log, f, indent=2)

    distilbert_average_epoch_loss = (
        distilbert_epoch_loss / len(distilbert_train_loader)
    )
    distilbert_training_log["epoch_losses"].append(
        distilbert_average_epoch_loss
    )

    print(
        f"Trial 3 | Epoch {epoch + 1}/{DISTILBERT_EPOCHS} "
        f"| Average training loss: "
        f"{distilbert_average_epoch_loss:.4f}"
    )

distilbert_progress.close()

# Save final adapter and tokenizer
distilbert_lora_model.save_pretrained(DISTILBERT_OUTPUT_DIR)
distilbert_tokenizer.save_pretrained(DISTILBERT_OUTPUT_DIR)

distilbert_elapsed_seconds = time.time() - distilbert_start_time

distilbert_training_log["average_training_loss"] = (
    distilbert_running_loss / distilbert_global_step
)
distilbert_training_log["elapsed_seconds"] = (
    distilbert_elapsed_seconds
)
distilbert_training_log["last_100_step_mean_loss"] = (
    sum(
        item["loss"]
        for item in distilbert_training_log["step_losses"][-100:]
    )
    / min(100, len(distilbert_training_log["step_losses"]))
)

with open(DISTILBERT_LOG_PATH, "w") as f:
    json.dump(distilbert_training_log, f, indent=2)

print("\nTrial 3 training complete.")
print(json.dumps({
    "trial": 3,
    "learning_rate": DISTILBERT_LR,
    "epochs": DISTILBERT_EPOCHS,
    "training_examples": len(distilbert_train_loader_ds),
    "total_steps": distilbert_global_step,
    "average_training_loss":
        distilbert_training_log["average_training_loss"],
    "last_100_step_mean_loss":
        distilbert_training_log["last_100_step_mean_loss"],
    "elapsed_seconds": distilbert_elapsed_seconds,
    "adapter_directory": DISTILBERT_OUTPUT_DIR,
    "training_log": DISTILBERT_LOG_PATH,
}, indent=2))

DistilBERT LoRA Trial 3:   0%|          | 0/30000 [00:00<?, ?it/s]

Trial 3 | Epoch 1/2 | Average training loss: 0.1855
Trial 3 | Epoch 2/2 | Average training loss: 0.1777

Trial 3 training complete.
{
  "trial": 3,
  "learning_rate": 2e-05,
  "epochs": 2,
  "training_examples": 120000,
  "total_steps": 30000,
  "average_training_loss": 0.18159635504507485,
  "last_100_step_mean_loss": 0.19542416166441398,
  "elapsed_seconds": 1484.6532125473022,
  "adapter_directory": "/content/drive/MyDrive/COMP6012_Assignment3/DistilBERT_LoRA/distilbert_lora_trial3_lr2e5_2epochs",
  "training_log": "/content/drive/MyDrive/COMP6012_Assignment3/DistilBERT_LoRA/distilbert_lora_trial3_lr2e5_2epochs/training_log.json"
}


Evaluating Trial 3

In [ ]:
import os
import json
import torch
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from tqdm.auto import tqdm

# Trial 3 evaluation paths
DISTILBERT_OUTPUT_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "DistilBERT_LoRA/distilbert_lora_trial3_lr2e5_2epochs"
)

DISTILBERT_RESULTS_PATH = os.path.join(
    DISTILBERT_OUTPUT_DIR, "test_results.json"
)

distilbert_lora_model.eval()

distilbert_all_predictions = []
distilbert_all_labels = []
distilbert_total_test_loss = 0.0
distilbert_test_batches = 0

with torch.no_grad():
    for batch in tqdm(
        distilbert_test_loader,
        desc="Evaluating DistilBERT Trial 3"
    ):
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        outputs = distilbert_lora_model(**batch)

        distilbert_total_test_loss += outputs.loss.item()
        distilbert_test_batches += 1

        predictions = torch.argmax(outputs.logits, dim=-1)

        distilbert_all_predictions.extend(
            predictions.cpu().tolist()
        )
        distilbert_all_labels.extend(
            batch["labels"].cpu().tolist()
        )

distilbert_test_loss = (
    distilbert_total_test_loss / distilbert_test_batches
)
distilbert_accuracy = accuracy_score(
    distilbert_all_labels,
    distilbert_all_predictions
)
distilbert_macro_f1 = f1_score(
    distilbert_all_labels,
    distilbert_all_predictions,
    average="macro",
    zero_division=0
)

distilbert_class_report = classification_report(
    distilbert_all_labels,
    distilbert_all_predictions,
    labels=[0, 1, 2, 3],
    target_names=["World", "Sports", "Business", "Sci/Tech"],
    output_dict=True,
    zero_division=0
)

distilbert_confusion_matrix = confusion_matrix(
    distilbert_all_labels,
    distilbert_all_predictions,
    labels=[0, 1, 2, 3]
)

distilbert_test_results = {
    "trial": 3,
    "model": "DistilBERT LoRA",
    "learning_rate": DISTILBERT_LR,
    "epochs": DISTILBERT_EPOCHS,
    "test_examples": len(distilbert_all_labels),
    "test_loss": distilbert_test_loss,
    "accuracy": distilbert_accuracy,
    "macro_f1": distilbert_macro_f1,
    "classification_report": distilbert_class_report,
    "confusion_matrix": distilbert_confusion_matrix.tolist()
}

with open(DISTILBERT_RESULTS_PATH, "w") as f:
    json.dump(distilbert_test_results, f, indent=2)

print("DistilBERT Trial 3 test results")
print(f"Test loss: {distilbert_test_loss:.4f}")
print(f"Accuracy: {distilbert_accuracy:.4%}")
print(f"Macro-F1: {distilbert_macro_f1:.4f}")

print("\nPer-class classification report:")
print(classification_report(
    distilbert_all_labels,
    distilbert_all_predictions,
    labels=[0, 1, 2, 3],
    target_names=["World", "Sports", "Business", "Sci/Tech"],
    zero_division=0
))

print("Confusion matrix (rows=true, columns=predicted):")
print(distilbert_confusion_matrix)

print("\nResults saved to:", DISTILBERT_RESULTS_PATH)

Evaluating DistilBERT Trial 3:   0%|          | 0/238 [00:00<?, ?it/s]

DistilBERT Trial 3 test results
Test loss: 0.2044
Accuracy: 94.2237%
Macro-F1: 0.9422

Per-class classification report:
              precision    recall  f1-score   support

       World       0.96      0.94      0.95      1900
      Sports       0.98      0.99      0.99      1900
    Business       0.92      0.90      0.91      1900
    Sci/Tech       0.90      0.94      0.92      1900

    accuracy                           0.94      7600
   macro avg       0.94      0.94      0.94      7600
weighted avg       0.94      0.94      0.94      7600

Confusion matrix (rows=true, columns=predicted):
[[1795   11   52   42]
 [  11 1876    7    6]
 [  38    9 1710  143]
 [  30   10   80 1780]]

Results saved to: /content/drive/MyDrive/COMP6012_Assignment3/DistilBERT_LoRA/distilbert_lora_trial3_lr2e5_2epochs/test_results.json


Task 2 : Generation

Loading PersonaChat dataset

In [ ]:

from datasets import load_dataset

# Load the exact dataset specified in the assignment guide
persona_dataset = load_dataset("AlekseyKorshuk/persona-chat")

# Display split sizes and dataset structure
print(persona_dataset)

print("\nDataset split sizes:")
for split in persona_dataset:
    print(f"{split}: {len(persona_dataset[split])} examples")

print("\nTraining columns:")
print(persona_dataset["train"].column_names)

print("\nFirst training example:")
print(persona_dataset["train"][0])

dataset_infos.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 97.5MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 4.82MB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/17878 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['personality', 'utterances'],
        num_rows: 17878
    })
    validation: Dataset({
        features: ['personality', 'utterances'],
        num_rows: 1000
    })
})

Dataset split sizes:
train: 17878 examples
validation: 1000 examples

Training columns:
['personality', 'utterances']

First training example:
{'personality': ['i like to remodel homes .', 'i like to go hunting .', 'i like to shoot a bow .', 'my favorite holiday is halloween .'], 'utterances': [{'candidates': ['my mom was single with 3 boys , so we never left the projects .', 'i try to wear all black every day . it makes me feel comfortable .', 'well nursing stresses you out so i wish luck with sister', 'yeah just want to pick up nba nfl getting old', 'i really like celine dion . what about you ?', 'no . i live near farms .', "i wish i had a daughter , i'm a boy mom . they're beautiful boys though still lucky", 'yeah when i get bored i play gone with the wind my fav

Dataset Exploration: Inspecting the target response

In [ ]:

# Inspect the final dialogue turn of the first training example
sample = persona_dataset["train"][0]

print("Personality:")
for fact in sample["personality"]:
    print("-", fact)

print("\nFinal dialogue history:")
for turn in sample["utterances"][-1]["history"]:
    print("-", turn)

print("\nNumber of candidates:",
      len(sample["utterances"][-1]["candidates"]))

print("\nLast candidate:")
print(sample["utterances"][-1]["candidates"][-1])

Personality:
- i like to remodel homes .
- i like to go hunting .
- i like to shoot a bow .
- my favorite holiday is halloween .

Final dialogue history:
- hi , how are you doing ? i'm getting ready to do some cheetah chasing to stay in shape .
- you must be very fast . hunting is one of my favorite hobbies .
- i am ! for my hobby i like to do canning or some whittling .
- i also remodel homes when i am not out bow hunting .
- that's neat . when i was in high school i placed 6th in 100m dash !
- that's awesome . do you have a favorite season or time of year ?
- i do not . but i do have a favorite meat since that is all i eat exclusively .
- what is your favorite meat to eat ?
- i would have to say its prime rib . do you have any favorite foods ?
- i like chicken or macaroni and cheese .
- do you have anything planned for today ? i think i am going to do some canning .
- i am going to watch football . what are you canning ?
- i think i will can some jam . do you also play footfall for f

Preparing the training examples

In [ ]:

def prepare_personachat_examples(dataset_split):
    examples = []

    for conversation in dataset_split:
        personality = conversation["personality"]
        utterances = conversation["utterances"]

        for turn in utterances:
            history = turn["history"]
            candidates = turn["candidates"]

            # The final candidate is the intended response
            target_response = candidates[-1]

            prompt = (
                "Persona:\n"
                + "\n".join(f"- {fact}" for fact in personality)
                + "\n\nConversation:\n"
                + "\n".join(history)
                + "\nAssistant:"
            )

            examples.append({
                "prompt": prompt,
                "response": target_response
            })

    return examples


persona_train_examples = prepare_personachat_examples(
    persona_dataset["train"]
)

persona_validation_examples = prepare_personachat_examples(
    persona_dataset["validation"]
)

print("Training examples:", len(persona_train_examples))
print("Validation examples:", len(persona_validation_examples))

print("\nFirst prepared prompt:")
print(persona_train_examples[0]["prompt"])

print("\nTarget response:")
print(persona_train_examples[0]["response"])

Training examples: 131438
Validation examples: 7801

First prepared prompt:
Persona:
- i like to remodel homes .
- i like to go hunting .
- i like to shoot a bow .
- my favorite holiday is halloween .

Conversation:
hi , how are you doing ? i'm getting ready to do some cheetah chasing to stay in shape .
Assistant:

Target response:
you must be very fast . hunting is one of my favorite hobbies .


Checking prompt and response lengths

Inspecting the text lengths to decide choosing a suitable token limit. Long conversation histories can be truncated during training.

In [ ]:

import numpy as np

train_prompt_lengths = [
    len(example["prompt"].split())
    for example in persona_train_examples
]

train_response_lengths = [
    len(example["response"].split())
    for example in persona_train_examples
]

print("Training prompt lengths (words):")
print("Median:", int(np.percentile(train_prompt_lengths, 50)))
print("95th percentile:", int(np.percentile(train_prompt_lengths, 95)))
print("Maximum:", max(train_prompt_lengths))

print("\nTraining response lengths (words):")
print("Median:", int(np.percentile(train_response_lengths, 50)))
print("95th percentile:", int(np.percentile(train_response_lengths, 95)))
print("Maximum:", max(train_response_lengths))

Training prompt lengths (words):
Median: 117
95th percentile: 214
Maximum: 520

Training response lengths (words):
Median: 11
95th percentile: 18
Maximum: 60


Loading Qwen and GPT-2 model tokenizers

In [ ]:

from transformers import AutoTokenizer

# Model names
GPT2_MODEL_NAME = "openai-community/gpt2"
QWEN_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# Load tokenizers
gpt2_tokenizer = AutoTokenizer.from_pretrained(GPT2_MODEL_NAME)
qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME)

# GPT-2 has no dedicated padding token
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token

# Use right padding for both models
gpt2_tokenizer.padding_side = "right"
qwen_tokenizer.padding_side = "right"

print("GPT-2 tokenizer loaded")
print("  Vocabulary size:", len(gpt2_tokenizer))
print("  Pad token:", gpt2_tokenizer.pad_token)
print("  EOS token:", gpt2_tokenizer.eos_token)

print("\nQwen tokenizer loaded")
print("  Vocabulary size:", len(qwen_tokenizer))
print("  Pad token:", qwen_tokenizer.pad_token)
print("  EOS token:", qwen_tokenizer.eos_token)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

GPT-2 tokenizer loaded
  Vocabulary size: 50257
  Pad token: <|endoftext|>
  EOS token: <|endoftext|>

Qwen tokenizer loaded
  Vocabulary size: 151665
  Pad token: <|endoftext|>
  EOS token: <|im_end|>


Tokenizing the PersonaChat data

tokenizing the prompt–response pairs separately for GPT-2 and Qwen
Max token length 320

In [ ]:

MAX_LENGTH = 320

def tokenize_personachat_examples(examples, tokenizer, max_length=MAX_LENGTH):
    input_ids_list = []
    attention_masks_list = []
    labels_list = []

    for example in examples:
        prompt = example["prompt"]
        response = example["response"]

        # Tokenize prompt and response separately
        prompt_ids = tokenizer(
            prompt,
            add_special_tokens=False
        )["input_ids"]

        response_ids = tokenizer(
            response,
            add_special_tokens=False
        )["input_ids"]

        # Reserve space for response and EOS token
        response_ids = response_ids[:max_length - 1]
        response_ids = response_ids + [tokenizer.eos_token_id]

        # Keep the response; truncate the prompt from the left if needed
        available_prompt_length = max_length - len(response_ids)
        prompt_ids = prompt_ids[-available_prompt_length:] if available_prompt_length > 0 else []

        input_ids = prompt_ids + response_ids
        labels = [-100] * len(prompt_ids) + response_ids.copy()
        attention_mask = [1] * len(input_ids)

        input_ids_list.append(input_ids)
        attention_masks_list.append(attention_mask)
        labels_list.append(labels)

    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_masks_list,
        "labels": labels_list
    }


# Tokenize the full training and validation data for both models
gpt2_train_tokenized = tokenize_personachat_examples(
    persona_train_examples, gpt2_tokenizer
)
gpt2_validation_tokenized = tokenize_personachat_examples(
    persona_validation_examples, gpt2_tokenizer
)

qwen_train_tokenized = tokenize_personachat_examples(
    persona_train_examples, qwen_tokenizer
)
qwen_validation_tokenized = tokenize_personachat_examples(
    persona_validation_examples, qwen_tokenizer
)

print("GPT-2 training examples:", len(gpt2_train_tokenized["input_ids"]))
print("GPT-2 validation examples:", len(gpt2_validation_tokenized["input_ids"]))
print("Qwen training examples:", len(qwen_train_tokenized["input_ids"]))
print("Qwen validation examples:", len(qwen_validation_tokenized["input_ids"]))

print("\nGPT-2 first example token length:",
      len(gpt2_train_tokenized["input_ids"][0]))
print("GPT-2 response tokens used in loss:",
      sum(label != -100 for label in gpt2_train_tokenized["labels"][0]))

print("\nQwen first example token length:",
      len(qwen_train_tokenized["input_ids"][0]))
print("Qwen response tokens used in loss:",
      sum(label != -100 for label in qwen_train_tokenized["labels"][0]))

GPT-2 training examples: 131438
GPT-2 validation examples: 7801
Qwen training examples: 131438
Qwen validation examples: 7801

GPT-2 first example token length: 87
GPT-2 response tokens used in loss: 15

Qwen first example token length: 74
Qwen response tokens used in loss: 15


Creating PyTorch datasets and data loaders

this prepares the data for model training

In [ ]:

import torch
from torch.utils.data import Dataset, DataLoader

class PersonaChatTorchDataset(Dataset):
    def __init__(self, tokenized_data):
        self.input_ids = tokenized_data["input_ids"]
        self.attention_masks = tokenized_data["attention_mask"]
        self.labels = tokenized_data["labels"]

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_masks[idx],
            "labels": self.labels[idx]
        }


def create_personachat_collator(tokenizer):
    def collate_fn(batch):
        max_len = max(len(item["input_ids"]) for item in batch)

        input_ids = []
        attention_masks = []
        labels = []

        for item in batch:
            pad_len = max_len - len(item["input_ids"])

            input_ids.append(
                item["input_ids"]
                + [tokenizer.pad_token_id] * pad_len
            )
            attention_masks.append(
                item["attention_mask"] + [0] * pad_len
            )
            labels.append(
                item["labels"] + [-100] * pad_len
            )

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long)
        }

    return collate_fn


# Create datasets
gpt2_train_dataset = PersonaChatTorchDataset(gpt2_train_tokenized)
gpt2_validation_dataset = PersonaChatTorchDataset(gpt2_validation_tokenized)

qwen_train_dataset = PersonaChatTorchDataset(qwen_train_tokenized)
qwen_validation_dataset = PersonaChatTorchDataset(qwen_validation_tokenized)

# Create collators
gpt2_collator = create_personachat_collator(gpt2_tokenizer)
qwen_collator = create_personachat_collator(qwen_tokenizer)

# Initial batch size; we'll adjust it for individual trials if needed
BATCH_SIZE = 16
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

# GPT-2 data loaders
gpt2_train_loader = DataLoader(
    gpt2_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=gpt2_collator,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

gpt2_validation_loader = DataLoader(
    gpt2_validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=gpt2_collator,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

# Qwen data loaders
qwen_train_loader = DataLoader(
    qwen_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=qwen_collator,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

qwen_validation_loader = DataLoader(
    qwen_validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=qwen_collator,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

print("GPT-2 train batches:", len(gpt2_train_loader))
print("GPT-2 validation batches:", len(gpt2_validation_loader))
print("Qwen train batches:", len(qwen_train_loader))
print("Qwen validation batches:", len(qwen_validation_loader))

# Inspects one batch from each model
for model_name, loader in [
    ("GPT-2", gpt2_train_loader),
    ("Qwen", qwen_train_loader)
]:
    batch = next(iter(loader))
    print(f"\n{model_name} batch shapes:")
    for key, value in batch.items():
        print(f"{key}: {tuple(value.shape)}")

GPT-2 train batches: 8215
GPT-2 validation batches: 488
Qwen train batches: 8215
Qwen validation batches: 488

GPT-2 batch shapes:
input_ids: (16, 305)
attention_mask: (16, 305)
labels: (16, 305)

Qwen batch shapes:
input_ids: (16, 170)
attention_mask: (16, 170)
labels: (16, 170)


Loading GPT-2 Model and attaching LoRA with it

In [ ]:

import sys
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, TaskType, get_peft_model

# torchao package created a problem for compatibility issue with this peft. so turning this off
sys.modules["torchao"] = None

# Load GPT-2 for causal language modelling
gpt2_model = AutoModelForCausalLM.from_pretrained(
    GPT2_MODEL_NAME
)

# Configure padding
gpt2_model.config.pad_token_id = gpt2_tokenizer.pad_token_id

# Apply LoRA to GPT-2 attention projections
gpt2_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["c_attn"],
    bias="none"
)

gpt2_model = get_peft_model(
    gpt2_model,
    gpt2_lora_config
)

# Move model to GPU when available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpt2_model = gpt2_model.to(device)

print("Device:", device)
gpt2_model.print_trainable_parameters()
print("Padding token ID:", gpt2_model.config.pad_token_id)
print("Maximum context length:", gpt2_model.config.n_positions)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/peft/tuners/lora/layer.py:2760: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


Device: cuda
trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364
Padding token ID: 50256
Maximum context length: 1024


Getting the experiment folder ready in google drive for Gpt2

In [ ]:
import os

PERSONACHAT_GPT2_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2"
)

for trial_name in ["trial_1", "trial_2", "trial_3"]:
    trial_dir = os.path.join(PERSONACHAT_GPT2_DIR, trial_name)

    for subfolder in ["checkpoints", "results", "logs"]:
        os.makedirs(os.path.join(trial_dir, subfolder), exist_ok=True)

print("GPT-2 experiment folders are ready.")
print("Base directory:", PERSONACHAT_GPT2_DIR)

for trial_name in ["trial_1", "trial_2", "trial_3"]:
    print(f"  {trial_name}: checkpoints/, results/, logs/")

GPT-2 experiment folders are ready.
Base directory: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2
  trial_1: checkpoints/, results/, logs/
  trial_2: checkpoints/, results/, logs/
  trial_3: checkpoints/, results/, logs/


Configuring GPT-2 Trial 1 - turn level

Learning rate : 2e-4

Batch size: 16

Epoch: 1

In [ ]:
import random
import numpy as np
import torch
import os

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# GPT-2 Trial 1 configuration
GPT2_TRIAL = "trial_1"
GPT2_LEARNING_RATE = 2e-4
GPT2_EPOCHS = 1
GPT2_BATCH_SIZE = 16
GPT2_MAX_GRAD_NORM = 1.0
GPT2_LOG_STEPS = 100
GPT2_SAVE_STEPS = 1000

GPT2_TRIAL_DIR = os.path.join(
    PERSONACHAT_GPT2_DIR, GPT2_TRIAL
)
GPT2_CHECKPOINT_DIR = os.path.join(
    GPT2_TRIAL_DIR, "checkpoints"
)
GPT2_RESULTS_DIR = os.path.join(
    GPT2_TRIAL_DIR, "results"
)
GPT2_LOG_DIR = os.path.join(
    GPT2_TRIAL_DIR, "logs"
)

print("Trial:", GPT2_TRIAL)
print("Learning rate:", GPT2_LEARNING_RATE)
print("Epochs:", GPT2_EPOCHS)
print("Batch size:", GPT2_BATCH_SIZE)
print("Training examples:", len(gpt2_train_dataset))
print("Validation examples:", len(gpt2_validation_dataset))
print("Training batches per epoch:", len(gpt2_train_loader))
print("Checkpoint directory:", GPT2_CHECKPOINT_DIR)

Trial: trial_1
Learning rate: 0.0002
Epochs: 1
Batch size: 16
Training examples: 131438
Validation examples: 7801
Training batches per epoch: 8215
Checkpoint directory: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/trial_1/checkpoints


Training GPT-2 Trial 1

In [ ]:
import os
import csv
import json
import time
import torch
from transformers import get_linear_schedule_with_warmup
from tqdm.auto import tqdm

# Prepare optimizer and learning-rate scheduler
gpt2_model.config.use_cache = False

optimizer = torch.optim.AdamW(
    (p for p in gpt2_model.parameters() if p.requires_grad),
    lr=GPT2_LEARNING_RATE,
    weight_decay=0.01
)

total_training_steps = len(gpt2_train_loader) * GPT2_EPOCHS
warmup_steps = int(0.03 * total_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)

# Store step-level logs
log_path = os.path.join(GPT2_LOG_DIR, "training_log.csv")

with open(log_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["step", "epoch", "training_loss", "learning_rate"])

training_losses = []
global_step = 0
start_time = time.time()

def save_gpt2_checkpoint(step, final=False):
    checkpoint_name = "final" if final else f"step_{step}"
    checkpoint_path = os.path.join(
        GPT2_CHECKPOINT_DIR, checkpoint_name
    )
    os.makedirs(checkpoint_path, exist_ok=True)

    # Save the LoRA adapter and training state
    gpt2_model.save_pretrained(
        os.path.join(checkpoint_path, "adapter")
    )
    torch.save(
        {
            "global_step": step,
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "python_rng_state": random.getstate(),
            "numpy_rng_state": np.random.get_state(),
            "torch_rng_state": torch.get_rng_state(),
            "cuda_rng_state": (
                torch.cuda.get_rng_state_all()
                if torch.cuda.is_available() else None
            ),
        },
        os.path.join(checkpoint_path, "training_state.pt")
    )
    print(f"\nCheckpoint saved: {checkpoint_name}")

gpt2_model.train()

progress_bar = tqdm(
    total=total_training_steps,
    desc="GPT-2 Trial 1",
    unit="step"
)

for epoch in range(GPT2_EPOCHS):
    epoch_loss_sum = 0.0
    epoch_start = time.time()

    for batch in gpt2_train_loader:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        optimizer.zero_grad(set_to_none=True)

        outputs = gpt2_model(**batch)
        loss = outputs.loss

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss at step {global_step + 1}: "
                f"{loss.item()}"
            )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            gpt2_model.parameters(),
            GPT2_MAX_GRAD_NORM
        )
        optimizer.step()
        scheduler.step()

        global_step += 1
        batch_loss = loss.item()
        training_losses.append(batch_loss)
        epoch_loss_sum += batch_loss

        progress_bar.update(1)
        progress_bar.set_postfix(
            loss=f"{batch_loss:.4f}",
            lr=f"{scheduler.get_last_lr()[0]:.2e}"
        )

        if global_step % GPT2_LOG_STEPS == 0:
            with open(log_path, "a", newline="") as f:
                writer = csv.writer(f)
                writer.writerow([
                    global_step,
                    epoch + 1,
                    batch_loss,
                    scheduler.get_last_lr()[0]
                ])

        if (
            global_step % GPT2_SAVE_STEPS == 0
            and global_step < total_training_steps
        ):
            save_gpt2_checkpoint(global_step)

    mean_epoch_loss = epoch_loss_sum / len(gpt2_train_loader)

    print(
        f"\nEpoch {epoch + 1} complete | "
        f"Mean training loss: {mean_epoch_loss:.4f} | "
        f"Duration: {(time.time() - epoch_start) / 60:.1f} minutes"
    )

progress_bar.close()

# Evaluate on the complete validation set
gpt2_model.config.use_cache = False
gpt2_model.eval()

validation_loss_sum = 0.0
validation_batches = 0

with torch.no_grad():
    for batch in tqdm(
        gpt2_validation_loader,
        desc="Evaluating validation loss"
    ):
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }
        outputs = gpt2_model(**batch)
        validation_loss_sum += outputs.loss.item()
        validation_batches += 1

mean_validation_loss = validation_loss_sum / validation_batches

# Save final adapter and training state
save_gpt2_checkpoint(global_step, final=True)

trial_results = {
    "model": GPT2_MODEL_NAME,
    "trial": GPT2_TRIAL,
    "learning_rate": GPT2_LEARNING_RATE,
    "epochs": GPT2_EPOCHS,
    "batch_size": GPT2_BATCH_SIZE,
    "training_examples": len(gpt2_train_dataset),
    "validation_examples": len(gpt2_validation_dataset),
    "training_steps": global_step,
    "mean_training_loss": mean_epoch_loss,
    "validation_loss": mean_validation_loss,
    "validation_perplexity": (
        float(torch.exp(torch.tensor(mean_validation_loss)).item())
        if mean_validation_loss < 80 else None
    ),
    "duration_minutes": (time.time() - start_time) / 60
}

results_path = os.path.join(
    GPT2_RESULTS_DIR, "trial_1_results.json"
)
with open(results_path, "w") as f:
    json.dump(trial_results, f, indent=2)

# Restore cache setting for later generation
gpt2_model.config.use_cache = True

print("\nGPT-2 Trial 1 completed.")
print(json.dumps(trial_results, indent=2))
print("Results saved to:", results_path)

GPT-2 Trial 1:   0%|          | 0/8215 [00:00<?, ?step/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.



Checkpoint saved: step_1000

Checkpoint saved: step_2000

Checkpoint saved: step_3000

Checkpoint saved: step_4000

Checkpoint saved: step_5000

Checkpoint saved: step_6000

Checkpoint saved: step_7000

Checkpoint saved: step_8000

Epoch 1 complete | Mean training loss: 3.0702 | Duration: 134.2 minutes


Evaluating validation loss:   0%|          | 0/488 [00:00<?, ?it/s]


Checkpoint saved: final

GPT-2 Trial 1 completed.
{
  "model": "openai-community/gpt2",
  "trial": "trial_1",
  "learning_rate": 0.0002,
  "epochs": 1,
  "batch_size": 16,
  "training_examples": 131438,
  "validation_examples": 7801,
  "training_steps": 8215,
  "mean_training_loss": 3.0701788815215676,
  "validation_loss": 2.834535640771272,
  "validation_perplexity": 17.022493362426758,
  "duration_minutes": 138.01908258597055
}
Results saved to: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/trial_1/results/trial_1_results.json


PersonaChat - Preparing conversation-level data

Previously I did One example per dialogue turn - which is very costly and time consuming

Now preparing the data for One example per conversation- which is less costly and less time consuming

In [ ]:

def prepare_personachat_conversation_examples(dataset_split):
    examples = []

    for conversation in dataset_split:
        personality = conversation["personality"]
        utterances = conversation["utterances"]

        # Use the final turn from each conversation
        final_turn = utterances[-1]
        history = final_turn["history"]
        candidates = final_turn["candidates"]

        # Use the dataset's designated target candidate
        target_response = candidates[-1]

        prompt = (
            "Persona:\n"
            + "\n".join(f"- {fact}" for fact in personality)
            + "\n\nConversation:\n"
            + "\n".join(history)
            + "\nAssistant:"
        )

        examples.append({
            "prompt": prompt,
            "response": target_response
        })

    return examples


# Keep the original turn-level variables unchanged
persona_train_conversation_examples = (
    prepare_personachat_conversation_examples(
        persona_dataset["train"]
    )
)

persona_validation_conversation_examples = (
    prepare_personachat_conversation_examples(
        persona_dataset["validation"]
    )
)

print(
    "Training examples:",
    len(persona_train_conversation_examples)
)
print(
    "Validation examples:",
    len(persona_validation_conversation_examples)
)

print("\nSample prompt:\n")
print(persona_train_conversation_examples[0]["prompt"])

print("\nTarget response:")
print(persona_train_conversation_examples[0]["response"])

Training examples: 17878
Validation examples: 1000

Sample prompt:

Persona:
- i like to remodel homes .
- i like to go hunting .
- i like to shoot a bow .
- my favorite holiday is halloween .

Conversation:
hi , how are you doing ? i'm getting ready to do some cheetah chasing to stay in shape .
you must be very fast . hunting is one of my favorite hobbies .
i am ! for my hobby i like to do canning or some whittling .
i also remodel homes when i am not out bow hunting .
that's neat . when i was in high school i placed 6th in 100m dash !
that's awesome . do you have a favorite season or time of year ?
i do not . but i do have a favorite meat since that is all i eat exclusively .
what is your favorite meat to eat ?
i would have to say its prime rib . do you have any favorite foods ?
i like chicken or macaroni and cheese .
do you have anything planned for today ? i think i am going to do some canning .
i am going to watch football . what are you canning ?
i think i will can some jam . do 

Tokenizing data for GPT-2 and Qwen

In [ ]:

MAX_LENGTH = 320

def tokenize_personachat_gpt2(examples, tokenizer, max_length=MAX_LENGTH):
    input_ids_list = []
    attention_masks_list = []
    labels_list = []

    for example in examples:
        prompt_ids = tokenizer(
            example["prompt"],
            add_special_tokens=False
        )["input_ids"]

        response_ids = tokenizer(
            example["response"],
            add_special_tokens=False
        )["input_ids"]

        response_ids = response_ids[:max_length - 1]
        response_ids = response_ids + [tokenizer.eos_token_id]

        available_prompt_length = max_length - len(response_ids)
        prompt_ids = (
            prompt_ids[-available_prompt_length:]
            if available_prompt_length > 0 else []
        )

        input_ids = prompt_ids + response_ids
        labels = [-100] * len(prompt_ids) + response_ids.copy()

        input_ids_list.append(input_ids)
        attention_masks_list.append([1] * len(input_ids))
        labels_list.append(labels)

    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_masks_list,
        "labels": labels_list
    }


def tokenize_personachat_qwen(examples, tokenizer, max_length=MAX_LENGTH):
    input_ids_list = []
    attention_masks_list = []
    labels_list = []

    end_turn_id = tokenizer.convert_tokens_to_ids("<|im_end|>")

    if end_turn_id is None or end_turn_id == tokenizer.unk_token_id:
        raise ValueError("Could not find Qwen's <|im_end|> token.")

    for example in examples:
        prompt_parts = example["prompt"].split(
            "\n\nConversation:\n", 1
        )
        persona_text = prompt_parts[0].replace("Persona:\n", "", 1)
        conversation_text = prompt_parts[1].rsplit(
            "\nAssistant:", 1
        )[0]

        response_ids = tokenizer(
            example["response"],
            add_special_tokens=False
        )["input_ids"]

        response_ids = response_ids[:max_length - 1]
        response_ids = response_ids + [end_turn_id]

        prompt_budget = max_length - len(response_ids)
        history_lines = conversation_text.split("\n")

        while True:
            messages = [
                {
                    "role": "system",
                    "content": (
                        "You are a helpful conversational partner. "
                        "Stay consistent with this persona:\n"
                        + persona_text
                    )
                },
                {
                    "role": "user",
                    "content": (
                        "Continue this conversation naturally. "
                        "Conversation history:\n"
                        + "\n".join(history_lines)
                    )
                }
            ]

            encoded_prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True
            )

            # apply_chat_template returns BatchEncoding in this environment.
            prompt_ids = encoded_prompt["input_ids"]

            if len(prompt_ids) <= prompt_budget:
                break

            # Remove the oldest history turn while preserving the persona.
            if len(history_lines) > 1:
                history_lines.pop(0)
            else:
                # Rare edge case: shorten the prompt to fit the sequence limit.
                prompt_ids = prompt_ids[-prompt_budget:] if prompt_budget > 0 else []
                break

        input_ids = prompt_ids + response_ids
        labels = [-100] * len(prompt_ids) + response_ids.copy()

        input_ids_list.append(input_ids)
        attention_masks_list.append([1] * len(input_ids))
        labels_list.append(labels)

    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_masks_list,
        "labels": labels_list
    }


# Tokenize the same conversation-level examples for both models
gpt2_train_conversation_tokenized = tokenize_personachat_gpt2(
    persona_train_conversation_examples, gpt2_tokenizer
)
gpt2_validation_conversation_tokenized = tokenize_personachat_gpt2(
    persona_validation_conversation_examples, gpt2_tokenizer
)

qwen_train_conversation_tokenized = tokenize_personachat_qwen(
    persona_train_conversation_examples, qwen_tokenizer
)
qwen_validation_conversation_tokenized = tokenize_personachat_qwen(
    persona_validation_conversation_examples, qwen_tokenizer
)

# Verify dataset counts, sequence lengths, and response labels
print("GPT-2 train/validation:",
      len(gpt2_train_conversation_tokenized["input_ids"]),
      len(gpt2_validation_conversation_tokenized["input_ids"]))

print("Qwen train/validation:",
      len(qwen_train_conversation_tokenized["input_ids"]),
      len(qwen_validation_conversation_tokenized["input_ids"]))

print("GPT-2 first sequence length:",
      len(gpt2_train_conversation_tokenized["input_ids"][0]))
print("GPT-2 first response tokens:",
      sum(x != -100 for x in gpt2_train_conversation_tokenized["labels"][0]))

print("Qwen first sequence length:",
      len(qwen_train_conversation_tokenized["input_ids"][0]))
print("Qwen first prompt tokens:",
      sum(x == -100 for x in qwen_train_conversation_tokenized["labels"][0]))
print("Qwen first response tokens:",
      sum(x != -100 for x in qwen_train_conversation_tokenized["labels"][0]))

print("Maximum GPT-2 sequence length:",
      max(len(x) for x in gpt2_train_conversation_tokenized["input_ids"]))
print("Maximum Qwen sequence length:",
      max(len(x) for x in qwen_train_conversation_tokenized["input_ids"]))

assert len(gpt2_train_conversation_tokenized["input_ids"]) == 17878
assert len(gpt2_validation_conversation_tokenized["input_ids"]) == 1000
assert len(qwen_train_conversation_tokenized["input_ids"]) == 17878
assert len(qwen_validation_conversation_tokenized["input_ids"]) == 1000

assert max(len(x) for x in gpt2_train_conversation_tokenized["input_ids"]) <= MAX_LENGTH
assert max(len(x) for x in qwen_train_conversation_tokenized["input_ids"]) <= MAX_LENGTH

print("Tokenization checks passed.")

GPT-2 train/validation: 17878 1000
Qwen train/validation: 17878 1000
GPT-2 first sequence length: 283
GPT-2 first response tokens: 18
Qwen first sequence length: 289
Qwen first prompt tokens: 272
Qwen first response tokens: 17
Maximum GPT-2 sequence length: 320
Maximum Qwen sequence length: 320
Tokenization checks passed.


Creating conversation-level PyTorch datasets for GPT-2 and Qwen

In [ ]:

import torch
from torch.utils.data import Dataset

class PersonaChatConversationDataset(Dataset):
    def __init__(self, tokenized_data):
        self.input_ids = tokenized_data["input_ids"]
        self.attention_mask = tokenized_data["attention_mask"]
        self.labels = tokenized_data["labels"]

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, index):
        return {
            "input_ids": torch.tensor(
                self.input_ids[index], dtype=torch.long
            ),
            "attention_mask": torch.tensor(
                self.attention_mask[index], dtype=torch.long
            ),
            "labels": torch.tensor(
                self.labels[index], dtype=torch.long
            )
        }


gpt2_train_conversation_dataset = PersonaChatConversationDataset(
    gpt2_train_conversation_tokenized
)
gpt2_validation_conversation_dataset = PersonaChatConversationDataset(
    gpt2_validation_conversation_tokenized
)

qwen_train_conversation_dataset = PersonaChatConversationDataset(
    qwen_train_conversation_tokenized
)
qwen_validation_conversation_dataset = PersonaChatConversationDataset(
    qwen_validation_conversation_tokenized
)

print("GPT-2 dataset sizes:",
      len(gpt2_train_conversation_dataset),
      len(gpt2_validation_conversation_dataset))

print("Qwen dataset sizes:",
      len(qwen_train_conversation_dataset),
      len(qwen_validation_conversation_dataset))

print("GPT-2 tensor shapes:",
      {key: tuple(value.shape) for key, value
       in gpt2_train_conversation_dataset[0].items()})

print("Qwen tensor shapes:",
      {key: tuple(value.shape) for key, value
       in qwen_train_conversation_dataset[0].items()})

GPT-2 dataset sizes: 17878 1000
Qwen dataset sizes: 17878 1000
GPT-2 tensor shapes: {'input_ids': (283,), 'attention_mask': (283,), 'labels': (283,)}
Qwen tensor shapes: {'input_ids': (289,), 'attention_mask': (289,), 'labels': (289,)}


Creating Collator and DataLoaders

In [ ]:

from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

GPT2_BATCH_SIZE = 16
QWEN_BATCH_SIZE = 16
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()


class PersonaChatCollator:
    def __init__(self, pad_token_id):
        self.pad_token_id = pad_token_id

    def __call__(self, batch):
        input_ids = pad_sequence(
            [item["input_ids"] for item in batch],
            batch_first=True,
            padding_value=self.pad_token_id
        )
        attention_mask = pad_sequence(
            [item["attention_mask"] for item in batch],
            batch_first=True,
            padding_value=0
        )
        labels = pad_sequence(
            [item["labels"] for item in batch],
            batch_first=True,
            padding_value=-100
        )
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }


gpt2_collator = PersonaChatCollator(gpt2_tokenizer.pad_token_id)
qwen_collator = PersonaChatCollator(qwen_tokenizer.pad_token_id)

# Temporary batch-check loaders only; training uses seeded loaders.
gpt2_train_conversation_loader = DataLoader(
    gpt2_train_conversation_dataset,
    batch_size=GPT2_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=gpt2_collator
)
gpt2_validation_conversation_loader = DataLoader(
    gpt2_validation_conversation_dataset,
    batch_size=GPT2_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=gpt2_collator
)

qwen_train_conversation_loader = DataLoader(
    qwen_train_conversation_dataset,
    batch_size=QWEN_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=qwen_collator
)
qwen_validation_conversation_loader = DataLoader(
    qwen_validation_conversation_dataset,
    batch_size=QWEN_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=qwen_collator
)

# Check batch shapes without starting training
gpt2_batch = next(iter(gpt2_train_conversation_loader))
qwen_batch = next(iter(qwen_train_conversation_loader))

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("GPT-2 batch size:", gpt2_batch["input_ids"].shape[0])
print("GPT-2 batch shapes:",
      {key: tuple(value.shape) for key, value in gpt2_batch.items()})
print("Qwen batch size:", qwen_batch["input_ids"].shape[0])
print("Qwen batch shapes:",
      {key: tuple(value.shape) for key, value in qwen_batch.items()})

print("GPT-2 training batches per epoch:",
      len(gpt2_train_conversation_loader))
print("Qwen training batches per epoch:",
      len(qwen_train_conversation_loader))

GPU available: True
GPU: Tesla T4
GPT-2 batch size: 16
GPT-2 batch shapes: {'input_ids': (16, 312), 'attention_mask': (16, 312), 'labels': (16, 312)}
Qwen batch size: 16
Qwen batch shapes: {'input_ids': (16, 306), 'attention_mask': (16, 306), 'labels': (16, 306)}
GPT-2 training batches per epoch: 1118
Qwen training batches per epoch: 1118


Creating conversation level trial directory

In [ ]:

import inspect
import os

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Not available")

print("\nExisting training helpers:")
for name in ["make_epoch_loader", "save_checkpoint"]:
    if name in globals() and callable(globals()[name]):
        print(f"{name}: available")
        try:
            print(inspect.getsource(globals()[name]))
        except (OSError, TypeError):
            print("Source unavailable in this runtime.")
    else:
        print(f"{name}: not found")

GPT2_CONVERSATION_TRIAL1_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "PersonaChat_GPT2/trial_1_conversation_level"
)

os.makedirs(GPT2_CONVERSATION_TRIAL1_DIR, exist_ok=True)

print("\nConversation-level output directory:")
print(GPT2_CONVERSATION_TRIAL1_DIR)
print("Directory exists:", os.path.isdir(GPT2_CONVERSATION_TRIAL1_DIR))

GPU: Tesla T4

Existing training helpers:
make_epoch_loader: not found
save_checkpoint: not found

Conversation-level output directory:
/content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/trial_1_conversation_level
Directory exists: True


In [ ]:

import os
import torch
import transformers
from peft import PeftModel, LoraConfig, get_peft_model

required_variables = [
    "GPT2_MODEL_NAME",
    "gpt2_tokenizer",
    "gpt2_train_conversation_dataset",
    "gpt2_validation_conversation_dataset",
    "gpt2_collator",
    "GPT2_CONVERSATION_TRIAL1_DIR",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Not available")
print("Missing variables:", missing_variables)

if not missing_variables:
    print("GPT-2 model:", GPT2_MODEL_NAME)
    print("Training examples:", len(gpt2_train_conversation_dataset))
    print("Validation examples:", len(gpt2_validation_conversation_dataset))
    print("Tokenizer vocabulary:", len(gpt2_tokenizer))
    print("Padding token ID:", gpt2_tokenizer.pad_token_id)
    print("EOS token ID:", gpt2_tokenizer.eos_token_id)
    print("Output directory:", GPT2_CONVERSATION_TRIAL1_DIR)
    print("Directory exists:", os.path.isdir(GPT2_CONVERSATION_TRIAL1_DIR))
    assert len(gpt2_train_conversation_dataset) == 17878
    assert len(gpt2_validation_conversation_dataset) == 1000
    assert gpt2_tokenizer.pad_token_id is not None
    print("\nPre-training checks passed.")
else:
    print("Please restore the missing variables before proceeding.")

PyTorch: 2.11.0+cu130
Transformers: 5.18.0
GPU: Tesla T4
Missing variables: []
GPT-2 model: openai-community/gpt2
Training examples: 17878
Validation examples: 1000
Tokenizer vocabulary: 50257
Padding token ID: 50256
EOS token ID: 50256
Output directory: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/trial_1_conversation_level
Directory exists: True

Pre-training checks passed.


Evaluatng GPT-2 Base Model

In [ ]:

import os
import csv
import time
import torch
import pandas as pd
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM

# Separate baseline output directory
GPT2_BASELINE_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "PersonaChat_GPT2/base_model_evaluation"
)
os.makedirs(GPT2_BASELINE_DIR, exist_ok=True)

GPT2_BASELINE_CSV = os.path.join(
    GPT2_BASELINE_DIR, "gpt2_base_validation_generations.csv"
)

# Load the un-fine-tuned base model
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpt2_base_model = AutoModelForCausalLM.from_pretrained(GPT2_MODEL_NAME)
gpt2_base_model.config.pad_token_id = gpt2_tokenizer.pad_token_id
gpt2_base_model.config.use_cache = True
gpt2_base_model.to(DEVICE)
gpt2_base_model.eval()

# Use left padding for batched generation
original_padding_side = gpt2_tokenizer.padding_side
gpt2_tokenizer.padding_side = "left"

# Ensure reproducible greedy generation
generation_batch_size = 8
max_new_tokens = 40
generation_rows = []
generation_start = time.time()

try:
    for start in range(0, len(persona_validation_conversation_examples), generation_batch_size):
        examples = persona_validation_conversation_examples[
            start:start + generation_batch_size
        ]
        prompts = [example["prompt"] for example in examples]

        encoded = gpt2_tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=280,
        )
        encoded = {key: value.to(DEVICE) for key, value in encoded.items()}
        prompt_width = encoded["input_ids"].shape[1]

        with torch.inference_mode():
            generated = gpt2_base_model.generate(
                **encoded,
                do_sample=False,
                max_new_tokens=max_new_tokens,
                pad_token_id=gpt2_tokenizer.pad_token_id,
                eos_token_id=gpt2_tokenizer.eos_token_id,
            )

        # Decode only newly generated tokens
        continuations = generated[:, prompt_width:]
        replies = gpt2_tokenizer.batch_decode(
            continuations, skip_special_tokens=True
        )

        for example, reply in zip(examples, replies):
            generation_rows.append({
                "prompt": example["prompt"],
                "reference_response": example["response"],
                "generated_response": reply.strip(),
            })

        if (start + len(examples)) % 100 == 0 or start + len(examples) == len(persona_validation_conversation_examples):
            print(
                f"Generated {start + len(examples)}/"
                f"{len(persona_validation_conversation_examples)} replies"
            )
finally:
    gpt2_tokenizer.padding_side = original_padding_side

# Save baseline generations
pd.DataFrame(generation_rows).to_csv(
    GPT2_BASELINE_CSV, index=False, quoting=csv.QUOTE_MINIMAL
)

elapsed_minutes = (time.time() - generation_start) / 60
print("\n===== BASE GPT-2 GENERATION RESULTS =====")
print("Replies generated:", len(generation_rows))
print("Generation time (minutes):", round(elapsed_minutes, 2))
print("CSV:", GPT2_BASELINE_CSV)
print("\nSample outputs:")
for row in generation_rows[:3]:
    print("\nPROMPT:", row["prompt"][-300:])
    print("REFERENCE:", row["reference_response"])
    print("GENERATED:", row["generated_response"])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Generated 200/1000 replies
Generated 400/1000 replies
Generated 600/1000 replies
Generated 800/1000 replies
Generated 1000/1000 replies

===== BASE GPT-2 GENERATION RESULTS =====
Replies generated: 1000
Generation time (minutes): 1.61
CSV: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/base_model_evaluation/gpt2_base_validation_generations.csv

Sample outputs:

PROMPT:  double , it is so much fun and hard work .
yes i bet you can get hurt . my wife works and i stay at home
nice , i only have one parent so now i help out my mom .
i bet she appreciates that very much .
she raised me right , i am just like her .
my dad was always busy working at home depot
Assistant:
REFERENCE: now that i am older home depot is my toy r us .
GENERATED: - i work in the office , i work in the kitchen , i work in the kitchen , i work in the kitchen , i work in the kitchen , i work in the kitchen , i work

PROMPT:  people .
i didn't think about the crowd aspect of wrestling . i do not like crowd

Calculating base GPT-2 validation loss and perplexity

In [ ]:

import math
import time
import torch
from torch.utils.data import DataLoader

gpt2_base_model.eval()

base_validation_loader = DataLoader(
    gpt2_validation_conversation_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=gpt2_collator,
)

total_nll = 0.0
total_target_tokens = 0
validation_start = time.time()

with torch.inference_mode():
    for batch_index, batch in enumerate(base_validation_loader):
        batch = {
            key: value.to(DEVICE, non_blocking=True)
            for key, value in batch.items()
        }

        outputs = gpt2_base_model(**batch)
        target_tokens = int((batch["labels"][:, 1:] != -100).sum().item())

        # GPT-2 causal LM shifts labels internally, so weight loss
        # by the number of non-masked target tokens after the shift.
        if target_tokens > 0:
            total_nll += outputs.loss.item() * target_tokens
            total_target_tokens += target_tokens

        if (batch_index + 1) % 20 == 0:
            print(
                f"Validated {min((batch_index + 1) * 16, 1000)}/1000 "
                "conversations"
            )

base_validation_loss = total_nll / total_target_tokens
base_validation_perplexity = math.exp(base_validation_loss)

base_validation_results = {
    "model": "GPT-2",
    "stage": "base",
    "validation_conversations": len(gpt2_validation_conversation_dataset),
    "target_tokens": total_target_tokens,
    "validation_loss": base_validation_loss,
    "validation_perplexity": base_validation_perplexity,
    "validation_time_minutes": (time.time() - validation_start) / 60,
}

print("\n===== BASE GPT-2 VALIDATION METRICS =====")
for key, value in base_validation_results.items():
    print(
        f"{key}: {value:.4f}"
        if isinstance(value, float)
        else f"{key}: {value}"
    )

Validated 320/1000 conversations
Validated 640/1000 conversations
Validated 960/1000 conversations

===== BASE GPT-2 VALIDATION METRICS =====
model: GPT-2
stage: base
validation_conversations: 1000
target_tokens: 12552
validation_loss: 4.1830
validation_perplexity: 65.5595
validation_time_minutes: 0.5767


In [ ]:

%pip install -q rouge-score bert-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.5 MB/s eta 0:00:00


Calculating all base GPT-2 generation metrics

In [ ]:

import os
import json
import time
import pandas as pd
import nltk
import torch

from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bert_score


# Paths and input checks

GPT2_BASELINE_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "PersonaChat_GPT2/base_model_evaluation"
)
GPT2_BASELINE_CSV = os.path.join(
    GPT2_BASELINE_DIR, "gpt2_base_validation_generations.csv"
)

assert os.path.isfile(GPT2_BASELINE_CSV), (
    f"Baseline CSV not found: {GPT2_BASELINE_CSV}"
)

df = pd.read_csv(GPT2_BASELINE_CSV).fillna("")
required_columns = [
    "prompt", "reference_response", "generated_response"
]
assert all(column in df.columns for column in required_columns), (
    f"CSV must contain columns: {required_columns}"
)
assert len(df) == 1000, f"Expected 1000 rows, found {len(df)}"

references = df["reference_response"].astype(str).tolist()
predictions = df["generated_response"].astype(str).tolist()


# BLEU — corpus-level word overlap

references_tokens = [text.lower().split() for text in references]
predictions_tokens = [text.lower().split() for text in predictions]

bleu = corpus_bleu(
    [[reference] for reference in references_tokens],
    predictions_tokens,
    smoothing_function=SmoothingFunction().method1,
)


# ROUGE-L — F-measure

rouge = rouge_scorer.RougeScorer(
    ["rougeL"], use_stemmer=True
)

rouge_l_scores = [
    rouge.score(reference, prediction)["rougeL"].fmeasure
    for reference, prediction in zip(references, predictions)
]
mean_rouge_l = sum(rouge_l_scores) / len(rouge_l_scores)


# Distinct-1 and Distinct-2 — unique n-grams / total n-grams

all_unigrams = []
all_bigrams = []

for prediction in predictions_tokens:
    all_unigrams.extend(prediction)
    all_bigrams.extend(
        tuple(prediction[i:i + 2])
        for i in range(max(0, len(prediction) - 1))
    )

distinct_1 = (
    len(set(all_unigrams)) / len(all_unigrams)
    if all_unigrams else 0.0
)
distinct_2 = (
    len(set(all_bigrams)) / len(all_bigrams)
    if all_bigrams else 0.0
)


# Average reply length (words)

reply_lengths = [len(text.split()) for text in predictions]
average_reply_length = sum(reply_lengths) / len(reply_lengths)


# BERTScore — semantic similarity
# Use DistilBERT to keep this evaluation manageable on a T4.

device = "cuda" if torch.cuda.is_available() else "cpu"
bert_start = time.time()

P, R, F1 = bert_score(
    cands=predictions,
    refs=references,
    model_type="distilbert-base-uncased",
    lang="en",
    batch_size=16,
    device=device,
    verbose=True,
    rescale_with_baseline=False,
)

mean_bert_precision = P.mean().item()
mean_bert_recall = R.mean().item()
mean_bert_f1 = F1.mean().item()
bert_time_minutes = (time.time() - bert_start) / 60


# Save per-example and aggregate results

df["bleu_sentence_not_used"] = ""  # Corpus BLEU is reported below.
df["rougeL_f1"] = rouge_l_scores
df["bert_score_precision"] = P.tolist()
df["bert_score_recall"] = R.tolist()
df["bert_score_f1"] = F1.tolist()
df["generated_word_count"] = reply_lengths

per_example_path = os.path.join(
    GPT2_BASELINE_DIR, "gpt2_base_generation_metrics_per_example.csv"
)
df.to_csv(per_example_path, index=False)

metrics = {
    "model": "GPT-2",
    "stage": "base",
    "number_of_validation_conversations": len(df),
    "corpus_bleu": bleu,
    "mean_rougeL_f1": mean_rouge_l,
    "mean_bert_score_precision": mean_bert_precision,
    "mean_bert_score_recall": mean_bert_recall,
    "mean_bert_score_f1": mean_bert_f1,
    "distinct_1": distinct_1,
    "distinct_2": distinct_2,
    "average_reply_length_words": average_reply_length,
    "median_reply_length_words": float(pd.Series(reply_lengths).median()),
    "empty_reply_count": sum(not text.strip() for text in predictions),
    "bert_score_time_minutes": bert_time_minutes,
}

summary_path = os.path.join(
    GPT2_BASELINE_DIR, "gpt2_base_generation_metrics_summary.json"
)
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("\n===== BASE GPT-2 GENERATION METRICS =====")
for name, value in metrics.items():
    if isinstance(value, float):
        print(f"{name}: {value:.4f}")
    else:
        print(f"{name}: {value}")

print("\nPer-example metrics CSV:", per_example_path)
print("Summary JSON:", summary_path)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/111 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/63 [00:00<?, ?it/s]

done in 4.97 seconds, 201.36 sentences/sec

===== BASE GPT-2 GENERATION METRICS =====
model: GPT-2
stage: base
number_of_validation_conversations: 1000
corpus_bleu: 0.0019
mean_rougeL_f1: 0.0654
mean_bert_score_precision: 0.6548
mean_bert_score_recall: 0.6763
mean_bert_score_f1: 0.6651
distinct_1: 0.0312
distinct_2: 0.0949
average_reply_length_words: 33.7790
median_reply_length_words: 34.0000
empty_reply_count: 0
bert_score_time_minutes: 0.1585

Per-example metrics CSV: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/base_model_evaluation/gpt2_base_generation_metrics_per_example.csv
Summary JSON: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/base_model_evaluation/gpt2_base_generation_metrics_summary.json


Configuring GPT-2 + LoRA Trail 1

Learning rate: 2e-4

Epoch: 1

Batch size: 16

Maximum sequence length: 320

Gradient clipping 1.0

In [ ]:

import os
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from torch.optim import AdamW

# Trial 1 configuration — conversation-level examples
GPT2_TRIAL_NAME = "gpt2_personachat_conversation_trial1"
GPT2_LEARNING_RATE = 2e-4
GPT2_EPOCHS = 1
GPT2_BATCH_SIZE = 16
GPT2_MAX_GRAD_NORM = 1.0
GPT2_CHECKPOINT_EVERY = 250
GPT2_MAX_LENGTH = 320
GPT2_SEED = 42

GPT2_CONVERSATION_TRIAL1_DIR = os.path.abspath(
    GPT2_CONVERSATION_TRIAL1_DIR
)
os.makedirs(GPT2_CONVERSATION_TRIAL1_DIR, exist_ok=True)

# Reproducibility
torch.manual_seed(GPT2_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GPT2_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the base GPT-2 model
gpt2_conversation_model = AutoModelForCausalLM.from_pretrained(
    GPT2_MODEL_NAME
)
gpt2_conversation_model.config.pad_token_id = gpt2_tokenizer.pad_token_id
gpt2_conversation_model.config.use_cache = False

# LoRA configuration
gpt2_conversation_lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["c_attn", "c_proj"],
    bias="none",
)

gpt2_conversation_model = get_peft_model(
    gpt2_conversation_model,
    gpt2_conversation_lora_config
)
gpt2_conversation_model.to(DEVICE)

# Optimizer
gpt2_conversation_optimizer = AdamW(
    [p for p in gpt2_conversation_model.parameters() if p.requires_grad],
    lr=GPT2_LEARNING_RATE,
    weight_decay=0.01,
)

# Training size and step counts
GPT2_STEPS_PER_EPOCH = (
    len(gpt2_train_conversation_dataset) + GPT2_BATCH_SIZE - 1
) // GPT2_BATCH_SIZE
GPT2_TOTAL_STEPS = GPT2_STEPS_PER_EPOCH * GPT2_EPOCHS

print("Model configured successfully.")
print("Device:", DEVICE)
print("Trainable parameters:")
gpt2_conversation_model.print_trainable_parameters()
print("Training examples:", len(gpt2_train_conversation_dataset))
print("Validation examples:", len(gpt2_validation_conversation_dataset))
print("Batch size:", GPT2_BATCH_SIZE)
print("Steps per epoch:", GPT2_STEPS_PER_EPOCH)
print("Total planned optimizer steps:", GPT2_TOTAL_STEPS)
print("Learning rate:", GPT2_LEARNING_RATE)
print("Output directory:", GPT2_CONVERSATION_TRIAL1_DIR)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model configured successfully.
Device: cuda
Trainable parameters:
trainable params: 811,008 || all params: 125,250,816 || trainable%: 0.6475
Training examples: 17878
Validation examples: 1000
Batch size: 16
Steps per epoch: 1118
Total planned optimizer steps: 1118
Learning rate: 0.0002
Output directory: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/trial_1_conversation_level


GPT-2 CONVERSATION-LEVEL TRIAL 1 Training

In [ ]:

import os
import time
import math
import torch
from torch.utils.data import DataLoader
from torch.nn.utils import clip_grad_norm_


# Deterministic DataLoader: recreates the same order on resume

def make_epoch_loader(epoch_index):
    generator = torch.Generator()
    generator.manual_seed(GPT2_SEED + epoch_index)

    return DataLoader(
        gpt2_train_conversation_dataset,
        batch_size=GPT2_BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        collate_fn=gpt2_collator,
    )



# Checkpoint paths and mixed precision

GPT2_CHECKPOINT_PATH = os.path.join(
    GPT2_CONVERSATION_TRIAL1_DIR,
    "training_checkpoint.pt",
)
GPT2_FINAL_ADAPTER_DIR = os.path.join(
    GPT2_CONVERSATION_TRIAL1_DIR,
    "final_adapter",
)

use_amp = DEVICE.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

global_step = 0
start_epoch = 0
start_batch = 0
loss_history = []
elapsed_before = 0.0


# Resume from checkpoint, if present

if os.path.isfile(GPT2_CHECKPOINT_PATH):
    checkpoint = torch.load(
        GPT2_CHECKPOINT_PATH,
        map_location=DEVICE,
        weights_only=False,
    )

    gpt2_conversation_model.load_state_dict(checkpoint["model_state"])
    gpt2_conversation_optimizer.load_state_dict(checkpoint["optimizer_state"])

    if "scaler_state" in checkpoint:
        scaler.load_state_dict(checkpoint["scaler_state"])

    global_step = checkpoint["global_step"]
    start_epoch = checkpoint["next_epoch"]
    start_batch = checkpoint["next_batch"]
    loss_history = checkpoint.get("loss_history", [])
    elapsed_before = checkpoint.get("elapsed_seconds", 0.0)

    torch.set_rng_state(checkpoint["torch_rng_state"].cpu())
    if torch.cuda.is_available() and checkpoint.get("cuda_rng_state") is not None:
        torch.cuda.set_rng_state_all(checkpoint["cuda_rng_state"])

    print(
        f"Resuming at epoch {start_epoch + 1}, "
        f"batch {start_batch + 1}, global step {global_step}."
    )
else:
    print("No existing training checkpoint. Starting a fresh run.")


# Atomic checkpoint saving

def save_checkpoint(next_epoch, next_batch, elapsed_seconds):
    checkpoint_data = {
        "model_state": gpt2_conversation_model.state_dict(),
        "optimizer_state": gpt2_conversation_optimizer.state_dict(),
        "scaler_state": scaler.state_dict(),
        "global_step": global_step,
        "next_epoch": next_epoch,
        "next_batch": next_batch,
        "loss_history": loss_history,
        "elapsed_seconds": elapsed_seconds,
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else None
        ),
        "trial_name": GPT2_TRIAL_NAME,
        "learning_rate": GPT2_LEARNING_RATE,
        "batch_size": GPT2_BATCH_SIZE,
        "seed": GPT2_SEED,
    }

    temporary_path = GPT2_CHECKPOINT_PATH + ".tmp"
    torch.save(checkpoint_data, temporary_path)
    os.replace(temporary_path, GPT2_CHECKPOINT_PATH)



# Training

gpt2_conversation_model.train()
run_start = time.time()

for epoch_index in range(start_epoch, GPT2_EPOCHS):
    train_loader = make_epoch_loader(epoch_index)
    batch_offset = start_batch if epoch_index == start_epoch else 0

    for batch_index, batch in enumerate(train_loader):
        if batch_index < batch_offset:
            continue

        batch = {
            key: value.to(DEVICE, non_blocking=True)
            for key, value in batch.items()
        }

        gpt2_conversation_optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=use_amp,
        ):
            outputs = gpt2_conversation_model(**batch)
            loss = outputs.loss

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss at epoch {epoch_index + 1}, "
                f"batch {batch_index + 1}: {loss.item()}"
            )

        scaler.scale(loss).backward()
        scaler.unscale_(gpt2_conversation_optimizer)
        clip_grad_norm_(
            gpt2_conversation_model.parameters(),
            GPT2_MAX_GRAD_NORM,
        )
        scaler.step(gpt2_conversation_optimizer)
        scaler.update()

        global_step += 1
        loss_history.append(float(loss.detach().cpu()))

        if global_step % 25 == 0:
            recent_loss = sum(loss_history[-25:]) / len(loss_history[-25:])
            elapsed = elapsed_before + time.time() - run_start
            print(
                f"Epoch {epoch_index + 1}/{GPT2_EPOCHS} | "
                f"Step {global_step}/{GPT2_TOTAL_STEPS} | "
                f"Batch loss (last 25 avg): {recent_loss:.4f} | "
                f"Elapsed: {elapsed / 60:.1f} min"
            )

        next_epoch = epoch_index
        next_batch = batch_index + 1

        if next_batch >= len(train_loader):
            next_epoch = epoch_index + 1
            next_batch = 0

        if (
            global_step % GPT2_CHECKPOINT_EVERY == 0
            or next_epoch >= GPT2_EPOCHS
        ):
            elapsed = elapsed_before + time.time() - run_start
            save_checkpoint(next_epoch, next_batch, elapsed)
            print(
                f"Checkpoint saved at step {global_step}; "
                f"next position: epoch {next_epoch + 1}, "
                f"batch {next_batch + 1}."
            )

    start_batch = 0

elapsed_total = elapsed_before + time.time() - run_start


# Save final LoRA adapter and tokenizer

os.makedirs(GPT2_FINAL_ADAPTER_DIR, exist_ok=True)
gpt2_conversation_model.save_pretrained(GPT2_FINAL_ADAPTER_DIR)
gpt2_tokenizer.save_pretrained(GPT2_FINAL_ADAPTER_DIR)


# Validation loss and perplexity

gpt2_conversation_model.eval()
validation_loss_sum = 0.0
validation_batches = 0

validation_loader = DataLoader(
    gpt2_validation_conversation_dataset,
    batch_size=GPT2_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=gpt2_collator,
)

with torch.inference_mode():
    for batch in validation_loader:
        batch = {
            key: value.to(DEVICE, non_blocking=True)
            for key, value in batch.items()
        }

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=use_amp,
        ):
            outputs = gpt2_conversation_model(**batch)

        if torch.isfinite(outputs.loss):
            validation_loss_sum += outputs.loss.item()
            validation_batches += 1
        else:
            raise RuntimeError("Non-finite validation loss detected.")

validation_loss = validation_loss_sum / validation_batches
validation_perplexity = math.exp(validation_loss)

print("\n========== GPT-2 CONVERSATION-LEVEL TRIAL 1 ==========")
print("Training examples:", len(gpt2_train_conversation_dataset))
print("Validation examples:", len(gpt2_validation_conversation_dataset))
print("Optimizer steps completed:", global_step)
print("Mean training loss:", round(sum(loss_history) / len(loss_history), 4))
print("Validation loss:", round(validation_loss, 4))
print("Validation perplexity:", round(validation_perplexity, 4))
print("Elapsed training time (minutes):", round(elapsed_total / 60, 2))
print("Final adapter:", GPT2_FINAL_ADAPTER_DIR)
print("Training checkpoint:", GPT2_CHECKPOINT_PATH)
print("======================================================")

gpt2_conversation_model.train()

No existing training checkpoint. Starting a fresh run.
Epoch 1/1 | Step 25/1118 | Batch loss (last 25 avg): 3.9987 | Elapsed: 0.2 min
Epoch 1/1 | Step 50/1118 | Batch loss (last 25 avg): 3.5926 | Elapsed: 0.5 min
Epoch 1/1 | Step 75/1118 | Batch loss (last 25 avg): 3.5099 | Elapsed: 0.7 min
Epoch 1/1 | Step 100/1118 | Batch loss (last 25 avg): 3.4168 | Elapsed: 0.9 min
Epoch 1/1 | Step 125/1118 | Batch loss (last 25 avg): 3.3730 | Elapsed: 1.1 min
Epoch 1/1 | Step 150/1118 | Batch loss (last 25 avg): 3.4652 | Elapsed: 1.3 min
Epoch 1/1 | Step 175/1118 | Batch loss (last 25 avg): 3.3424 | Elapsed: 1.5 min
Epoch 1/1 | Step 200/1118 | Batch loss (last 25 avg): 3.3726 | Elapsed: 1.7 min
Epoch 1/1 | Step 225/1118 | Batch loss (last 25 avg): 3.4349 | Elapsed: 1.9 min
Epoch 1/1 | Step 250/1118 | Batch loss (last 25 avg): 3.3625 | Elapsed: 2.1 min
Checkpoint saved at step 250; next position: epoch 1, batch 251.
Epoch 1/1 | Step 275/1118 | Batch loss (last 25 avg): 3.3345 | Elapsed: 2.3 min
Epo

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GPT2LMHeadModel(
      (transformer): GPT2Model(
        (wte): Embedding(50257, 768)
        (wpe): Embedding(1024, 768)
        (drop): Dropout(p=0.1, inplace=False)
        (h): ModuleList(
          (0-11): 12 x GPT2Block(
            (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (attn): GPT2Attention(
              (c_attn): lora.Linear(
                (base_layer): Conv1D(nf=2304, nx=768)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=768, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2304, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
          

GPT-2 + LoRA Trial 1 generation and evaluation

In [ ]:

import os
import json
import time
import gc
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bert_score


# Paths

GPT2_TRIAL1_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "PersonaChat_GPT2/trial_1_conversation_level"
)

GPT2_ADAPTER_DIR = os.path.join(
    GPT2_TRIAL1_DIR, "final_adapter"
)

GPT2_BASELINE_CSV = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "PersonaChat_GPT2/base_model_evaluation/"
    "gpt2_base_validation_generations.csv"
)

GPT2_TRIAL1_GENERATIONS_CSV = os.path.join(
    GPT2_TRIAL1_DIR,
    "gpt2_lora_trial1_validation_generations.csv"
)

GPT2_TRIAL1_METRICS_CSV = os.path.join(
    GPT2_TRIAL1_DIR,
    "gpt2_lora_trial1_generation_metrics_per_example.csv"
)

GPT2_TRIAL1_SUMMARY_JSON = os.path.join(
    GPT2_TRIAL1_DIR,
    "gpt2_lora_trial1_generation_metrics_summary.json"
)

assert os.path.isdir(GPT2_ADAPTER_DIR), (
    f"Trained adapter not found: {GPT2_ADAPTER_DIR}"
)
assert os.path.isfile(GPT2_BASELINE_CSV), (
    f"Baseline generations not found: {GPT2_BASELINE_CSV}"
)

# Reuse exactly the baseline prompts and references.

baseline_df = pd.read_csv(GPT2_BASELINE_CSV).fillna("")

required_columns = [
    "prompt", "reference_response", "generated_response"
]

assert all(column in baseline_df.columns for column in required_columns)
assert len(baseline_df) == 1000, (
    f"Expected 1000 validation examples, found {len(baseline_df)}"
)

prompts = baseline_df["prompt"].astype(str).tolist()
references = baseline_df["reference_response"].astype(str).tolist()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Evaluation device:", device)


# Generate Trial 1 responses, or reuse existing generations.

generation_time_minutes = 0.0

if os.path.isfile(GPT2_TRIAL1_GENERATIONS_CSV):
    print("Existing Trial 1 generations found. Reusing them.")

    trial1_df = pd.read_csv(
        GPT2_TRIAL1_GENERATIONS_CSV
    ).fillna("")

    assert len(trial1_df) == 1000
    assert all(
        column in trial1_df.columns
        for column in required_columns
    )

    assert trial1_df["prompt"].astype(str).tolist() == prompts
    assert (
        trial1_df["reference_response"].astype(str).tolist()
        == references
    )

    predictions = (
        trial1_df["generated_response"].astype(str).tolist()
    )

else:
    print("Loading trained GPT-2 LoRA Trial 1 adapter...")

    tokenizer = AutoTokenizer.from_pretrained(
        GPT2_ADAPTER_DIR
    )
    tokenizer.padding_side = "left"

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        "openai-community/gpt2"
    )

    model = PeftModel.from_pretrained(
        base_model,
        GPT2_ADAPTER_DIR
    )

    model.config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.to(device)
    model.eval()

    BATCH_SIZE = 8
    MAX_NEW_TOKENS = 40

    generation_start = time.time()
    predictions = []

    for start in range(0, len(prompts), BATCH_SIZE):
        batch_prompts = prompts[start:start + BATCH_SIZE]

        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=960
        ).to(device)

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                do_sample=False,
                max_new_tokens=MAX_NEW_TOKENS,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        # Exclude the padded input prompt from decoded responses.
        input_width = inputs["input_ids"].shape[1]
        generated_ids = output_ids[:, input_width:]

        batch_predictions = tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        predictions.extend(
            response.strip() for response in batch_predictions
        )

        if (start + BATCH_SIZE) % 200 == 0:
            print(
                f"Generated {min(start + BATCH_SIZE, len(prompts))}"
                f"/{len(prompts)} responses"
            )

    generation_time_minutes = (
        time.time() - generation_start
    ) / 60

    trial1_df = pd.DataFrame({
        "prompt": prompts,
        "reference_response": references,
        "generated_response": predictions
    })

    assert len(trial1_df) == 1000

    # Save separately; never modify baseline files.
    trial1_df.to_csv(
        GPT2_TRIAL1_GENERATIONS_CSV,
        index=False
    )

    print("Generated responses saved:", GPT2_TRIAL1_GENERATIONS_CSV)

    del model, base_model, tokenizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# BLEU — corpus-level word overlap.

references_tokens = [
    text.lower().split() for text in references
]
predictions_tokens = [
    text.lower().split() for text in predictions
]

bleu = corpus_bleu(
    [[reference] for reference in references_tokens],
    predictions_tokens,
    smoothing_function=SmoothingFunction().method1
)


# ROUGE-L — per-example F1.

rouge = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True
)

rouge_l_scores = [
    rouge.score(reference, prediction)["rougeL"].fmeasure
    for reference, prediction in zip(references, predictions)
]

mean_rouge_l = sum(rouge_l_scores) / len(rouge_l_scores)


# Distinct-1 and Distinct-2.

all_unigrams = []
all_bigrams = []

for prediction in predictions_tokens:
    all_unigrams.extend(prediction)
    all_bigrams.extend(
        tuple(prediction[i:i + 2])
        for i in range(max(0, len(prediction) - 1))
    )

distinct_1 = (
    len(set(all_unigrams)) / len(all_unigrams)
    if all_unigrams else 0.0
)

distinct_2 = (
    len(set(all_bigrams)) / len(all_bigrams)
    if all_bigrams else 0.0
)


# Response lengths.

reply_lengths = [
    len(text.split()) for text in predictions
]

average_reply_length = sum(reply_lengths) / len(reply_lengths)
median_reply_length = float(
    pd.Series(reply_lengths).median()
)


# BERTScore — use the same encoder and settings as the baseline.

bert_start = time.time()

P, R, F1 = bert_score(
    cands=predictions,
    refs=references,
    model_type="distilbert-base-uncased",
    lang="en",
    batch_size=16,
    device=str(device),
    verbose=True,
    rescale_with_baseline=False
)

mean_bert_precision = P.mean().item()
mean_bert_recall = R.mean().item()
mean_bert_f1 = F1.mean().item()

bert_time_minutes = (time.time() - bert_start) / 60


# Save per-example metrics.

trial1_df["rougeL_f1"] = rouge_l_scores
trial1_df["bert_score_precision"] = P.tolist()
trial1_df["bert_score_recall"] = R.tolist()
trial1_df["bert_score_f1"] = F1.tolist()
trial1_df["generated_word_count"] = reply_lengths

trial1_df.to_csv(
    GPT2_TRIAL1_METRICS_CSV,
    index=False
)


# Save aggregate metrics.

metrics = {
    "model": "GPT-2",
    "stage": "LoRA Trial 1",
    "number_of_validation_conversations": len(trial1_df),
    "training_examples": 17878,
    "validation_loss": 3.0375,
    "validation_perplexity": 20.8532,
    "corpus_bleu": bleu,
    "mean_rougeL_f1": mean_rouge_l,
    "mean_bert_score_precision": mean_bert_precision,
    "mean_bert_score_recall": mean_bert_recall,
    "mean_bert_score_f1": mean_bert_f1,
    "distinct_1": distinct_1,
    "distinct_2": distinct_2,
    "average_reply_length_words": average_reply_length,
    "median_reply_length_words": median_reply_length,
    "empty_reply_count": sum(
        not text.strip() for text in predictions
    ),
    "generation_time_minutes": generation_time_minutes,
    "bert_score_time_minutes": bert_time_minutes,
    "generation_method": "greedy decoding",
    "max_new_tokens": 40
}

with open(
    GPT2_TRIAL1_SUMMARY_JSON,
    "w",
    encoding="utf-8"
) as f:
    json.dump(metrics, f, indent=2)


# Display results.

print("\n===== GPT-2 LoRA TRIAL 1 GENERATION METRICS =====")

for name, value in metrics.items():
    if isinstance(value, float):
        print(f"{name}: {value:.4f}")
    else:
        print(f"{name}: {value}")

print("\nGeneration CSV:", GPT2_TRIAL1_GENERATIONS_CSV)
print("Per-example metrics CSV:", GPT2_TRIAL1_METRICS_CSV)
print("Summary JSON:", GPT2_TRIAL1_SUMMARY_JSON)
print("================================================")

Evaluation device: cuda
Loading trained GPT-2 LoRA Trial 1 adapter...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Generated 200/1000 responses
Generated 400/1000 responses
Generated 600/1000 responses
Generated 800/1000 responses
Generated 1000/1000 responses
Generated responses saved: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/trial_1_conversation_level/gpt2_lora_trial1_validation_generations.csv


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/119 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/63 [00:00<?, ?it/s]

done in 1.64 seconds, 610.40 sentences/sec

===== GPT-2 LoRA TRIAL 1 GENERATION METRICS =====
model: GPT-2
stage: LoRA Trial 1
number_of_validation_conversations: 1000
training_examples: 17878
validation_loss: 3.0375
validation_perplexity: 20.8532
corpus_bleu: 0.0139
mean_rougeL_f1: 0.1462
mean_bert_score_precision: 0.7230
mean_bert_score_recall: 0.7067
mean_bert_score_f1: 0.7144
distinct_1: 0.0870
distinct_2: 0.2761
average_reply_length_words: 9.8670
median_reply_length_words: 10.0000
empty_reply_count: 0
generation_time_minutes: 1.3309
bert_score_time_minutes: 0.0593
generation_method: greedy decoding
max_new_tokens: 40

Generation CSV: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/trial_1_conversation_level/gpt2_lora_trial1_validation_generations.csv
Per-example metrics CSV: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/trial_1_conversation_level/gpt2_lora_trial1_generation_metrics_per_example.csv
Summary JSON: /content/drive/MyDrive/COMP6012_Assignment

Verifying GPT-2 Baseline vs LoRA Trial 1 and Comparing Generation Lengths

In [ ]:

import os
import json
import pandas as pd

GPT2_BASELINE_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "PersonaChat_GPT2/base_model_evaluation"
)

GPT2_TRIAL1_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "PersonaChat_GPT2/trial_1_conversation_level"
)

baseline_csv = os.path.join(
    GPT2_BASELINE_DIR,
    "gpt2_base_validation_generations.csv"
)

trial1_csv = os.path.join(
    GPT2_TRIAL1_DIR,
    "gpt2_lora_trial1_validation_generations.csv"
)

baseline_metrics_json = os.path.join(
    GPT2_BASELINE_DIR,
    "gpt2_base_generation_metrics_summary.json"
)

trial1_metrics_json = os.path.join(
    GPT2_TRIAL1_DIR,
    "gpt2_lora_trial1_generation_metrics_summary.json"
)

assert os.path.isfile(baseline_csv), "Base generation CSV is missing."
assert os.path.isfile(trial1_csv), "Trial 1 generation CSV is missing."

base_df = pd.read_csv(baseline_csv).fillna("")
trial1_df = pd.read_csv(trial1_csv).fillna("")

required_columns = [
    "prompt", "reference_response", "generated_response"
]

assert all(c in base_df.columns for c in required_columns)
assert all(c in trial1_df.columns for c in required_columns)
assert len(base_df) == 1000
assert len(trial1_df) == 1000

# Verify the exact prompts and reference responses, including row order.
prompts_match = (
    base_df["prompt"].astype(str).tolist()
    == trial1_df["prompt"].astype(str).tolist()
)

references_match = (
    base_df["reference_response"].astype(str).tolist()
    == trial1_df["reference_response"].astype(str).tolist()
)

print("===== BASE vs GPT-2 LoRA TRIAL 1 VERIFICATION =====")
print("Base validation rows:", len(base_df))
print("Trial 1 validation rows:", len(trial1_df))
print("Prompts match exactly and in order:", prompts_match)
print("Reference responses match exactly and in order:", references_match)
print(
    "Base and Trial 1 generated responses identical:",
    base_df["generated_response"].astype(str).tolist()
    == trial1_df["generated_response"].astype(str).tolist()
)

assert prompts_match, "Prompt mismatch detected."
assert references_match, "Reference-response mismatch detected."

# Report settings recorded in the Trial 1 metrics summary.
if os.path.isfile(trial1_metrics_json):
    with open(trial1_metrics_json, "r", encoding="utf-8") as f:
        trial1_metrics = json.load(f)

    print("\nTrial 1 recorded generation settings:")
    for key in [
        "generation_method",
        "max_new_tokens",
        "generation_time_minutes"
    ]:
        print(f"{key}: {trial1_metrics.get(key, 'Not recorded')}")

if os.path.isfile(baseline_metrics_json):
    with open(baseline_metrics_json, "r", encoding="utf-8") as f:
        base_metrics = json.load(f)

    print("\nBaseline recorded generation settings:")
    for key in [
        "generation_method",
        "max_new_tokens",
        "generation_time_minutes"
    ]:
        print(f"{key}: {base_metrics.get(key, 'Not recorded')}")

print("\n===== GENERATION LENGTH DIAGNOSTICS =====")
for name, df in [
    ("Base GPT-2", base_df),
    ("GPT-2 LoRA Trial 1", trial1_df)
]:
    lengths = df["generated_response"].astype(str).str.split().str.len()
    print(
        f"{name}: mean={lengths.mean():.2f} words, "
        f"median={lengths.median():.0f}, "
        f"empty={int(lengths.eq(0).sum())}"
    )

print("\nPrompt and reference verification passed.")
print("No files were modified.")

===== BASE vs GPT-2 LoRA TRIAL 1 VERIFICATION =====
Base validation rows: 1000
Trial 1 validation rows: 1000
Prompts match exactly and in order: True
Reference responses match exactly and in order: True
Base and Trial 1 generated responses identical: False

Trial 1 recorded generation settings:
generation_method: greedy decoding
max_new_tokens: 40
generation_time_minutes: 1.3308504859606425

Baseline recorded generation settings:
generation_method: Not recorded
max_new_tokens: Not recorded
generation_time_minutes: Not recorded

===== GENERATION LENGTH DIAGNOSTICS =====
Base GPT-2: mean=33.78 words, median=34, empty=0
GPT-2 LoRA Trial 1: mean=9.87 words, median=10, empty=0

Prompt and reference verification passed.
No files were modified.


Configuring GPT-2 + LoRA Trail 2

Learning rate: 5e-5

Epoch: 1

Batch size: 16


In [ ]:

import os
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from torch.optim import AdamW

# Trial 2 configuration — conversation-level examples
GPT2_TRIAL_NAME = "gpt2_personachat_conversation_trial2"
GPT2_LEARNING_RATE = 5e-5
GPT2_EPOCHS = 1
GPT2_BATCH_SIZE = 16
GPT2_MAX_GRAD_NORM = 1.0
GPT2_CHECKPOINT_EVERY = 250
GPT2_MAX_LENGTH = 320
GPT2_SEED = 42

GPT2_CONVERSATION_TRIAL2_DIR = os.path.abspath(
    "/content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/trial_2_conversation_level"
)
os.makedirs(GPT2_CONVERSATION_TRIAL2_DIR, exist_ok=True)

# Reproducibility
torch.manual_seed(GPT2_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GPT2_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the base GPT-2 model
gpt2_conversation_model = AutoModelForCausalLM.from_pretrained(
    GPT2_MODEL_NAME
)
gpt2_conversation_model.config.pad_token_id = gpt2_tokenizer.pad_token_id
gpt2_conversation_model.config.use_cache = False

# LoRA configuration
gpt2_conversation_lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["c_attn", "c_proj"],
    bias="none",
)

gpt2_conversation_model = get_peft_model(
    gpt2_conversation_model,
    gpt2_conversation_lora_config
)
gpt2_conversation_model.to(DEVICE)

# Optimizer
gpt2_conversation_optimizer = AdamW(
    [p for p in gpt2_conversation_model.parameters() if p.requires_grad],
    lr=GPT2_LEARNING_RATE,
    weight_decay=0.01,
)

# Training size and step counts
GPT2_STEPS_PER_EPOCH = (
    len(gpt2_train_conversation_dataset) + GPT2_BATCH_SIZE - 1
) // GPT2_BATCH_SIZE
GPT2_TOTAL_STEPS = GPT2_STEPS_PER_EPOCH * GPT2_EPOCHS

print("Model configured successfully.")
print("Device:", DEVICE)
print("Trainable parameters:")
gpt2_conversation_model.print_trainable_parameters()
print("Training examples:", len(gpt2_train_conversation_dataset))
print("Validation examples:", len(gpt2_validation_conversation_dataset))
print("Batch size:", GPT2_BATCH_SIZE)
print("Steps per epoch:", GPT2_STEPS_PER_EPOCH)
print("Total planned optimizer steps:", GPT2_TOTAL_STEPS)
print("Learning rate:", GPT2_LEARNING_RATE)
print("Output directory:", GPT2_CONVERSATION_TRIAL2_DIR)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/peft/tuners/lora/layer.py:2760: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


Model configured successfully.
Device: cuda
Trainable parameters:
trainable params: 811,008 || all params: 125,250,816 || trainable%: 0.6475
Training examples: 17878
Validation examples: 1000
Batch size: 16
Steps per epoch: 1118
Total planned optimizer steps: 1118
Learning rate: 5e-05
Output directory: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/trial_2_conversation_level


GPT-2 + LoRA Trial 2 training

In [ ]:

import os
import time
import math
import torch
from torch.utils.data import DataLoader
from torch.nn.utils import clip_grad_norm_


# Deterministic DataLoader: recreates the same order on resume

def make_epoch_loader(epoch_index):
    generator = torch.Generator()
    generator.manual_seed(GPT2_SEED + epoch_index)

    return DataLoader(
        gpt2_train_conversation_dataset,
        batch_size=GPT2_BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        collate_fn=gpt2_collator,
    )



# Checkpoint paths and mixed precision

GPT2_CHECKPOINT_PATH = os.path.join(
    GPT2_CONVERSATION_TRIAL2_DIR,
    "training_checkpoint.pt",
)
GPT2_FINAL_ADAPTER_DIR = os.path.join(
    GPT2_CONVERSATION_TRIAL2_DIR,
    "final_adapter",
)

use_amp = DEVICE.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

global_step = 0
start_epoch = 0
start_batch = 0
loss_history = []
elapsed_before = 0.0


# Resume from checkpoint, if present

if os.path.isfile(GPT2_CHECKPOINT_PATH):
    checkpoint = torch.load(
        GPT2_CHECKPOINT_PATH,
        map_location=DEVICE,
        weights_only=False,
    )

    gpt2_conversation_model.load_state_dict(checkpoint["model_state"])
    gpt2_conversation_optimizer.load_state_dict(checkpoint["optimizer_state"])

    if "scaler_state" in checkpoint:
        scaler.load_state_dict(checkpoint["scaler_state"])

    global_step = checkpoint["global_step"]
    start_epoch = checkpoint["next_epoch"]
    start_batch = checkpoint["next_batch"]
    loss_history = checkpoint.get("loss_history", [])
    elapsed_before = checkpoint.get("elapsed_seconds", 0.0)

    torch.set_rng_state(checkpoint["torch_rng_state"].cpu())
    if torch.cuda.is_available() and checkpoint.get("cuda_rng_state") is not None:
        torch.cuda.set_rng_state_all(checkpoint["cuda_rng_state"])

    print(
        f"Resuming at epoch {start_epoch + 1}, "
        f"batch {start_batch + 1}, global step {global_step}."
    )
else:
    print("No existing training checkpoint. Starting a fresh run.")


# Atomic checkpoint saving

def save_checkpoint(next_epoch, next_batch, elapsed_seconds):
    checkpoint_data = {
        "model_state": gpt2_conversation_model.state_dict(),
        "optimizer_state": gpt2_conversation_optimizer.state_dict(),
        "scaler_state": scaler.state_dict(),
        "global_step": global_step,
        "next_epoch": next_epoch,
        "next_batch": next_batch,
        "loss_history": loss_history,
        "elapsed_seconds": elapsed_seconds,
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else None
        ),
        "trial_name": GPT2_TRIAL_NAME,
        "learning_rate": GPT2_LEARNING_RATE,
        "batch_size": GPT2_BATCH_SIZE,
        "seed": GPT2_SEED,
    }

    temporary_path = GPT2_CHECKPOINT_PATH + ".tmp"
    torch.save(checkpoint_data, temporary_path)
    os.replace(temporary_path, GPT2_CHECKPOINT_PATH)



# Training

gpt2_conversation_model.train()
run_start = time.time()

for epoch_index in range(start_epoch, GPT2_EPOCHS):
    train_loader = make_epoch_loader(epoch_index)
    batch_offset = start_batch if epoch_index == start_epoch else 0

    for batch_index, batch in enumerate(train_loader):
        if batch_index < batch_offset:
            continue

        batch = {
            key: value.to(DEVICE, non_blocking=True)
            for key, value in batch.items()
        }

        gpt2_conversation_optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=use_amp,
        ):
            outputs = gpt2_conversation_model(**batch)
            loss = outputs.loss

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss at epoch {epoch_index + 1}, "
                f"batch {batch_index + 1}: {loss.item()}"
            )

        scaler.scale(loss).backward()
        scaler.unscale_(gpt2_conversation_optimizer)
        clip_grad_norm_(
            gpt2_conversation_model.parameters(),
            GPT2_MAX_GRAD_NORM,
        )
        scaler.step(gpt2_conversation_optimizer)
        scaler.update()

        global_step += 1
        loss_history.append(float(loss.detach().cpu()))

        if global_step % 25 == 0:
            recent_loss = sum(loss_history[-25:]) / len(loss_history[-25:])
            elapsed = elapsed_before + time.time() - run_start
            print(
                f"Epoch {epoch_index + 1}/{GPT2_EPOCHS} | "
                f"Step {global_step}/{GPT2_TOTAL_STEPS} | "
                f"Batch loss (last 25 avg): {recent_loss:.4f} | "
                f"Elapsed: {elapsed / 60:.1f} min"
            )

        next_epoch = epoch_index
        next_batch = batch_index + 1

        if next_batch >= len(train_loader):
            next_epoch = epoch_index + 1
            next_batch = 0

        if (
            global_step % GPT2_CHECKPOINT_EVERY == 0
            or next_epoch >= GPT2_EPOCHS
        ):
            elapsed = elapsed_before + time.time() - run_start
            save_checkpoint(next_epoch, next_batch, elapsed)
            print(
                f"Checkpoint saved at step {global_step}; "
                f"next position: epoch {next_epoch + 1}, "
                f"batch {next_batch + 1}."
            )

    start_batch = 0

elapsed_total = elapsed_before + time.time() - run_start


# Save final LoRA adapter and tokenizer

os.makedirs(GPT2_FINAL_ADAPTER_DIR, exist_ok=True)
gpt2_conversation_model.save_pretrained(GPT2_FINAL_ADAPTER_DIR)
gpt2_tokenizer.save_pretrained(GPT2_FINAL_ADAPTER_DIR)


# Validation loss and perplexity

gpt2_conversation_model.eval()
validation_loss_sum = 0.0
validation_batches = 0

validation_loader = DataLoader(
    gpt2_validation_conversation_dataset,
    batch_size=GPT2_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=gpt2_collator,
)

with torch.inference_mode():
    for batch in validation_loader:
        batch = {
            key: value.to(DEVICE, non_blocking=True)
            for key, value in batch.items()
        }

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=use_amp,
        ):
            outputs = gpt2_conversation_model(**batch)

        if torch.isfinite(outputs.loss):
            validation_loss_sum += outputs.loss.item()
            validation_batches += 1
        else:
            raise RuntimeError("Non-finite validation loss detected.")

validation_loss = validation_loss_sum / validation_batches
validation_perplexity = math.exp(validation_loss)

print("\n========== GPT-2 CONVERSATION-LEVEL TRIAL 2 ==========")
print("Training examples:", len(gpt2_train_conversation_dataset))
print("Validation examples:", len(gpt2_validation_conversation_dataset))
print("Optimizer steps completed:", global_step)
print("Mean training loss:", round(sum(loss_history) / len(loss_history), 4))
print("Validation loss:", round(validation_loss, 4))
print("Validation perplexity:", round(validation_perplexity, 4))
print("Elapsed training time (minutes):", round(elapsed_total / 60, 2))
print("Final adapter:", GPT2_FINAL_ADAPTER_DIR)
print("Training checkpoint:", GPT2_CHECKPOINT_PATH)
print("======================================================")

gpt2_conversation_model.train()

No existing training checkpoint. Starting a fresh run.
Epoch 1/1 | Step 25/1118 | Batch loss (last 25 avg): 4.3497 | Elapsed: 0.2 min
Epoch 1/1 | Step 50/1118 | Batch loss (last 25 avg): 3.9905 | Elapsed: 0.4 min
Epoch 1/1 | Step 75/1118 | Batch loss (last 25 avg): 3.7563 | Elapsed: 0.6 min
Epoch 1/1 | Step 100/1118 | Batch loss (last 25 avg): 3.6125 | Elapsed: 0.8 min
Epoch 1/1 | Step 125/1118 | Batch loss (last 25 avg): 3.5515 | Elapsed: 1.0 min
Epoch 1/1 | Step 150/1118 | Batch loss (last 25 avg): 3.6244 | Elapsed: 1.2 min
Epoch 1/1 | Step 175/1118 | Batch loss (last 25 avg): 3.4757 | Elapsed: 1.5 min
Epoch 1/1 | Step 200/1118 | Batch loss (last 25 avg): 3.4943 | Elapsed: 1.7 min
Epoch 1/1 | Step 225/1118 | Batch loss (last 25 avg): 3.5509 | Elapsed: 1.9 min
Epoch 1/1 | Step 250/1118 | Batch loss (last 25 avg): 3.4625 | Elapsed: 2.1 min
Checkpoint saved at step 250; next position: epoch 1, batch 251.
Epoch 1/1 | Step 275/1118 | Batch loss (last 25 avg): 3.4344 | Elapsed: 2.3 min
Epo

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GPT2LMHeadModel(
      (transformer): GPT2Model(
        (wte): Embedding(50257, 768)
        (wpe): Embedding(1024, 768)
        (drop): Dropout(p=0.1, inplace=False)
        (h): ModuleList(
          (0-11): 12 x GPT2Block(
            (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (attn): GPT2Attention(
              (c_attn): lora.Linear(
                (base_layer): Conv1D(nf=2304, nx=768)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=768, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2304, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
          

GPT-2 + LoRA Trial 2 Generation and evaluation

In [ ]:

import os
import json
import time
import gc
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bert_score


# Paths

GPT2_TRIAL2_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "PersonaChat_GPT2/trial_2_conversation_level"
)

GPT2_ADAPTER_DIR = os.path.join(
    GPT2_TRIAL2_DIR, "final_adapter"
)

GPT2_BASELINE_CSV = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "PersonaChat_GPT2/base_model_evaluation/"
    "gpt2_base_validation_generations.csv"
)

GPT2_TRIAL2_GENERATIONS_CSV = os.path.join(
    GPT2_TRIAL2_DIR,
    "gpt2_lora_trial2_validation_generations.csv"
)

GPT2_TRIAL2_METRICS_CSV = os.path.join(
    GPT2_TRIAL2_DIR,
    "gpt2_lora_trial2_generation_metrics_per_example.csv"
)

GPT2_TRIAL2_SUMMARY_JSON = os.path.join(
    GPT2_TRIAL2_DIR,
    "gpt2_lora_trial2_generation_metrics_summary.json"
)

assert os.path.isdir(GPT2_ADAPTER_DIR), (
    f"Trained adapter not found: {GPT2_ADAPTER_DIR}"
)
assert os.path.isfile(GPT2_BASELINE_CSV), (
    f"Baseline generations not found: {GPT2_BASELINE_CSV}"
)

# Reuse exactly the baseline prompts and references.

baseline_df = pd.read_csv(GPT2_BASELINE_CSV).fillna("")

required_columns = [
    "prompt", "reference_response", "generated_response"
]

assert all(column in baseline_df.columns for column in required_columns)
assert len(baseline_df) == 1000, (
    f"Expected 1000 validation examples, found {len(baseline_df)}"
)

prompts = baseline_df["prompt"].astype(str).tolist()
references = baseline_df["reference_response"].astype(str).tolist()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Evaluation device:", device)


# Generate Trial 2 responses, or reuse existing generations.

generation_time_minutes = 0.0

if os.path.isfile(GPT2_TRIAL2_GENERATIONS_CSV):
    print("Existing Trial 2 generations found. Reusing them.")

    trial2_df = pd.read_csv(
        GPT2_TRIAL2_GENERATIONS_CSV
    ).fillna("")

    assert len(trial2_df) == 1000
    assert all(
        column in trial2_df.columns
        for column in required_columns
    )

    assert trial2_df["prompt"].astype(str).tolist() == prompts
    assert (
        trial2_df["reference_response"].astype(str).tolist()
        == references
    )

    predictions = (
        trial2_df["generated_response"].astype(str).tolist()
    )

else:
    print("Loading trained GPT-2 LoRA Trial 2 adapter...")

    tokenizer = AutoTokenizer.from_pretrained(
        GPT2_ADAPTER_DIR
    )
    tokenizer.padding_side = "left"

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        "openai-community/gpt2"
    )

    model = PeftModel.from_pretrained(
        base_model,
        GPT2_ADAPTER_DIR
    )

    model.config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.to(device)
    model.eval()

    BATCH_SIZE = 8
    MAX_NEW_TOKENS = 40

    generation_start = time.time()
    predictions = []

    for start in range(0, len(prompts), BATCH_SIZE):
        batch_prompts = prompts[start:start + BATCH_SIZE]

        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=960
        ).to(device)

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                do_sample=False,
                max_new_tokens=MAX_NEW_TOKENS,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        # Exclude the padded input prompt from decoded responses.
        input_width = inputs["input_ids"].shape[1]
        generated_ids = output_ids[:, input_width:]

        batch_predictions = tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        predictions.extend(
            response.strip() for response in batch_predictions
        )

        if (start + BATCH_SIZE) % 200 == 0:
            print(
                f"Generated {min(start + BATCH_SIZE, len(prompts))}"
                f"/{len(prompts)} responses"
            )

    generation_time_minutes = (
        time.time() - generation_start
    ) / 60

    trial2_df = pd.DataFrame({
        "prompt": prompts,
        "reference_response": references,
        "generated_response": predictions
    })

    assert len(trial2_df) == 1000

    # Save separately; never modify baseline or Trial 1 files.
    trial2_df.to_csv(
        GPT2_TRIAL2_GENERATIONS_CSV,
        index=False
    )

    print("Generated responses saved:", GPT2_TRIAL2_GENERATIONS_CSV)

    del model, base_model, tokenizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# BLEU — corpus-level word overlap.

references_tokens = [
    text.lower().split() for text in references
]
predictions_tokens = [
    text.lower().split() for text in predictions
]

bleu = corpus_bleu(
    [[reference] for reference in references_tokens],
    predictions_tokens,
    smoothing_function=SmoothingFunction().method1
)


# ROUGE-L — per-example F1.

rouge = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True
)

rouge_l_scores = [
    rouge.score(reference, prediction)["rougeL"].fmeasure
    for reference, prediction in zip(references, predictions)
]

mean_rouge_l = sum(rouge_l_scores) / len(rouge_l_scores)


# Distinct-1 and Distinct-2.

all_unigrams = []
all_bigrams = []

for prediction in predictions_tokens:
    all_unigrams.extend(prediction)
    all_bigrams.extend(
        tuple(prediction[i:i + 2])
        for i in range(max(0, len(prediction) - 1))
    )

distinct_1 = (
    len(set(all_unigrams)) / len(all_unigrams)
    if all_unigrams else 0.0
)

distinct_2 = (
    len(set(all_bigrams)) / len(all_bigrams)
    if all_bigrams else 0.0
)


# Response lengths.

reply_lengths = [
    len(text.split()) for text in predictions
]

average_reply_length = sum(reply_lengths) / len(reply_lengths)
median_reply_length = float(
    pd.Series(reply_lengths).median()
)


# BERTScore — use the same encoder and settings as the baseline.

bert_start = time.time()

P, R, F1 = bert_score(
    cands=predictions,
    refs=references,
    model_type="distilbert-base-uncased",
    lang="en",
    batch_size=16,
    device=str(device),
    verbose=True,
    rescale_with_baseline=False
)

mean_bert_precision = P.mean().item()
mean_bert_recall = R.mean().item()
mean_bert_f1 = F1.mean().item()

bert_time_minutes = (time.time() - bert_start) / 60


# Save per-example metrics.

trial2_df["rougeL_f1"] = rouge_l_scores
trial2_df["bert_score_precision"] = P.tolist()
trial2_df["bert_score_recall"] = R.tolist()
trial2_df["bert_score_f1"] = F1.tolist()
trial2_df["generated_word_count"] = reply_lengths

trial2_df.to_csv(
    GPT2_TRIAL2_METRICS_CSV,
    index=False
)


# Save aggregate metrics.

metrics = {
    "model": "GPT-2",
    "stage": "LoRA Trial 2",
    "number_of_validation_conversations": len(trial2_df),
    "training_examples": 17878,
    "validation_loss": 3.0986,
    "validation_perplexity": 22.1661,
    "corpus_bleu": bleu,
    "mean_rougeL_f1": mean_rouge_l,
    "mean_bert_score_precision": mean_bert_precision,
    "mean_bert_score_recall": mean_bert_recall,
    "mean_bert_score_f1": mean_bert_f1,
    "distinct_1": distinct_1,
    "distinct_2": distinct_2,
    "average_reply_length_words": average_reply_length,
    "median_reply_length_words": median_reply_length,
    "empty_reply_count": sum(
        not text.strip() for text in predictions
    ),
    "generation_time_minutes": generation_time_minutes,
    "bert_score_time_minutes": bert_time_minutes,
    "generation_method": "greedy decoding",
    "max_new_tokens": 40
}

with open(
    GPT2_TRIAL2_SUMMARY_JSON,
    "w",
    encoding="utf-8"
) as f:
    json.dump(metrics, f, indent=2)


# Display results.

print("\n===== GPT-2 LoRA TRIAL 2 GENERATION METRICS =====")

for name, value in metrics.items():
    if isinstance(value, float):
        print(f"{name}: {value:.4f}")
    else:
        print(f"{name}: {value}")

print("\nGeneration CSV:", GPT2_TRIAL2_GENERATIONS_CSV)
print("Per-example metrics CSV:", GPT2_TRIAL2_METRICS_CSV)
print("Summary JSON:", GPT2_TRIAL2_SUMMARY_JSON)
print("================================================")

Evaluation device: cuda
Loading trained GPT-2 LoRA Trial 2 adapter...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Generated 200/1000 responses
Generated 400/1000 responses
Generated 600/1000 responses
Generated 800/1000 responses
Generated 1000/1000 responses
Generated responses saved: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/trial_2_conversation_level/gpt2_lora_trial2_validation_generations.csv


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/120 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/63 [00:00<?, ?it/s]

done in 1.58 seconds, 634.42 sentences/sec

===== GPT-2 LoRA TRIAL 2 GENERATION METRICS =====
model: GPT-2
stage: LoRA Trial 2
number_of_validation_conversations: 1000
training_examples: 17878
validation_loss: 3.0986
validation_perplexity: 22.1661
corpus_bleu: 0.0138
mean_rougeL_f1: 0.1401
mean_bert_score_precision: 0.7153
mean_bert_score_recall: 0.7032
mean_bert_score_f1: 0.7089
distinct_1: 0.0799
distinct_2: 0.2455
average_reply_length_words: 11.0590
median_reply_length_words: 10.0000
empty_reply_count: 0
generation_time_minutes: 1.0567
bert_score_time_minutes: 0.0439
generation_method: greedy decoding
max_new_tokens: 40

Generation CSV: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/trial_2_conversation_level/gpt2_lora_trial2_validation_generations.csv
Per-example metrics CSV: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/trial_2_conversation_level/gpt2_lora_trial2_generation_metrics_per_example.csv
Summary JSON: /content/drive/MyDrive/COMP6012_Assignmen

Verify GPT-2 Baseline vs LoRA Trial 2

In [ ]:

# ============================================================
# VERIFY GPT-2 BASELINE vs LoRA TRIAL 2
# Read-only verification: does not train, regenerate, or save
# ============================================================

import os
import json
import pandas as pd

GPT2_BASE_DIR = "/content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/base_model_evaluation"
GPT2_TRIAL2_DIR = "/content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/trial_2_conversation_level"

BASE_GENERATIONS_FILE = os.path.join(
    GPT2_BASE_DIR, "gpt2_base_validation_generations.csv"
)
TRIAL2_GENERATIONS_FILE = os.path.join(
    GPT2_TRIAL2_DIR, "gpt2_lora_trial2_validation_generations.csv"
)
TRIAL2_PER_EXAMPLE_FILE = os.path.join(
    GPT2_TRIAL2_DIR, "gpt2_lora_trial2_generation_metrics_per_example.csv"
)
TRIAL2_SUMMARY_FILE = os.path.join(
    GPT2_TRIAL2_DIR, "gpt2_lora_trial2_generation_metrics_summary.json"
)

# 1. Check that expected files exist
required_files = [
    BASE_GENERATIONS_FILE,
    TRIAL2_GENERATIONS_FILE,
    TRIAL2_PER_EXAMPLE_FILE,
    TRIAL2_SUMMARY_FILE,
]

print("Checking required files...")
for file_path in required_files:
    print(f"{'FOUND' if os.path.isfile(file_path) else 'MISSING'}: {file_path}")

assert all(os.path.isfile(path) for path in required_files), (
    "One or more required files are missing. Check the paths before proceeding."
)

# 2. Load existing generation files
base_df = pd.read_csv(BASE_GENERATIONS_FILE)
trial2_df = pd.read_csv(TRIAL2_GENERATIONS_FILE)

required_columns = ["prompt", "reference_response", "generated_response"]

for name, df in [("Baseline", base_df), ("Trial 2", trial2_df)]:
    missing_columns = [col for col in required_columns if col not in df.columns]
    assert not missing_columns, f"{name} is missing columns: {missing_columns}"

print("\n--- Dataset checks ---")
print(f"Baseline examples: {len(base_df)}")
print(f"Trial 2 examples: {len(trial2_df)}")

assert len(base_df) == 1000, "Baseline should contain 1,000 examples."
assert len(trial2_df) == 1000, "Trial 2 should contain 1,000 examples."

prompts_match = base_df["prompt"].fillna("").equals(
    trial2_df["prompt"].fillna("")
)
references_match = base_df["reference_response"].fillna("").equals(
    trial2_df["reference_response"].fillna("")
)

print(f"Prompts match baseline in the same order: {prompts_match}")
print(f"Reference responses match baseline in the same order: {references_match}")

assert prompts_match, "Prompt mismatch detected."
assert references_match, "Reference-response mismatch detected."

# 3. Check whether generated responses differ
responses_match = base_df["generated_response"].fillna("").equals(
    trial2_df["generated_response"].fillna("")
)
print(f"Generated responses identical to baseline: {responses_match}")

# 4. Inspect Trial 2 metric files
trial2_per_example_df = pd.read_csv(TRIAL2_PER_EXAMPLE_FILE)

with open(TRIAL2_SUMMARY_FILE, "r", encoding="utf-8") as file:
    trial2_summary = json.load(file)

print("\n--- Trial 2 metric files ---")
print(f"Per-example metric rows: {len(trial2_per_example_df)}")
print("Trial 2 summary JSON:")
print(json.dumps(trial2_summary, indent=2, ensure_ascii=False))

# 5. Compare generated-response lengths
def word_count(text):
    if pd.isna(text):
        return 0
    return len(str(text).split())

base_lengths = base_df["generated_response"].apply(word_count)
trial2_lengths = trial2_df["generated_response"].apply(word_count)

print("\n--- Generated-response length comparison ---")
print(f"Baseline average words: {base_lengths.mean():.3f}")
print(f"Baseline median words:  {base_lengths.median():.3f}")
print(f"Baseline empty replies: {(base_lengths == 0).sum()}")

print(f"Trial 2 average words:  {trial2_lengths.mean():.3f}")
print(f"Trial 2 median words:   {trial2_lengths.median():.3f}")
print(f"Trial 2 empty replies:  {(trial2_lengths == 0).sum()}")

print("\nVERIFICATION PASSED: Trial 2 files exist and prompts/references match.")
print("No model training, generation, or file modification was performed.")


Checking required files...
FOUND: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/base_model_evaluation/gpt2_base_validation_generations.csv
FOUND: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/trial_2_conversation_level/gpt2_lora_trial2_validation_generations.csv
FOUND: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/trial_2_conversation_level/gpt2_lora_trial2_generation_metrics_per_example.csv
FOUND: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/trial_2_conversation_level/gpt2_lora_trial2_generation_metrics_summary.json

--- Dataset checks ---
Baseline examples: 1000
Trial 2 examples: 1000
Prompts match baseline in the same order: True
Reference responses match baseline in the same order: True
Generated responses identical to baseline: False

--- Trial 2 metric files ---
Per-example metric rows: 1000
Trial 2 summary JSON:
{
  "model": "GPT-2",
  "stage": "LoRA Trial 2",
  "number_of_validation_conversations": 1000,
  "training_e

Configuring GPT-2 + LoRA Trail 3

Learning rate: 2e-4

Epoch: 2

Batch size: 16

Gradient Accumulation: 2

Effective Batch Size: 16X2 = 32


In [ ]:

# Configuring GPT-2 + LoRA Trial 3
# Effective batch size: 32 (16 x 2 gradient accumulation)

import os
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from torch.optim import AdamW

# Trial 3 configuration — conversation-level examples
GPT2_TRIAL_NAME = "gpt2_personachat_conversation_trial3"
GPT2_LEARNING_RATE = 2e-4
GPT2_EPOCHS = 2
GPT2_BATCH_SIZE = 16
GPT2_GRAD_ACCUMULATION_STEPS = 2
GPT2_EFFECTIVE_BATCH_SIZE = (
    GPT2_BATCH_SIZE * GPT2_GRAD_ACCUMULATION_STEPS
)
GPT2_MAX_GRAD_NORM = 1.0
GPT2_CHECKPOINT_EVERY = 250
GPT2_MAX_LENGTH = 320
GPT2_SEED = 42

GPT2_CONVERSATION_TRIAL3_DIR = os.path.abspath(
    "/content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/trial_3_conversation_level"
)
os.makedirs(GPT2_CONVERSATION_TRIAL3_DIR, exist_ok=True)

# Reproducibility
torch.manual_seed(GPT2_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GPT2_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the base GPT-2 model
gpt2_conversation_model = AutoModelForCausalLM.from_pretrained(
    GPT2_MODEL_NAME
)
gpt2_conversation_model.config.pad_token_id = gpt2_tokenizer.pad_token_id
gpt2_conversation_model.config.use_cache = False

# LoRA configuration
gpt2_conversation_lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["c_attn", "c_proj"],
    bias="none",
)

gpt2_conversation_model = get_peft_model(
    gpt2_conversation_model,
    gpt2_conversation_lora_config
)
gpt2_conversation_model.to(DEVICE)

# Optimizer
gpt2_conversation_optimizer = AdamW(
    [p for p in gpt2_conversation_model.parameters() if p.requires_grad],
    lr=GPT2_LEARNING_RATE,
    weight_decay=0.01,
)

# Training size and step counts
GPT2_MICRO_STEPS_PER_EPOCH = (
    len(gpt2_train_conversation_dataset) + GPT2_BATCH_SIZE - 1
) // GPT2_BATCH_SIZE

GPT2_STEPS_PER_EPOCH = (
    GPT2_MICRO_STEPS_PER_EPOCH
    + GPT2_GRAD_ACCUMULATION_STEPS - 1
) // GPT2_GRAD_ACCUMULATION_STEPS

GPT2_TOTAL_STEPS = GPT2_STEPS_PER_EPOCH * GPT2_EPOCHS

print("Model configured successfully.")
print("Device:", DEVICE)
print("Trainable parameters:")
gpt2_conversation_model.print_trainable_parameters()
print("Training examples:", len(gpt2_train_conversation_dataset))
print("Validation examples:", len(gpt2_validation_conversation_dataset))
print("Micro-batch size:", GPT2_BATCH_SIZE)
print("Gradient accumulation steps:", GPT2_GRAD_ACCUMULATION_STEPS)
print("Effective batch size:", GPT2_EFFECTIVE_BATCH_SIZE)
print("Optimizer steps per epoch:", GPT2_STEPS_PER_EPOCH)
print("Total planned optimizer steps:", GPT2_TOTAL_STEPS)
print("Learning rate:", GPT2_LEARNING_RATE)
print("Output directory:", GPT2_CONVERSATION_TRIAL3_DIR)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model configured successfully.
Device: cuda
Trainable parameters:
trainable params: 811,008 || all params: 125,250,816 || trainable%: 0.6475
Training examples: 17878
Validation examples: 1000
Micro-batch size: 16
Gradient accumulation steps: 2
Effective batch size: 32
Optimizer steps per epoch: 559
Total planned optimizer steps: 1118
Learning rate: 0.0002
Output directory: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/trial_3_conversation_level


Training GPT-2 + LoRA Trial 3

In [ ]:

import os
import time
import math
import torch
from torch.utils.data import DataLoader
from torch.nn.utils import clip_grad_norm_

# Deterministic DataLoader: recreates the same order on resume

def make_epoch_loader(epoch_index):
    generator = torch.Generator()
    generator.manual_seed(GPT2_SEED + epoch_index)

    return DataLoader(
        gpt2_train_conversation_dataset,
        batch_size=GPT2_BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        collate_fn=gpt2_collator,
    )


# Checkpoint paths and mixed precision

GPT2_CHECKPOINT_PATH = os.path.join(
    GPT2_CONVERSATION_TRIAL3_DIR,
    "training_checkpoint.pt",
)
GPT2_FINAL_ADAPTER_DIR = os.path.join(
    GPT2_CONVERSATION_TRIAL3_DIR,
    "final_adapter",
)

use_amp = DEVICE.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

global_step = 0
start_epoch = 0
start_batch = 0
loss_history = []
elapsed_before = 0.0


# Resume from checkpoint, if present

if os.path.isfile(GPT2_CHECKPOINT_PATH):
    checkpoint = torch.load(
        GPT2_CHECKPOINT_PATH,
        map_location=DEVICE,
        weights_only=False,
    )

    assert checkpoint.get("trial_name") == GPT2_TRIAL_NAME, (
        "Checkpoint belongs to a different trial."
    )
    assert checkpoint.get("batch_size") == GPT2_BATCH_SIZE, (
        "Checkpoint micro-batch size differs from current configuration."
    )
    assert checkpoint.get("learning_rate") == GPT2_LEARNING_RATE, (
        "Checkpoint learning rate differs from current configuration."
    )

    gpt2_conversation_model.load_state_dict(checkpoint["model_state"])
    gpt2_conversation_optimizer.load_state_dict(checkpoint["optimizer_state"])

    if "scaler_state" in checkpoint:
        scaler.load_state_dict(checkpoint["scaler_state"])

    global_step = checkpoint["global_step"]
    start_epoch = checkpoint["next_epoch"]
    start_batch = checkpoint["next_batch"]
    loss_history = checkpoint.get("loss_history", [])
    elapsed_before = checkpoint.get("elapsed_seconds", 0.0)

    torch.set_rng_state(checkpoint["torch_rng_state"].cpu())
    if torch.cuda.is_available() and checkpoint.get("cuda_rng_state") is not None:
        torch.cuda.set_rng_state_all(checkpoint["cuda_rng_state"])

    print(
        f"Resuming at epoch {start_epoch + 1}, "
        f"batch {start_batch + 1}, global step {global_step}."
    )
else:
    print("No existing training checkpoint. Starting a fresh run.")


# Atomic checkpoint saving

def save_checkpoint(next_epoch, next_batch, elapsed_seconds):
    checkpoint_data = {
        "model_state": gpt2_conversation_model.state_dict(),
        "optimizer_state": gpt2_conversation_optimizer.state_dict(),
        "scaler_state": scaler.state_dict(),
        "global_step": global_step,
        "next_epoch": next_epoch,
        "next_batch": next_batch,
        "loss_history": loss_history,
        "elapsed_seconds": elapsed_seconds,
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else None
        ),
        "trial_name": GPT2_TRIAL_NAME,
        "learning_rate": GPT2_LEARNING_RATE,
        "batch_size": GPT2_BATCH_SIZE,
        "gradient_accumulation_steps": GPT2_GRAD_ACCUMULATION_STEPS,
        "seed": GPT2_SEED,
    }

    temporary_path = GPT2_CHECKPOINT_PATH + ".tmp"
    torch.save(checkpoint_data, temporary_path)
    os.replace(temporary_path, GPT2_CHECKPOINT_PATH)


# Training

gpt2_conversation_model.train()
run_start = time.time()

for epoch_index in range(start_epoch, GPT2_EPOCHS):
    train_loader = make_epoch_loader(epoch_index)
    batch_offset = start_batch if epoch_index == start_epoch else 0

    gpt2_conversation_optimizer.zero_grad(set_to_none=True)
    accumulated_micro_batches = 0

    for batch_index, batch in enumerate(train_loader):
        if batch_index < batch_offset:
            continue

        batch = {
            key: value.to(DEVICE, non_blocking=True)
            for key, value in batch.items()
        }

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=use_amp,
        ):
            outputs = gpt2_conversation_model(**batch)
            loss = outputs.loss
            scaled_loss = loss / GPT2_GRAD_ACCUMULATION_STEPS

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss at epoch {epoch_index + 1}, "
                f"batch {batch_index + 1}: {loss.item()}"
            )

        scaler.scale(scaled_loss).backward()
        accumulated_micro_batches += 1
        loss_history.append(float(loss.detach().cpu()))

        # Update on each full accumulation group or final batch
        is_last_batch = batch_index + 1 == len(train_loader)
        should_update = (
            accumulated_micro_batches == GPT2_GRAD_ACCUMULATION_STEPS
            or is_last_batch
        )

        if should_update:
            scaler.unscale_(gpt2_conversation_optimizer)

            # Correct for a possibly incomplete final accumulation group
            if accumulated_micro_batches < GPT2_GRAD_ACCUMULATION_STEPS:
                correction = (
                    GPT2_GRAD_ACCUMULATION_STEPS
                    / accumulated_micro_batches
                )
                for parameter in gpt2_conversation_model.parameters():
                    if parameter.grad is not None:
                        parameter.grad.mul_(correction)

            clip_grad_norm_(
                gpt2_conversation_model.parameters(),
                GPT2_MAX_GRAD_NORM,
            )
            scaler.step(gpt2_conversation_optimizer)
            scaler.update()
            gpt2_conversation_optimizer.zero_grad(set_to_none=True)

            global_step += 1
            accumulated_micro_batches = 0

            if global_step % 25 == 0:
                recent_loss = sum(loss_history[-25:]) / len(loss_history[-25:])
                elapsed = elapsed_before + time.time() - run_start
                print(
                    f"Epoch {epoch_index + 1}/{GPT2_EPOCHS} | "
                    f"Step {global_step}/{GPT2_TOTAL_STEPS} | "
                    f"Micro-batch loss (last 25 avg): {recent_loss:.4f} | "
                    f"Elapsed: {elapsed / 60:.1f} min"
                )

            next_epoch = epoch_index
            next_batch = batch_index + 1

            if next_batch >= len(train_loader):
                next_epoch = epoch_index + 1
                next_batch = 0

            if (
                global_step % GPT2_CHECKPOINT_EVERY == 0
                or next_epoch >= GPT2_EPOCHS
            ):
                elapsed = elapsed_before + time.time() - run_start
                save_checkpoint(next_epoch, next_batch, elapsed)
                print(
                    f"Checkpoint saved at step {global_step}; "
                    f"next position: epoch {next_epoch + 1}, "
                    f"batch {next_batch + 1}."
                )

elapsed_total = elapsed_before + time.time() - run_start


# Save final LoRA adapter and tokenizer

os.makedirs(GPT2_FINAL_ADAPTER_DIR, exist_ok=True)
gpt2_conversation_model.save_pretrained(GPT2_FINAL_ADAPTER_DIR)
gpt2_tokenizer.save_pretrained(GPT2_FINAL_ADAPTER_DIR)


# Validation loss and perplexity

gpt2_conversation_model.eval()
validation_loss_sum = 0.0
validation_batches = 0

validation_loader = DataLoader(
    gpt2_validation_conversation_dataset,
    batch_size=GPT2_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=gpt2_collator,
)

with torch.inference_mode():
    for batch in validation_loader:
        batch = {
            key: value.to(DEVICE, non_blocking=True)
            for key, value in batch.items()
        }

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=use_amp,
        ):
            outputs = gpt2_conversation_model(**batch)

        if torch.isfinite(outputs.loss):
            validation_loss_sum += outputs.loss.item()
            validation_batches += 1
        else:
            raise RuntimeError("Non-finite validation loss detected.")

validation_loss = validation_loss_sum / validation_batches
validation_perplexity = math.exp(validation_loss)

print("\n========== GPT-2 CONVERSATION-LEVEL TRIAL 3 ==========")
print("Training examples:", len(gpt2_train_conversation_dataset))
print("Validation examples:", len(gpt2_validation_conversation_dataset))
print("Optimizer steps completed:", global_step)
print("Mean training loss:", round(sum(loss_history) / len(loss_history), 4))
print("Validation loss:", round(validation_loss, 4))
print("Validation perplexity:", round(validation_perplexity, 4))
print("Elapsed training time (minutes):", round(elapsed_total / 60, 2))
print("Effective batch size:", GPT2_EFFECTIVE_BATCH_SIZE)
print("Final adapter:", GPT2_FINAL_ADAPTER_DIR)
print("Training checkpoint:", GPT2_CHECKPOINT_PATH)
print("======================================================")

gpt2_conversation_model.train()

No existing training checkpoint. Starting a fresh run.


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch 1/2 | Step 25/1118 | Micro-batch loss (last 25 avg): 3.7365 | Elapsed: 0.5 min
Epoch 1/2 | Step 50/1118 | Micro-batch loss (last 25 avg): 3.4909 | Elapsed: 0.9 min
Epoch 1/2 | Step 75/1118 | Micro-batch loss (last 25 avg): 3.5157 | Elapsed: 1.3 min
Epoch 1/2 | Step 100/1118 | Micro-batch loss (last 25 avg): 3.4071 | Elapsed: 1.7 min
Epoch 1/2 | Step 125/1118 | Micro-batch loss (last 25 avg): 3.3922 | Elapsed: 2.1 min
Epoch 1/2 | Step 150/1118 | Micro-batch loss (last 25 avg): 3.3954 | Elapsed: 2.5 min
Epoch 1/2 | Step 175/1118 | Micro-batch loss (last 25 avg): 3.3650 | Elapsed: 2.9 min
Epoch 1/2 | Step 200/1118 | Micro-batch loss (last 25 avg): 3.3211 | Elapsed: 3.4 min
Epoch 1/2 | Step 225/1118 | Micro-batch loss (last 25 avg): 3.3214 | Elapsed: 3.8 min
Epoch 1/2 | Step 250/1118 | Micro-batch loss (last 25 avg): 3.3666 | Elapsed: 4.2 min
Checkpoint saved at step 250; next position: epoch 1, batch 501.
Epoch 1/2 | Step 275/1118 | Micro-batch loss (last 25 avg): 3.2920 | Elapsed: 

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GPT2LMHeadModel(
      (transformer): GPT2Model(
        (wte): Embedding(50257, 768)
        (wpe): Embedding(1024, 768)
        (drop): Dropout(p=0.1, inplace=False)
        (h): ModuleList(
          (0-11): 12 x GPT2Block(
            (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (attn): GPT2Attention(
              (c_attn): lora.Linear(
                (base_layer): Conv1D(nf=2304, nx=768)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=768, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2304, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
          

GPT-2 + LoRA Trial 3 Generation and evaluation

In [ ]:

import os
import json
import time
import gc
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bert_score


# Paths

GPT2_TRIAL3_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "PersonaChat_GPT2/trial_3_conversation_level"
)

GPT2_ADAPTER_DIR = os.path.join(
    GPT2_TRIAL3_DIR, "final_adapter"
)

GPT2_BASELINE_CSV = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "PersonaChat_GPT2/base_model_evaluation/"
    "gpt2_base_validation_generations.csv"
)

GPT2_TRIAL3_GENERATIONS_CSV = os.path.join(
    GPT2_TRIAL3_DIR,
    "gpt2_lora_trial3_validation_generations.csv"
)

GPT2_TRIAL3_METRICS_CSV = os.path.join(
    GPT2_TRIAL3_DIR,
    "gpt2_lora_trial3_generation_metrics_per_example.csv"
)

GPT2_TRIAL3_SUMMARY_JSON = os.path.join(
    GPT2_TRIAL3_DIR,
    "gpt2_lora_trial3_generation_metrics_summary.json"
)

assert os.path.isdir(GPT2_ADAPTER_DIR), (
    f"Trained adapter not found: {GPT2_ADAPTER_DIR}"
)
assert os.path.isfile(GPT2_BASELINE_CSV), (
    f"Baseline generations not found: {GPT2_BASELINE_CSV}"
)

# Reuse exactly the baseline prompts and references.

baseline_df = pd.read_csv(GPT2_BASELINE_CSV).fillna("")

required_columns = [
    "prompt", "reference_response", "generated_response"
]

assert all(column in baseline_df.columns for column in required_columns)
assert len(baseline_df) == 1000, (
    f"Expected 1000 validation examples, found {len(baseline_df)}"
)

prompts = baseline_df["prompt"].astype(str).tolist()
references = baseline_df["reference_response"].astype(str).tolist()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Evaluation device:", device)


# Generate Trial 3 responses, or reuse existing generations.

generation_time_minutes = 0.0

if os.path.isfile(GPT2_TRIAL3_GENERATIONS_CSV):
    print("Existing Trial 3 generations found. Reusing them.")

    trial3_df = pd.read_csv(
        GPT2_TRIAL3_GENERATIONS_CSV
    ).fillna("")

    assert len(trial3_df) == 1000
    assert all(
        column in trial3_df.columns
        for column in required_columns
    )

    assert trial3_df["prompt"].astype(str).tolist() == prompts
    assert (
        trial3_df["reference_response"].astype(str).tolist()
        == references
    )

    predictions = (
        trial3_df["generated_response"].astype(str).tolist()
    )

else:
    print("Loading trained GPT-2 LoRA Trial 3 adapter...")

    tokenizer = AutoTokenizer.from_pretrained(
        GPT2_ADAPTER_DIR
    )
    tokenizer.padding_side = "left"

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        "openai-community/gpt2"
    )

    model = PeftModel.from_pretrained(
        base_model,
        GPT2_ADAPTER_DIR
    )

    model.config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.to(device)
    model.eval()

    BATCH_SIZE = 8
    MAX_NEW_TOKENS = 40

    generation_start = time.time()
    predictions = []

    for start in range(0, len(prompts), BATCH_SIZE):
        batch_prompts = prompts[start:start + BATCH_SIZE]

        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=960
        ).to(device)

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                do_sample=False,
                max_new_tokens=MAX_NEW_TOKENS,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        # Exclude the padded input prompt from decoded responses.
        input_width = inputs["input_ids"].shape[1]
        generated_ids = output_ids[:, input_width:]

        batch_predictions = tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        predictions.extend(
            response.strip() for response in batch_predictions
        )

        if (start + BATCH_SIZE) % 200 == 0:
            print(
                f"Generated {min(start + BATCH_SIZE, len(prompts))}"
                f"/{len(prompts)} responses"
            )

    generation_time_minutes = (
        time.time() - generation_start
    ) / 60

    trial3_df = pd.DataFrame({
        "prompt": prompts,
        "reference_response": references,
        "generated_response": predictions
    })

    assert len(trial3_df) == 1000

    # Save separately; never modify baseline or previous trial files.
    trial3_df.to_csv(
        GPT2_TRIAL3_GENERATIONS_CSV,
        index=False
    )

    print("Generated responses saved:", GPT2_TRIAL3_GENERATIONS_CSV)

    del model, base_model, tokenizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# BLEU — corpus-level word overlap.

references_tokens = [
    text.lower().split() for text in references
]
predictions_tokens = [
    text.lower().split() for text in predictions
]

bleu = corpus_bleu(
    [[reference] for reference in references_tokens],
    predictions_tokens,
    smoothing_function=SmoothingFunction().method1
)


# ROUGE-L — per-example F1.

rouge = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True
)

rouge_l_scores = [
    rouge.score(reference, prediction)["rougeL"].fmeasure
    for reference, prediction in zip(references, predictions)
]

mean_rouge_l = sum(rouge_l_scores) / len(rouge_l_scores)


# Distinct-1 and Distinct-2.

all_unigrams = []
all_bigrams = []

for prediction in predictions_tokens:
    all_unigrams.extend(prediction)
    all_bigrams.extend(
        tuple(prediction[i:i + 2])
        for i in range(max(0, len(prediction) - 1))
    )

distinct_1 = (
    len(set(all_unigrams)) / len(all_unigrams)
    if all_unigrams else 0.0
)

distinct_2 = (
    len(set(all_bigrams)) / len(all_bigrams)
    if all_bigrams else 0.0
)


# Response lengths.

reply_lengths = [
    len(text.split()) for text in predictions
]

average_reply_length = sum(reply_lengths) / len(reply_lengths)
median_reply_length = float(
    pd.Series(reply_lengths).median()
)


# BERTScore — use the same encoder and settings as the baseline.

bert_start = time.time()

P, R, F1 = bert_score(
    cands=predictions,
    refs=references,
    model_type="distilbert-base-uncased",
    lang="en",
    batch_size=16,
    device=str(device),
    verbose=True,
    rescale_with_baseline=False
)

mean_bert_precision = P.mean().item()
mean_bert_recall = R.mean().item()
mean_bert_f1 = F1.mean().item()

bert_time_minutes = (time.time() - bert_start) / 60


# Save per-example metrics.

trial3_df = pd.DataFrame({
    "prompt": prompts,
    "reference_response": references,
    "generated_response": predictions
})

trial3_df["rougeL_f1"] = rouge_l_scores
trial3_df["bert_score_precision"] = P.tolist()
trial3_df["bert_score_recall"] = R.tolist()
trial3_df["bert_score_f1"] = F1.tolist()
trial3_df["generated_word_count"] = reply_lengths

trial3_df.to_csv(
    GPT2_TRIAL3_METRICS_CSV,
    index=False
)


# Save aggregate metrics.

metrics = {
    "model": "GPT-2",
    "stage": "LoRA Trial 3",
    "number_of_validation_conversations": len(trial3_df),
    "training_examples": 17878,
    "validation_loss": 3.0077,
    "validation_perplexity": 20.2418,
    "corpus_bleu": bleu,
    "mean_rougeL_f1": mean_rouge_l,
    "mean_bert_score_precision": mean_bert_precision,
    "mean_bert_score_recall": mean_bert_recall,
    "mean_bert_score_f1": mean_bert_f1,
    "distinct_1": distinct_1,
    "distinct_2": distinct_2,
    "average_reply_length_words": average_reply_length,
    "median_reply_length_words": median_reply_length,
    "empty_reply_count": sum(
        not text.strip() for text in predictions
    ),
    "generation_time_minutes": generation_time_minutes,
    "bert_score_time_minutes": bert_time_minutes,
    "generation_method": "greedy decoding",
    "max_new_tokens": 40
}

with open(
    GPT2_TRIAL3_SUMMARY_JSON,
    "w",
    encoding="utf-8"
) as f:
    json.dump(metrics, f, indent=2)


# Display results.

print("\n===== GPT-2 LoRA TRIAL 3 GENERATION METRICS =====")

for name, value in metrics.items():
    if isinstance(value, float):
        print(f"{name}: {value:.4f}")
    else:
        print(f"{name}: {value}")

print("\nGeneration CSV:", GPT2_TRIAL3_GENERATIONS_CSV)
print("Per-example metrics CSV:", GPT2_TRIAL3_METRICS_CSV)
print("Summary JSON:", GPT2_TRIAL3_SUMMARY_JSON)
print("================================================")

Evaluation device: cuda
Loading trained GPT-2 LoRA Trial 3 adapter...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Generated 200/1000 responses
Generated 400/1000 responses
Generated 600/1000 responses
Generated 800/1000 responses
Generated 1000/1000 responses
Generated responses saved: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/trial_3_conversation_level/gpt2_lora_trial3_validation_generations.csv


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/121 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/63 [00:00<?, ?it/s]

done in 1.88 seconds, 531.61 sentences/sec

===== GPT-2 LoRA TRIAL 3 GENERATION METRICS =====
model: GPT-2
stage: LoRA Trial 3
number_of_validation_conversations: 1000
training_examples: 17878
validation_loss: 3.0077
validation_perplexity: 20.2418
corpus_bleu: 0.0181
mean_rougeL_f1: 0.1411
mean_bert_score_precision: 0.7211
mean_bert_score_recall: 0.7078
mean_bert_score_f1: 0.7140
distinct_1: 0.0863
distinct_2: 0.2650
average_reply_length_words: 10.8970
median_reply_length_words: 10.0000
empty_reply_count: 0
generation_time_minutes: 1.0431
bert_score_time_minutes: 0.0639
generation_method: greedy decoding
max_new_tokens: 40

Generation CSV: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/trial_3_conversation_level/gpt2_lora_trial3_validation_generations.csv
Per-example metrics CSV: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/trial_3_conversation_level/gpt2_lora_trial3_generation_metrics_per_example.csv
Summary JSON: /content/drive/MyDrive/COMP6012_Assignmen

Verify GPT-2 Baseline vs LoRA Trial 3

In [ ]:

# ============================================================
# VERIFY GPT-2 BASELINE vs LoRA TRIAL 3
# Read-only verification: does not train, regenerate, or save
# ============================================================

import os
import json
import pandas as pd

GPT2_BASE_DIR = "/content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/base_model_evaluation"
GPT2_TRIAL3_DIR = "/content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/trial_3_conversation_level"

BASE_GENERATIONS_FILE = os.path.join(
    GPT2_BASE_DIR, "gpt2_base_validation_generations.csv"
)
TRIAL3_GENERATIONS_FILE = os.path.join(
    GPT2_TRIAL3_DIR, "gpt2_lora_trial3_validation_generations.csv"
)
TRIAL3_PER_EXAMPLE_FILE = os.path.join(
    GPT2_TRIAL3_DIR, "gpt2_lora_trial3_generation_metrics_per_example.csv"
)
TRIAL3_SUMMARY_FILE = os.path.join(
    GPT2_TRIAL3_DIR, "gpt2_lora_trial3_generation_metrics_summary.json"
)

# 1. Check that expected files exist
required_files = [
    BASE_GENERATIONS_FILE,
    TRIAL3_GENERATIONS_FILE,
    TRIAL3_PER_EXAMPLE_FILE,
    TRIAL3_SUMMARY_FILE,
]

print("Checking required files...")
for file_path in required_files:
    print(f"{'FOUND' if os.path.isfile(file_path) else 'MISSING'}: {file_path}")

assert all(os.path.isfile(path) for path in required_files), (
    "One or more required files are missing. Check the paths before proceeding."
)

# 2. Load existing generation files
base_df = pd.read_csv(BASE_GENERATIONS_FILE)
trial3_df = pd.read_csv(TRIAL3_GENERATIONS_FILE)

required_columns = ["prompt", "reference_response", "generated_response"]

for name, df in [("Baseline", base_df), ("Trial 3", trial3_df)]:
    missing_columns = [col for col in required_columns if col not in df.columns]
    assert not missing_columns, f"{name} is missing columns: {missing_columns}"

print("\n--- Dataset checks ---")
print(f"Baseline examples: {len(base_df)}")
print(f"Trial 3 examples: {len(trial3_df)}")

assert len(base_df) == 1000, "Baseline should contain 1,000 examples."
assert len(trial3_df) == 1000, "Trial 3 should contain 1,000 examples."

prompts_match = base_df["prompt"].fillna("").equals(
    trial3_df["prompt"].fillna("")
)
references_match = base_df["reference_response"].fillna("").equals(
    trial3_df["reference_response"].fillna("")
)

print(f"Prompts match baseline in the same order: {prompts_match}")
print(f"Reference responses match baseline in the same order: {references_match}")

assert prompts_match, "Prompt mismatch detected."
assert references_match, "Reference-response mismatch detected."

# 3. Check whether generated responses differ
responses_match = base_df["generated_response"].fillna("").equals(
    trial3_df["generated_response"].fillna("")
)
print(f"Generated responses identical to baseline: {responses_match}")

# 4. Inspect Trial 3 metric files
trial3_per_example_df = pd.read_csv(TRIAL3_PER_EXAMPLE_FILE)

with open(TRIAL3_SUMMARY_FILE, "r", encoding="utf-8") as file:
    trial3_summary = json.load(file)

print("\n--- Trial 3 metric files ---")
print(f"Per-example metric rows: {len(trial3_per_example_df)}")
print("Trial 3 summary JSON:")
print(json.dumps(trial3_summary, indent=2, ensure_ascii=False))

# 5. Compare generated-response lengths
def word_count(text):
    if pd.isna(text):
        return 0
    return len(str(text).split())

base_lengths = base_df["generated_response"].apply(word_count)
trial3_lengths = trial3_df["generated_response"].apply(word_count)

print("\n--- Generated-response length comparison ---")
print(f"Baseline average words: {base_lengths.mean():.3f}")
print(f"Baseline median words:  {base_lengths.median():.3f}")
print(f"Baseline empty replies: {(base_lengths == 0).sum()}")

print(f"Trial 3 average words:  {trial3_lengths.mean():.3f}")
print(f"Trial 3 median words:   {trial3_lengths.median():.3f}")
print(f"Trial 3 empty replies:  {(trial3_lengths == 0).sum()}")

print("\nVERIFICATION PASSED: Trial 3 files exist and prompts/references match.")
print("No model training, generation, or file modification was performed.")


Checking required files...
FOUND: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/base_model_evaluation/gpt2_base_validation_generations.csv
FOUND: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/trial_3_conversation_level/gpt2_lora_trial3_validation_generations.csv
FOUND: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/trial_3_conversation_level/gpt2_lora_trial3_generation_metrics_per_example.csv
FOUND: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_GPT2/trial_3_conversation_level/gpt2_lora_trial3_generation_metrics_summary.json

--- Dataset checks ---
Baseline examples: 1000
Trial 3 examples: 1000
Prompts match baseline in the same order: True
Reference responses match baseline in the same order: True
Generated responses identical to baseline: False

--- Trial 3 metric files ---
Per-example metric rows: 1000
Trial 3 summary JSON:
{
  "model": "GPT-2",
  "stage": "LoRA Trial 3",
  "number_of_validation_conversations": 1000,
  "training_e

Evaluating Qwen Base Model

In [ ]:

import os
import csv
import time
import torch
import pandas as pd
from transformers import AutoModelForCausalLM

# Separate baseline output directory
QWEN_BASELINE_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "PersonaChat_Qwen/base_model_evaluation"
)
os.makedirs(QWEN_BASELINE_DIR, exist_ok=True)

QWEN_BASELINE_CSV = os.path.join(
    QWEN_BASELINE_DIR, "qwen_base_validation_generations.csv"
)

# Load the un-fine-tuned base model
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
qwen_base_model = AutoModelForCausalLM.from_pretrained(QWEN_MODEL_NAME)
qwen_base_model.config.pad_token_id = qwen_tokenizer.pad_token_id
qwen_base_model.config.use_cache = True
qwen_base_model.to(DEVICE)
qwen_base_model.eval()

# Use left padding for batched generation
original_padding_side = qwen_tokenizer.padding_side
original_truncation_side = qwen_tokenizer.truncation_side
qwen_tokenizer.padding_side = "left"
qwen_tokenizer.truncation_side = "left"

# Ensure reproducible greedy generation
generation_batch_size = 8
max_new_tokens = 40
generation_rows = []
generation_start = time.time()

try:
    for start in range(
        0,
        len(persona_validation_conversation_examples),
        generation_batch_size
    ):
        examples = persona_validation_conversation_examples[
            start:start + generation_batch_size
        ]

        # Convert the prepared PersonaChat prompts to Qwen chat prompts
        prompts = []

        for example in examples:
            raw_prompt = example["prompt"]

            if "\n\nConversation:\n" not in raw_prompt:
                raise ValueError(
                    "Unexpected PersonaChat prompt format."
                )

            persona_section, history_section = raw_prompt.split(
                "\n\nConversation:\n", 1
            )

            persona_text = persona_section.replace(
                "Persona:\n", "", 1
            ).strip()

            history_text = history_section.rsplit(
                "\nAssistant:", 1
            )[0].strip()

            messages = [
                {
                    "role": "system",
                    "content": (
                        "You are a helpful conversational partner. "
                        "Stay consistent with this persona:\n"
                        f"{persona_text}"
                    ),
                },
                {
                    "role": "user",
                    "content": (
                        "Continue this conversation naturally. "
                        "Conversation history:\n"
                        f"{history_text}"
                    ),
                },
            ]

            formatted_prompt = qwen_tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
            prompts.append(formatted_prompt)

        encoded = qwen_tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=280,
            add_special_tokens=False,
        )
        encoded = {
            key: value.to(DEVICE)
            for key, value in encoded.items()
        }
        prompt_width = encoded["input_ids"].shape[1]

        with torch.inference_mode():
            generated = qwen_base_model.generate(
                **encoded,
                do_sample=False,
                max_new_tokens=max_new_tokens,
                pad_token_id=qwen_tokenizer.pad_token_id,
                eos_token_id=qwen_tokenizer.convert_tokens_to_ids(
                    "<|im_end|>"
                ),
            )

        # Decode only newly generated tokens
        continuations = generated[:, prompt_width:]
        replies = qwen_tokenizer.batch_decode(
            continuations, skip_special_tokens=True
        )

        for example, reply in zip(examples, replies):
            generation_rows.append({
                "prompt": example["prompt"],
                "reference_response": (
                    example["reference_response"]
                    if "reference_response" in example
                    else example["response"]
                ),
                "generated_response": reply.strip(),
            })

        if (
            (start + len(examples)) % 100 == 0
            or start + len(examples)
            == len(persona_validation_conversation_examples)
        ):
            print(
                f"Generated {start + len(examples)}/"
                f"{len(persona_validation_conversation_examples)} replies"
            )

finally:
    qwen_tokenizer.padding_side = original_padding_side
    qwen_tokenizer.truncation_side = original_truncation_side

# Save baseline generations
pd.DataFrame(generation_rows).to_csv(
    QWEN_BASELINE_CSV,
    index=False,
    quoting=csv.QUOTE_MINIMAL
)

elapsed_minutes = (time.time() - generation_start) / 60
print("\n===== BASE QWEN GENERATION RESULTS =====")
print("Replies generated:", len(generation_rows))
print("Generation time (minutes):", round(elapsed_minutes, 2))
print("CSV:", QWEN_BASELINE_CSV)
print("\nSample outputs:")

for row in generation_rows[:3]:
    print("\nPROMPT:", row["prompt"][-300:])
    print("REFERENCE:", row["reference_response"])
    print("GENERATED:", row["generated_response"])


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Generated 200/1000 replies
Generated 400/1000 replies
Generated 600/1000 replies
Generated 800/1000 replies
Generated 1000/1000 replies

===== BASE QWEN GENERATION RESULTS =====
Replies generated: 1000
Generation time (minutes): 5.26
CSV: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_Qwen/base_model_evaluation/qwen_base_validation_generations.csv

Sample outputs:

PROMPT:  double , it is so much fun and hard work .
yes i bet you can get hurt . my wife works and i stay at home
nice , i only have one parent so now i help out my mom .
i bet she appreciates that very much .
she raised me right , i am just like her .
my dad was always busy working at home depot
Assistant:
REFERENCE: now that i am older home depot is my toy r us .
GENERATED: That's great to hear! It sounds like you're doing well, both physically and emotionally. Watching movies has been a great way for you to unwind after a long day at work. Have you noticed

PROMPT:  people .
i didn't think about the crowd aspect 

Calculating Base Qwen Validation Loss and Perplexity

In [ ]:

import math
import time
import torch
from torch.utils.data import DataLoader

qwen_base_model.eval()

base_validation_loader = DataLoader(
    qwen_validation_conversation_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=qwen_collator,
)

total_nll = 0.0
total_target_tokens = 0
validation_start = time.time()

with torch.inference_mode():
    for batch_index, batch in enumerate(base_validation_loader):
        batch = {
            key: value.to(DEVICE, non_blocking=True)
            for key, value in batch.items()
        }

        outputs = qwen_base_model(**batch)
        target_tokens = int(
            (batch["labels"][:, 1:] != -100).sum().item()
        )

        # Qwen causal LM shifts labels internally, so weight loss
        # by the number of non-masked target tokens after the shift.
        if target_tokens > 0:
            total_nll += outputs.loss.item() * target_tokens
            total_target_tokens += target_tokens

        if (batch_index + 1) % 20 == 0:
            print(
                f"Validated {min((batch_index + 1) * 16, 1000)}/1000 "
                "conversations"
            )

assert total_target_tokens > 0, "No target tokens were evaluated."

base_validation_loss = total_nll / total_target_tokens
base_validation_perplexity = math.exp(base_validation_loss)

base_validation_results = {
    "model": "Qwen2.5-0.5B-Instruct",
    "stage": "base",
    "validation_conversations": len(qwen_validation_conversation_dataset),
    "target_tokens": total_target_tokens,
    "validation_loss": base_validation_loss,
    "validation_perplexity": base_validation_perplexity,
    "validation_time_minutes": (time.time() - validation_start) / 60,
}

print("\n===== BASE QWEN VALIDATION METRICS =====")
for key, value in base_validation_results.items():
    print(
        f"{key}: {value:.4f}"
        if isinstance(value, float)
        else f"{key}: {value}"
    )


Validated 320/1000 conversations
Validated 640/1000 conversations
Validated 960/1000 conversations

===== BASE QWEN VALIDATION METRICS =====
model: Qwen2.5-0.5B-Instruct
stage: base
validation_conversations: 1000
target_tokens: 12515
validation_loss: 4.4796
validation_perplexity: 88.1976
validation_time_minutes: 2.5420


Calculating all base Qwen genration metrics

In [ ]:

import os
import json
import time
import pandas as pd
import nltk
import torch

from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bert_score


# Paths and input checks

QWEN_BASELINE_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "PersonaChat_Qwen/base_model_evaluation"
)
QWEN_BASELINE_CSV = os.path.join(
    QWEN_BASELINE_DIR, "qwen_base_validation_generations.csv"
)

assert os.path.isfile(QWEN_BASELINE_CSV), (
    f"Baseline CSV not found: {QWEN_BASELINE_CSV}"
)

df = pd.read_csv(QWEN_BASELINE_CSV).fillna("")
required_columns = [
    "prompt", "reference_response", "generated_response"
]
assert all(column in df.columns for column in required_columns), (
    f"CSV must contain columns: {required_columns}"
)
assert len(df) == 1000, f"Expected 1000 rows, found {len(df)}"

references = df["reference_response"].astype(str).tolist()
predictions = df["generated_response"].astype(str).tolist()


# BLEU — corpus-level word overlap

references_tokens = [text.lower().split() for text in references]
predictions_tokens = [text.lower().split() for text in predictions]

bleu = corpus_bleu(
    [[reference] for reference in references_tokens],
    predictions_tokens,
    smoothing_function=SmoothingFunction().method1,
)


# ROUGE-L — F-measure

rouge = rouge_scorer.RougeScorer(
    ["rougeL"], use_stemmer=True
)

rouge_l_scores = [
    rouge.score(reference, prediction)["rougeL"].fmeasure
    for reference, prediction in zip(references, predictions)
]
mean_rouge_l = sum(rouge_l_scores) / len(rouge_l_scores)


# Distinct-1 and Distinct-2 — unique n-grams / total n-grams

all_unigrams = []
all_bigrams = []

for prediction in predictions_tokens:
    all_unigrams.extend(prediction)
    all_bigrams.extend(
        tuple(prediction[i:i + 2])
        for i in range(max(0, len(prediction) - 1))
    )

distinct_1 = (
    len(set(all_unigrams)) / len(all_unigrams)
    if all_unigrams else 0.0
)
distinct_2 = (
    len(set(all_bigrams)) / len(all_bigrams)
    if all_bigrams else 0.0
)


# Average reply length (words)

reply_lengths = [len(text.split()) for text in predictions]
average_reply_length = sum(reply_lengths) / len(reply_lengths)


# BERTScore — semantic similarity
# Use DistilBERT to keep this evaluation manageable on a T4.

device = "cuda" if torch.cuda.is_available() else "cpu"
bert_start = time.time()

P, R, F1 = bert_score(
    cands=predictions,
    refs=references,
    model_type="distilbert-base-uncased",
    lang="en",
    batch_size=16,
    device=device,
    verbose=True,
    rescale_with_baseline=False,
)

mean_bert_precision = P.mean().item()
mean_bert_recall = R.mean().item()
mean_bert_f1 = F1.mean().item()
bert_time_minutes = (time.time() - bert_start) / 60


# Save per-example and aggregate results

df["bleu_sentence_not_used"] = ""  # Corpus BLEU is reported below.
df["rougeL_f1"] = rouge_l_scores
df["bert_score_precision"] = P.tolist()
df["bert_score_recall"] = R.tolist()
df["bert_score_f1"] = F1.tolist()
df["generated_word_count"] = reply_lengths

per_example_path = os.path.join(
    QWEN_BASELINE_DIR, "qwen_base_generation_metrics_per_example.csv"
)
df.to_csv(per_example_path, index=False)

metrics = {
    "model": "Qwen2.5-0.5B-Instruct",
    "stage": "base",
    "number_of_validation_conversations": len(df),
    "corpus_bleu": bleu,
    "mean_rougeL_f1": mean_rouge_l,
    "mean_bert_score_precision": mean_bert_precision,
    "mean_bert_score_recall": mean_bert_recall,
    "mean_bert_score_f1": mean_bert_f1,
    "distinct_1": distinct_1,
    "distinct_2": distinct_2,
    "average_reply_length_words": average_reply_length,
    "median_reply_length_words": float(pd.Series(reply_lengths).median()),
    "empty_reply_count": sum(not text.strip() for text in predictions),
    "bert_score_time_minutes": bert_time_minutes,
}

summary_path = os.path.join(
    QWEN_BASELINE_DIR, "qwen_base_generation_metrics_summary.json"
)
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("\n===== BASE QWEN GENERATION METRICS =====")
for name, value in metrics.items():
    if isinstance(value, float):
        print(f"{name}: {value:.4f}")
    else:
        print(f"{name}: {value}")

print("\nPer-example metrics CSV:", per_example_path)
print("Summary JSON:", summary_path)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

...

calculating scores...
computing bert embedding.


  0%|          | 0/125 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/63 [00:00<?, ?it/s]

done in 2.54 seconds, 393.59 sentences/sec

===== BASE QWEN GENERATION METRICS =====
model: Qwen2.5-0.5B-Instruct
stage: base
number_of_validation_conversations: 1000
corpus_bleu: 0.0014
mean_rougeL_f1: 0.0937
mean_bert_score_precision: 0.6811
mean_bert_score_recall: 0.7304
mean_bert_score_f1: 0.7046
distinct_1: 0.1174
distinct_2: 0.4061
average_reply_length_words: 29.4560
median_reply_length_words: 32.0000
empty_reply_count: 0
bert_score_time_minutes: 0.1431

Per-example metrics CSV: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_Qwen/base_model_evaluation/qwen_base_generation_metrics_per_example.csv
Summary JSON: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_Qwen/base_model_evaluation/qwen_base_generation_metrics_summary.json


Configuring Qwen + LoRA Trail 1

Learning rate: 2e-4

Epoch: 1

Batch size: 4

Gradient Accumulation: 4

Effective Batch Size: 4X4 = 16

Maximum sequence length: 320



In [ ]:

import sys
import os
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from torch.optim import AdamW

# torchao package created a problem for compatibility issue with this peft. so turning this off
sys.modules["torchao"] = None

# Trial 1 configuration — conversation-level examples
QWEN_TRIAL_NAME = "qwen_personachat_conversation_trial1"
QWEN_LEARNING_RATE = 2e-4
QWEN_EPOCHS = 1
QWEN_BATCH_SIZE = 4
QWEN_GRAD_ACCUMULATION_STEPS = 4
QWEN_EFFECTIVE_BATCH_SIZE = (
    QWEN_BATCH_SIZE * QWEN_GRAD_ACCUMULATION_STEPS
)
QWEN_MAX_GRAD_NORM = 1.0
QWEN_CHECKPOINT_EVERY = 250
QWEN_MAX_LENGTH = 320
QWEN_SEED = 42

QWEN_CONVERSATION_TRIAL1_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "PersonaChat_Qwen/trial_1_conversation_level"
)
QWEN_CONVERSATION_TRIAL1_DIR = os.path.abspath(
    QWEN_CONVERSATION_TRIAL1_DIR
)
os.makedirs(QWEN_CONVERSATION_TRIAL1_DIR, exist_ok=True)

# Reproducibility
torch.manual_seed(QWEN_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(QWEN_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the base Qwen model
qwen_conversation_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_NAME
)
qwen_conversation_model.config.pad_token_id = qwen_tokenizer.pad_token_id
qwen_conversation_model.config.use_cache = False

# LoRA configuration
qwen_conversation_lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
)

qwen_conversation_model = get_peft_model(
    qwen_conversation_model,
    qwen_conversation_lora_config
)
qwen_conversation_model.to(DEVICE)

# Optimizer
qwen_conversation_optimizer = AdamW(
    [
        p for p in qwen_conversation_model.parameters()
        if p.requires_grad
    ],
    lr=QWEN_LEARNING_RATE,
    weight_decay=0.01,
)

# Training size and step counts
QWEN_MICRO_BATCHES_PER_EPOCH = (
    len(qwen_train_conversation_dataset) + QWEN_BATCH_SIZE - 1
) // QWEN_BATCH_SIZE

QWEN_STEPS_PER_EPOCH = (
    QWEN_MICRO_BATCHES_PER_EPOCH
    + QWEN_GRAD_ACCUMULATION_STEPS - 1
) // QWEN_GRAD_ACCUMULATION_STEPS

QWEN_TOTAL_STEPS = QWEN_STEPS_PER_EPOCH * QWEN_EPOCHS

print("Model configured successfully.")
print("Device:", DEVICE)
print("Trainable parameters:")
qwen_conversation_model.print_trainable_parameters()
print("Training examples:", len(qwen_train_conversation_dataset))
print("Validation examples:", len(qwen_validation_conversation_dataset))
print("Micro-batch size:", QWEN_BATCH_SIZE)
print("Gradient accumulation steps:", QWEN_GRAD_ACCUMULATION_STEPS)
print("Effective batch size:", QWEN_EFFECTIVE_BATCH_SIZE)
print("Steps per epoch:", QWEN_STEPS_PER_EPOCH)
print("Total planned optimizer steps:", QWEN_TOTAL_STEPS)
print("Learning rate:", QWEN_LEARNING_RATE)
print("Output directory:", QWEN_CONVERSATION_TRIAL1_DIR)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model configured successfully.
Device: cuda
Trainable parameters:
trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184
Training examples: 17878
Validation examples: 1000
Micro-batch size: 4
Gradient accumulation steps: 4
Effective batch size: 16
Steps per epoch: 1118
Total planned optimizer steps: 1118
Learning rate: 0.0002
Output directory: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_Qwen/trial_1_conversation_level


Training Qwen + LoRA Trial 1

In [ ]:

import os
import time
import math
import torch
from torch.utils.data import DataLoader
from torch.nn.utils import clip_grad_norm_


# Deterministic DataLoader: recreates the same order on resume

def make_epoch_loader(epoch_index):
    generator = torch.Generator()
    generator.manual_seed(QWEN_SEED + epoch_index)

    return DataLoader(
        qwen_train_conversation_dataset,
        batch_size=QWEN_BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        collate_fn=qwen_collator,
    )


# Checkpoint paths and mixed precision

QWEN_CHECKPOINT_PATH = os.path.join(
    QWEN_CONVERSATION_TRIAL1_DIR,
    "training_checkpoint.pt",
)
QWEN_FINAL_ADAPTER_DIR = os.path.join(
    QWEN_CONVERSATION_TRIAL1_DIR,
    "final_adapter",
)

use_amp = DEVICE.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

global_step = 0
start_epoch = 0
start_batch = 0
loss_history = []
elapsed_before = 0.0


# Resume from checkpoint, if present

if os.path.isfile(QWEN_CHECKPOINT_PATH):
    checkpoint = torch.load(
        QWEN_CHECKPOINT_PATH,
        map_location=DEVICE,
        weights_only=False,
    )

    qwen_conversation_model.load_state_dict(checkpoint["model_state"])
    qwen_conversation_optimizer.load_state_dict(
        checkpoint["optimizer_state"]
    )

    if "scaler_state" in checkpoint:
        scaler.load_state_dict(checkpoint["scaler_state"])

    global_step = checkpoint["global_step"]
    start_epoch = checkpoint["next_epoch"]
    start_batch = checkpoint["next_batch"]
    loss_history = checkpoint.get("loss_history", [])
    elapsed_before = checkpoint.get("elapsed_seconds", 0.0)

    torch.set_rng_state(checkpoint["torch_rng_state"].cpu())
    if (
        torch.cuda.is_available()
        and checkpoint.get("cuda_rng_state") is not None
    ):
        torch.cuda.set_rng_state_all(checkpoint["cuda_rng_state"])

    print(
        f"Resuming at epoch {start_epoch + 1}, "
        f"batch {start_batch + 1}, global step {global_step}."
    )
else:
    print("No existing training checkpoint. Starting a fresh run.")


# Atomic checkpoint saving

def save_checkpoint(next_epoch, next_batch, elapsed_seconds):
    checkpoint_data = {
        "model_state": qwen_conversation_model.state_dict(),
        "optimizer_state": qwen_conversation_optimizer.state_dict(),
        "scaler_state": scaler.state_dict(),
        "global_step": global_step,
        "next_epoch": next_epoch,
        "next_batch": next_batch,
        "loss_history": loss_history,
        "elapsed_seconds": elapsed_seconds,
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else None
        ),
        "trial_name": QWEN_TRIAL_NAME,
        "learning_rate": QWEN_LEARNING_RATE,
        "batch_size": QWEN_BATCH_SIZE,
        "gradient_accumulation_steps": QWEN_GRAD_ACCUMULATION_STEPS,
        "seed": QWEN_SEED,
    }

    temporary_path = QWEN_CHECKPOINT_PATH + ".tmp"
    torch.save(checkpoint_data, temporary_path)
    os.replace(temporary_path, QWEN_CHECKPOINT_PATH)


# Training

qwen_conversation_model.train()
run_start = time.time()

for epoch_index in range(start_epoch, QWEN_EPOCHS):
    train_loader = make_epoch_loader(epoch_index)
    batch_offset = start_batch if epoch_index == start_epoch else 0

    qwen_conversation_optimizer.zero_grad(set_to_none=True)
    accumulated_micro_batches = 0

    for batch_index, batch in enumerate(train_loader):
        if batch_index < batch_offset:
            continue

        batch = {
            key: value.to(DEVICE, non_blocking=True)
            for key, value in batch.items()
        }

        # Correctly scale the final, possibly incomplete accumulation group.
        group_start = (
            batch_index // QWEN_GRAD_ACCUMULATION_STEPS
        ) * QWEN_GRAD_ACCUMULATION_STEPS
        group_size = min(
            QWEN_GRAD_ACCUMULATION_STEPS,
            len(train_loader) - group_start,
        )

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=use_amp,
        ):
            outputs = qwen_conversation_model(**batch)
            loss = outputs.loss

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss at epoch {epoch_index + 1}, "
                f"batch {batch_index + 1}: {loss.item()}"
            )

        scaler.scale(loss / group_size).backward()
        accumulated_micro_batches += 1
        loss_history.append(float(loss.detach().cpu()))

        is_accumulation_boundary = (
            accumulated_micro_batches >= group_size
        )
        is_last_batch = batch_index + 1 >= len(train_loader)

        if is_accumulation_boundary or is_last_batch:
            scaler.unscale_(qwen_conversation_optimizer)
            clip_grad_norm_(
                qwen_conversation_model.parameters(),
                QWEN_MAX_GRAD_NORM,
            )
            scaler.step(qwen_conversation_optimizer)
            scaler.update()
            qwen_conversation_optimizer.zero_grad(set_to_none=True)

            global_step += 1
            accumulated_micro_batches = 0

            if global_step % 25 == 0:
                recent_losses = loss_history[-25:]
                recent_loss = sum(recent_losses) / len(recent_losses)
                elapsed = elapsed_before + time.time() - run_start

                print(
                    f"Epoch {epoch_index + 1}/{QWEN_EPOCHS} | "
                    f"Step {global_step}/{QWEN_TOTAL_STEPS} | "
                    f"Micro-batch loss (last 25 avg): "
                    f"{recent_loss:.4f} | "
                    f"Elapsed: {elapsed / 60:.1f} min"
                )

            next_epoch = epoch_index
            next_batch = batch_index + 1

            if next_batch >= len(train_loader):
                next_epoch = epoch_index + 1
                next_batch = 0

            if (
                global_step % QWEN_CHECKPOINT_EVERY == 0
                or next_epoch >= QWEN_EPOCHS
            ):
                elapsed = elapsed_before + time.time() - run_start
                save_checkpoint(next_epoch, next_batch, elapsed)

                print(
                    f"Checkpoint saved at step {global_step}; "
                    f"next position: epoch {next_epoch + 1}, "
                    f"batch {next_batch + 1}."
                )

    start_batch = 0

elapsed_total = elapsed_before + time.time() - run_start


# Save final LoRA adapter and tokenizer

os.makedirs(QWEN_FINAL_ADAPTER_DIR, exist_ok=True)
qwen_conversation_model.save_pretrained(QWEN_FINAL_ADAPTER_DIR)
qwen_tokenizer.save_pretrained(QWEN_FINAL_ADAPTER_DIR)


# Validation loss and perplexity

qwen_conversation_model.eval()
validation_loss_sum = 0.0
validation_batches = 0

validation_loader = DataLoader(
    qwen_validation_conversation_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=qwen_collator,
)

with torch.inference_mode():
    for batch in validation_loader:
        batch = {
            key: value.to(DEVICE, non_blocking=True)
            for key, value in batch.items()
        }

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=use_amp,
        ):
            outputs = qwen_conversation_model(**batch)

        if torch.isfinite(outputs.loss):
            validation_loss_sum += outputs.loss.item()
            validation_batches += 1
        else:
            raise RuntimeError("Non-finite validation loss detected.")

validation_loss = validation_loss_sum / validation_batches
validation_perplexity = math.exp(validation_loss)

print("\n========== QWEN CONVERSATION-LEVEL TRIAL 1 ==========")
print("Training examples:", len(qwen_train_conversation_dataset))
print("Validation examples:", len(qwen_validation_conversation_dataset))
print("Optimizer steps completed:", global_step)
print("Mean training loss:", round(sum(loss_history) / len(loss_history), 4))
print("Validation loss:", round(validation_loss, 4))
print("Validation perplexity:", round(validation_perplexity, 4))
print("Elapsed training time (minutes):", round(elapsed_total / 60, 2))
print("Final adapter:", QWEN_FINAL_ADAPTER_DIR)
print("Training checkpoint:", QWEN_CHECKPOINT_PATH)
print("=====================================================")

qwen_conversation_model.train()


No existing training checkpoint. Starting a fresh run.
Epoch 1/1 | Step 25/1118 | Micro-batch loss (last 25 avg): 3.0860 | Elapsed: 0.6 min
Epoch 1/1 | Step 50/1118 | Micro-batch loss (last 25 avg): 2.8656 | Elapsed: 1.2 min
Epoch 1/1 | Step 75/1118 | Micro-batch loss (last 25 avg): 2.8498 | Elapsed: 1.8 min
Epoch 1/1 | Step 100/1118 | Micro-batch loss (last 25 avg): 2.9431 | Elapsed: 2.2 min
Epoch 1/1 | Step 125/1118 | Micro-batch loss (last 25 avg): 2.7916 | Elapsed: 2.7 min
Epoch 1/1 | Step 150/1118 | Micro-batch loss (last 25 avg): 2.9896 | Elapsed: 3.1 min
Epoch 1/1 | Step 175/1118 | Micro-batch loss (last 25 avg): 2.8826 | Elapsed: 3.6 min
Epoch 1/1 | Step 200/1118 | Micro-batch loss (last 25 avg): 2.9601 | Elapsed: 4.0 min
Epoch 1/1 | Step 225/1118 | Micro-batch loss (last 25 avg): 2.9787 | Elapsed: 4.5 min
Epoch 1/1 | Step 250/1118 | Micro-batch loss (last 25 avg): 2.7281 | Elapsed: 4.9 min
Checkpoint saved at step 250; next position: epoch 1, batch 1001.
Epoch 1/1 | Step 275/1

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 896)
        (layers): ModuleList(
          (0-23): 24 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=896, out_features=896, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=896, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=896, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
      

Qwen + LoRA Trial 1 generation and evaluation

In [ ]:

import os
import json
import time
import gc
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bert_score


# Paths

QWEN_TRIAL1_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "PersonaChat_Qwen/trial_1_conversation_level"
)

QWEN_ADAPTER_DIR = os.path.join(
    QWEN_TRIAL1_DIR, "final_adapter"
)

QWEN_BASELINE_CSV = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "PersonaChat_Qwen/base_model_evaluation/"
    "qwen_base_validation_generations.csv"
)

QWEN_TRIAL1_GENERATIONS_CSV = os.path.join(
    QWEN_TRIAL1_DIR,
    "qwen_lora_trial1_validation_generations.csv"
)

QWEN_TRIAL1_METRICS_CSV = os.path.join(
    QWEN_TRIAL1_DIR,
    "qwen_lora_trial1_generation_metrics_per_example.csv"
)

QWEN_TRIAL1_SUMMARY_JSON = os.path.join(
    QWEN_TRIAL1_DIR,
    "qwen_lora_trial1_generation_metrics_summary.json"
)

assert os.path.isdir(QWEN_ADAPTER_DIR), (
    f"Trained adapter not found: {QWEN_ADAPTER_DIR}"
)
assert os.path.isfile(QWEN_BASELINE_CSV), (
    f"Baseline generations not found: {QWEN_BASELINE_CSV}"
)

# Reuse exactly the baseline prompts and references.

baseline_df = pd.read_csv(QWEN_BASELINE_CSV).fillna("")

required_columns = [
    "prompt", "reference_response", "generated_response"
]

assert all(column in baseline_df.columns for column in required_columns)
assert len(baseline_df) == 1000, (
    f"Expected 1000 validation examples, found {len(baseline_df)}"
)

prompts = baseline_df["prompt"].astype(str).tolist()
references = baseline_df["reference_response"].astype(str).tolist()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Evaluation device:", device)


# Generate Trial 1 responses, or reuse existing generations.

generation_time_minutes = 0.0

if os.path.isfile(QWEN_TRIAL1_GENERATIONS_CSV):
    print("Existing Trial 1 generations found. Reusing them.")

    trial1_df = pd.read_csv(
        QWEN_TRIAL1_GENERATIONS_CSV
    ).fillna("")

    assert len(trial1_df) == 1000
    assert all(
        column in trial1_df.columns
        for column in required_columns
    )

    assert trial1_df["prompt"].astype(str).tolist() == prompts
    assert (
        trial1_df["reference_response"].astype(str).tolist()
        == references
    )

    predictions = (
        trial1_df["generated_response"].astype(str).tolist()
    )

else:
    print("Loading trained Qwen LoRA Trial 1 adapter...")

    tokenizer = AutoTokenizer.from_pretrained(
        QWEN_ADAPTER_DIR
    )
    tokenizer.padding_side = "left"
    tokenizer.truncation_side = "left"

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        QWEN_MODEL_NAME
    )

    model = PeftModel.from_pretrained(
        base_model,
        QWEN_ADAPTER_DIR
    )

    model.config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = True
    model.to(device)
    model.eval()

    BATCH_SIZE = 8
    MAX_NEW_TOKENS = 40
    QWEN_EOS_TOKEN_ID = tokenizer.convert_tokens_to_ids(
        "<|im_end|>"
    )

    generation_start = time.time()
    predictions = []

    for start in range(0, len(prompts), BATCH_SIZE):
        batch_prompts = prompts[start:start + BATCH_SIZE]
        formatted_prompts = []

        for raw_prompt in batch_prompts:
            if "\n\nConversation:\n" not in raw_prompt:
                raise ValueError(
                    "Unexpected prompt format. Expected "
                    "'\\n\\nConversation:\\n' in each prompt."
                )

            persona_text, history = raw_prompt.split(
                "\n\nConversation:\n",
                1
            )

            messages = [
                {
                    "role": "system",
                    "content": (
                        "You are a helpful conversational partner. "
                        "Stay consistent with this persona:\n"
                        f"{persona_text}"
                    )
                },
                {
                    "role": "user",
                    "content": (
                        "Continue this conversation naturally. "
                        "Conversation history:\n"
                        f"{history}"
                    )
                }
            ]

            formatted_prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            formatted_prompts.append(formatted_prompt)

        inputs = tokenizer(
            formatted_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=280,
            add_special_tokens=False
        ).to(device)

        with torch.inference_mode():
            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                output_ids = model.generate(
                    **inputs,
                    do_sample=False,
                    max_new_tokens=MAX_NEW_TOKENS,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=QWEN_EOS_TOKEN_ID,
                    use_cache=True
                )

        # Exclude the padded input prompt from decoded responses.
        input_width = inputs["input_ids"].shape[1]
        generated_ids = output_ids[:, input_width:]

        batch_predictions = tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        predictions.extend(
            response.strip() for response in batch_predictions
        )

        completed = min(start + len(batch_prompts), len(prompts))

        if completed % 200 == 0 or completed == len(prompts):
            print(
                f"Generated {completed}/{len(prompts)} responses"
            )

    generation_time_minutes = (
        time.time() - generation_start
    ) / 60

    trial1_df = pd.DataFrame({
        "prompt": prompts,
        "reference_response": references,
        "generated_response": predictions
    })

    assert len(trial1_df) == 1000

    # Save separately; never modify baseline files.
    trial1_df.to_csv(
        QWEN_TRIAL1_GENERATIONS_CSV,
        index=False
    )

    print("Generated responses saved:", QWEN_TRIAL1_GENERATIONS_CSV)

    del model, base_model, tokenizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# BLEU — corpus-level word overlap.

references_tokens = [
    text.lower().split() for text in references
]
predictions_tokens = [
    text.lower().split() for text in predictions
]

bleu = corpus_bleu(
    [[reference] for reference in references_tokens],
    predictions_tokens,
    smoothing_function=SmoothingFunction().method1
)


# ROUGE-L — per-example F1.

rouge = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True
)

rouge_l_scores = [
    rouge.score(reference, prediction)["rougeL"].fmeasure
    for reference, prediction in zip(references, predictions)
]

mean_rouge_l = sum(rouge_l_scores) / len(rouge_l_scores)


# Distinct-1 and Distinct-2.

all_unigrams = []
all_bigrams = []

for prediction in predictions_tokens:
    all_unigrams.extend(prediction)
    all_bigrams.extend(
        tuple(prediction[i:i + 2])
        for i in range(max(0, len(prediction) - 1))
    )

distinct_1 = (
    len(set(all_unigrams)) / len(all_unigrams)
    if all_unigrams else 0.0
)

distinct_2 = (
    len(set(all_bigrams)) / len(all_bigrams)
    if all_bigrams else 0.0
)


# Response lengths.

reply_lengths = [
    len(text.split()) for text in predictions
]

average_reply_length = sum(reply_lengths) / len(reply_lengths)
median_reply_length = float(
    pd.Series(reply_lengths).median()
)


# BERTScore — use the same encoder and settings as the baseline.

bert_start = time.time()

P, R, F1 = bert_score(
    cands=predictions,
    refs=references,
    model_type="distilbert-base-uncased",
    lang="en",
    batch_size=16,
    device=str(device),
    verbose=True,
    rescale_with_baseline=False
)

mean_bert_precision = P.mean().item()
mean_bert_recall = R.mean().item()
mean_bert_f1 = F1.mean().item()

bert_time_minutes = (time.time() - bert_start) / 60


# Save per-example metrics.

trial1_df["rougeL_f1"] = rouge_l_scores
trial1_df["bert_score_precision"] = P.tolist()
trial1_df["bert_score_recall"] = R.tolist()
trial1_df["bert_score_f1"] = F1.tolist()
trial1_df["generated_word_count"] = reply_lengths

trial1_df.to_csv(
    QWEN_TRIAL1_METRICS_CSV,
    index=False
)


# Save aggregate metrics.

metrics = {
    "model": QWEN_MODEL_NAME,
    "stage": "LoRA Trial 1",
    "number_of_validation_conversations": len(trial1_df),
    "training_examples": 17878,
    "validation_loss": 2.6967,
    "validation_perplexity": 14.83,
    "corpus_bleu": bleu,
    "mean_rougeL_f1": mean_rouge_l,
    "mean_bert_score_precision": mean_bert_precision,
    "mean_bert_score_recall": mean_bert_recall,
    "mean_bert_score_f1": mean_bert_f1,
    "distinct_1": distinct_1,
    "distinct_2": distinct_2,
    "average_reply_length_words": average_reply_length,
    "median_reply_length_words": median_reply_length,
    "empty_reply_count": sum(
        not text.strip() for text in predictions
    ),
    "generation_time_minutes": generation_time_minutes,
    "bert_score_time_minutes": bert_time_minutes,
    "generation_method": "greedy decoding",
    "max_new_tokens": 40
}

with open(
    QWEN_TRIAL1_SUMMARY_JSON,
    "w",
    encoding="utf-8"
) as f:
    json.dump(metrics, f, indent=2)


# Display results.

print("\n===== QWEN LoRA TRIAL 1 GENERATION METRICS =====")

for name, value in metrics.items():
    if isinstance(value, float):
        print(f"{name}: {value:.4f}")
    else:
        print(f"{name}: {value}")

print("\nGeneration CSV:", QWEN_TRIAL1_GENERATIONS_CSV)
print("Per-example metrics CSV:", QWEN_TRIAL1_METRICS_CSV)
print("Summary JSON:", QWEN_TRIAL1_SUMMARY_JSON)
print("================================================")


Evaluation device: cuda
Loading trained Qwen LoRA Trial 1 adapter...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Generated 200/1000 responses
Generated 400/1000 responses
Generated 600/1000 responses
Generated 800/1000 responses
Generated 1000/1000 responses
Generated responses saved: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_Qwen/trial_1_conversation_level/qwen_lora_trial1_validation_generations.csv


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/105 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/63 [00:00<?, ?it/s]

done in 1.54 seconds, 650.20 sentences/sec

===== QWEN LoRA TRIAL 1 GENERATION METRICS =====
model: Qwen/Qwen2.5-0.5B-Instruct
stage: LoRA Trial 1
number_of_validation_conversations: 1000
training_examples: 17878
validation_loss: 2.6967
validation_perplexity: 14.8300
corpus_bleu: 0.0139
mean_rougeL_f1: 0.1086
mean_bert_score_precision: 0.6998
mean_bert_score_recall: 0.6930
mean_bert_score_f1: 0.6960
distinct_1: 0.0996
distinct_2: 0.3657
average_reply_length_words: 7.7700
median_reply_length_words: 7.0000
empty_reply_count: 0
generation_time_minutes: 1.8629
bert_score_time_minutes: 0.0569
generation_method: greedy decoding
max_new_tokens: 40

Generation CSV: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_Qwen/trial_1_conversation_level/qwen_lora_trial1_validation_generations.csv
Per-example metrics CSV: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_Qwen/trial_1_conversation_level/qwen_lora_trial1_generation_metrics_per_example.csv
Summary JSON: /content/drive/MyDrive/

Verifying Qwen Baseline vs LoRA Trial 1 and Comparing Generation Lengths

In [ ]:
import os
import json
import pandas as pd

QWEN_BASELINE_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "PersonaChat_Qwen/base_model_evaluation"
)

QWEN_TRIAL1_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "PersonaChat_Qwen/trial_1_conversation_level"
)

baseline_csv = os.path.join(
    QWEN_BASELINE_DIR,
    "qwen_base_validation_generations.csv"
)

trial1_csv = os.path.join(
    QWEN_TRIAL1_DIR,
    "qwen_lora_trial1_validation_generations.csv"
)

baseline_metrics_json = os.path.join(
    QWEN_BASELINE_DIR,
    "qwen_base_generation_metrics_summary.json"
)

trial1_metrics_json = os.path.join(
    QWEN_TRIAL1_DIR,
    "qwen_lora_trial1_generation_metrics_summary.json"
)

assert os.path.isfile(baseline_csv), "Base generation CSV is missing."
assert os.path.isfile(trial1_csv), "Trial 1 generation CSV is missing."

base_df = pd.read_csv(baseline_csv).fillna("")
trial1_df = pd.read_csv(trial1_csv).fillna("")

required_columns = [
    "prompt", "reference_response", "generated_response"
]

assert all(c in base_df.columns for c in required_columns)
assert all(c in trial1_df.columns for c in required_columns)
assert len(base_df) == 1000
assert len(trial1_df) == 1000

# Verify the exact prompts and reference responses, including row order.
prompts_match = (
    base_df["prompt"].astype(str).tolist()
    ==
    trial1_df["prompt"].astype(str).tolist()
)

references_match = (
    base_df["reference_response"].astype(str).tolist()
    ==
    trial1_df["reference_response"].astype(str).tolist()
)

print("===== BASE vs QWEN LoRA TRIAL 1 VERIFICATION =====")
print("Base validation rows:", len(base_df))
print("Trial 1 validation rows:", len(trial1_df))
print("Prompts match exactly and in order:", prompts_match)
print("Reference responses match exactly and in order:", references_match)
print(
    "Base and Trial 1 generated responses identical:",
    base_df["generated_response"].astype(str).tolist()
    ==
    trial1_df["generated_response"].astype(str).tolist()
)

assert prompts_match, "Prompt mismatch detected."
assert references_match, "Reference-response mismatch detected."

# Report settings recorded in the Trial 1 metrics summary.
if os.path.isfile(trial1_metrics_json):
    with open(trial1_metrics_json, "r", encoding="utf-8") as f:
        trial1_metrics = json.load(f)

    print("\nTrial 1 recorded generation settings:")
    for key in [
        "generation_method",
        "max_new_tokens",
        "generation_time_minutes"
    ]:
        print(f"{key}: {trial1_metrics.get(key, 'Not recorded')}")

if os.path.isfile(baseline_metrics_json):
    with open(baseline_metrics_json, "r", encoding="utf-8") as f:
        base_metrics = json.load(f)

    print("\nBaseline recorded generation settings:")
    for key in [
        "generation_method",
        "max_new_tokens",
        "generation_time_minutes"
    ]:
        print(f"{key}: {base_metrics.get(key, 'Not recorded')}")

print("\n===== GENERATION LENGTH DIAGNOSTICS =====")
for name, df in [
    ("Base Qwen", base_df),
    ("Qwen LoRA Trial 1", trial1_df)
]:
    lengths = df["generated_response"].astype(str).str.split().str.len()
    print(
        f"{name}: mean={lengths.mean():.2f} words, "
        f"median={lengths.median():.0f}, "
        f"empty={int(lengths.eq(0).sum())}"
    )

print("\nPrompt and reference verification passed.")
print("No files were modified.")

===== BASE vs QWEN LoRA TRIAL 1 VERIFICATION =====
Base validation rows: 1000
Trial 1 validation rows: 1000
Prompts match exactly and in order: True
Reference responses match exactly and in order: True
Base and Trial 1 generated responses identical: False

Trial 1 recorded generation settings:
generation_method: greedy decoding
max_new_tokens: 40
generation_time_minutes: 1.8628627816836039

Baseline recorded generation settings:
generation_method: Not recorded
max_new_tokens: Not recorded
generation_time_minutes: Not recorded

===== GENERATION LENGTH DIAGNOSTICS =====
Base Qwen: mean=29.46 words, median=32, empty=0
Qwen LoRA Trial 1: mean=7.77 words, median=7, empty=0

Prompt and reference verification passed.
No files were modified.


Configuring Qwen + LoRA Trail 2

Learning rate: 5e-5

Epoch: 1

Batch size: 4

Gradient Accumulation: 4

Effective Batch Size: 4X4 = 16

Maximum sequence length: 320

In [ ]:

import os
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from torch.optim import AdamW

QWEN_TRIAL_NAME = "qwen_personachat_conversation_trial2"
QWEN_LEARNING_RATE = 5e-5
QWEN_EPOCHS = 1
QWEN_BATCH_SIZE = 4
QWEN_GRAD_ACCUMULATION_STEPS = 4
QWEN_EFFECTIVE_BATCH_SIZE = 16
QWEN_MAX_GRAD_NORM = 1.0
QWEN_CHECKPOINT_EVERY = 250
QWEN_MAX_LENGTH = 320
QWEN_SEED = 42

QWEN_CONVERSATION_TRIAL2_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "PersonaChat_Qwen/trial_2_conversation_level"
)

os.makedirs(QWEN_CONVERSATION_TRIAL2_DIR, exist_ok=True)

torch.manual_seed(QWEN_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(QWEN_SEED)

qwen_conversation_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_NAME
)

qwen_conversation_model.config.pad_token_id = (
    qwen_tokenizer.pad_token_id
)
qwen_conversation_model.config.use_cache = False

qwen_lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

qwen_conversation_model = get_peft_model(
    qwen_conversation_model,
    qwen_lora_config
)

qwen_conversation_model.to(DEVICE)

qwen_conversation_model.print_trainable_parameters()

qwen_conversation_optimizer = AdamW(
    (
        parameter
        for parameter in qwen_conversation_model.parameters()
        if parameter.requires_grad
    ),
    lr=QWEN_LEARNING_RATE,
    weight_decay=0.01
)

print("\n========== QWEN CONVERSATION-LEVEL TRIAL 2 ==========")
print("Model:", QWEN_MODEL_NAME)
print("Training examples:", len(qwen_train_conversation_dataset))
print("Validation examples:", len(qwen_validation_conversation_dataset))
print("Learning rate:", QWEN_LEARNING_RATE)
print("Epochs:", QWEN_EPOCHS)
print("Micro-batch size:", QWEN_BATCH_SIZE)
print("Gradient accumulation steps:", QWEN_GRAD_ACCUMULATION_STEPS)
print("Effective batch size:", QWEN_EFFECTIVE_BATCH_SIZE)
print("Weight decay:", 0.01)
print("Maximum gradient norm:", QWEN_MAX_GRAD_NORM)
print("Maximum sequence length:", QWEN_MAX_LENGTH)
print("Checkpoint interval:", QWEN_CHECKPOINT_EVERY)
print("Output directory:", QWEN_CONVERSATION_TRIAL2_DIR)
print("=====================================================")


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184

========== QWEN CONVERSATION-LEVEL TRIAL 2 ==========
Model: Qwen/Qwen2.5-0.5B-Instruct
Training examples: 17878
Validation examples: 1000
Learning rate: 5e-05
Epochs: 1
Micro-batch size: 4
Gradient accumulation steps: 4
Effective batch size: 16
Weight decay: 0.01
Maximum gradient norm: 1.0
Maximum sequence length: 320
Checkpoint interval: 250
Output directory: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_Qwen/trial_2_conversation_level


Training Qwen LoRA Trial 2

In [ ]:

import os
import time
import math
import torch
from torch.utils.data import DataLoader
from torch.nn.utils import clip_grad_norm_


# Deterministic DataLoader: recreates the same order on resume

def make_epoch_loader(epoch_index):
    generator = torch.Generator()
    generator.manual_seed(QWEN_SEED + epoch_index)

    return DataLoader(
        qwen_train_conversation_dataset,
        batch_size=QWEN_BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        collate_fn=qwen_collator,
    )


# Checkpoint paths and mixed precision

QWEN_CHECKPOINT_PATH = os.path.join(
    QWEN_CONVERSATION_TRIAL2_DIR,
    "training_checkpoint.pt",
)
QWEN_FINAL_ADAPTER_DIR = os.path.join(
    QWEN_CONVERSATION_TRIAL2_DIR,
    "final_adapter",
)

use_amp = DEVICE.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

global_step = 0
start_epoch = 0
start_batch = 0
loss_history = []
elapsed_before = 0.0


# Resume from Trial 2 checkpoint, if present

if os.path.isfile(QWEN_CHECKPOINT_PATH):
    checkpoint = torch.load(
        QWEN_CHECKPOINT_PATH,
        map_location=DEVICE,
        weights_only=False,
    )

    # Prevent accidentally resuming a checkpoint from another trial.
    assert checkpoint.get("trial_name") == QWEN_TRIAL_NAME, (
        "Checkpoint trial name mismatch. "
        "Check the Trial 2 checkpoint path before continuing."
    )

    qwen_conversation_model.load_state_dict(checkpoint["model_state"])
    qwen_conversation_optimizer.load_state_dict(
        checkpoint["optimizer_state"]
    )

    if "scaler_state" in checkpoint:
        scaler.load_state_dict(checkpoint["scaler_state"])

    global_step = checkpoint["global_step"]
    start_epoch = checkpoint["next_epoch"]
    start_batch = checkpoint["next_batch"]
    loss_history = checkpoint.get("loss_history", [])
    elapsed_before = checkpoint.get("elapsed_seconds", 0.0)

    torch.set_rng_state(checkpoint["torch_rng_state"].cpu())
    if (
        torch.cuda.is_available()
        and checkpoint.get("cuda_rng_state") is not None
    ):
        torch.cuda.set_rng_state_all(checkpoint["cuda_rng_state"])

    print(
        f"Resuming Trial 2 at epoch {start_epoch + 1}, "
        f"batch {start_batch + 1}, global step {global_step}."
    )
else:
    print("No existing Trial 2 training checkpoint. Starting a fresh run.")


# Atomic checkpoint saving

def save_checkpoint(next_epoch, next_batch, elapsed_seconds):
    checkpoint_data = {
        "model_state": qwen_conversation_model.state_dict(),
        "optimizer_state": qwen_conversation_optimizer.state_dict(),
        "scaler_state": scaler.state_dict(),
        "global_step": global_step,
        "next_epoch": next_epoch,
        "next_batch": next_batch,
        "loss_history": loss_history,
        "elapsed_seconds": elapsed_seconds,
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else None
        ),
        "trial_name": QWEN_TRIAL_NAME,
        "learning_rate": QWEN_LEARNING_RATE,
        "batch_size": QWEN_BATCH_SIZE,
        "gradient_accumulation_steps": QWEN_GRAD_ACCUMULATION_STEPS,
        "seed": QWEN_SEED,
    }

    temporary_path = QWEN_CHECKPOINT_PATH + ".tmp"
    torch.save(checkpoint_data, temporary_path)
    os.replace(temporary_path, QWEN_CHECKPOINT_PATH)


# Training

qwen_conversation_model.train()
run_start = time.time()

for epoch_index in range(start_epoch, QWEN_EPOCHS):
    train_loader = make_epoch_loader(epoch_index)
    batch_offset = start_batch if epoch_index == start_epoch else 0

    qwen_conversation_optimizer.zero_grad(set_to_none=True)
    accumulated_micro_batches = 0

    for batch_index, batch in enumerate(train_loader):
        if batch_index < batch_offset:
            continue

        batch = {
            key: value.to(DEVICE, non_blocking=True)
            for key, value in batch.items()
        }

        # Correctly scale the final, possibly incomplete accumulation group.
        group_start = (
            batch_index // QWEN_GRAD_ACCUMULATION_STEPS
        ) * QWEN_GRAD_ACCUMULATION_STEPS
        group_size = min(
            QWEN_GRAD_ACCUMULATION_STEPS,
            len(train_loader) - group_start,
        )

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=use_amp,
        ):
            outputs = qwen_conversation_model(**batch)
            loss = outputs.loss

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss at epoch {epoch_index + 1}, "
                f"batch {batch_index + 1}: {loss.item()}"
            )

        scaler.scale(loss / group_size).backward()
        accumulated_micro_batches += 1
        loss_history.append(float(loss.detach().cpu()))

        is_accumulation_boundary = (
            accumulated_micro_batches >= group_size
        )
        is_last_batch = batch_index + 1 >= len(train_loader)

        if is_accumulation_boundary or is_last_batch:
            scaler.unscale_(qwen_conversation_optimizer)
            clip_grad_norm_(
                qwen_conversation_model.parameters(),
                QWEN_MAX_GRAD_NORM,
            )
            scaler.step(qwen_conversation_optimizer)
            scaler.update()
            qwen_conversation_optimizer.zero_grad(set_to_none=True)

            global_step += 1
            accumulated_micro_batches = 0

            if global_step % 25 == 0:
                recent_losses = loss_history[-25:]
                recent_loss = sum(recent_losses) / len(recent_losses)
                elapsed = elapsed_before + time.time() - run_start

                print(
                    f"Epoch {epoch_index + 1}/{QWEN_EPOCHS} | "
                    f"Step {global_step}/{QWEN_TOTAL_STEPS} | "
                    f"Micro-batch loss (last 25 avg): "
                    f"{recent_loss:.4f} | "
                    f"Elapsed: {elapsed / 60:.1f} min"
                )

            next_epoch = epoch_index
            next_batch = batch_index + 1

            if next_batch >= len(train_loader):
                next_epoch = epoch_index + 1
                next_batch = 0

            if (
                global_step % QWEN_CHECKPOINT_EVERY == 0
                or next_epoch >= QWEN_EPOCHS
            ):
                elapsed = elapsed_before + time.time() - run_start
                save_checkpoint(next_epoch, next_batch, elapsed)

                print(
                    f"Checkpoint saved at step {global_step}; "
                    f"next position: epoch {next_epoch + 1}, "
                    f"batch {next_batch + 1}."
                )

    start_batch = 0

elapsed_total = elapsed_before + time.time() - run_start


# Save final LoRA adapter and tokenizer

os.makedirs(QWEN_FINAL_ADAPTER_DIR, exist_ok=True)
qwen_conversation_model.save_pretrained(QWEN_FINAL_ADAPTER_DIR)
qwen_tokenizer.save_pretrained(QWEN_FINAL_ADAPTER_DIR)


# Validation loss and perplexity

qwen_conversation_model.eval()
validation_loss_sum = 0.0
validation_batches = 0

validation_loader = DataLoader(
    qwen_validation_conversation_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=qwen_collator,
)

with torch.inference_mode():
    for batch in validation_loader:
        batch = {
            key: value.to(DEVICE, non_blocking=True)
            for key, value in batch.items()
        }

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=use_amp,
        ):
            outputs = qwen_conversation_model(**batch)

        if torch.isfinite(outputs.loss):
            validation_loss_sum += outputs.loss.item()
            validation_batches += 1
        else:
            raise RuntimeError("Non-finite validation loss detected.")

validation_loss = validation_loss_sum / validation_batches
validation_perplexity = math.exp(validation_loss)

print("\n========== QWEN CONVERSATION-LEVEL TRIAL 2 ==========")
print("Training examples:", len(qwen_train_conversation_dataset))
print("Validation examples:", len(qwen_validation_conversation_dataset))
print("Optimizer steps completed:", global_step)
print("Mean training loss:", round(sum(loss_history) / len(loss_history), 4))
print("Validation loss:", round(validation_loss, 4))
print("Validation perplexity:", round(validation_perplexity, 4))
print("Elapsed training time (minutes):", round(elapsed_total / 60, 2))
print("Final adapter:", QWEN_FINAL_ADAPTER_DIR)
print("Training checkpoint:", QWEN_CHECKPOINT_PATH)
print("=====================================================")

qwen_conversation_model.train()


No existing Trial 2 training checkpoint. Starting a fresh run.
Epoch 1/1 | Step 25/1118 | Micro-batch loss (last 25 avg): 3.2665 | Elapsed: 0.5 min
Epoch 1/1 | Step 50/1118 | Micro-batch loss (last 25 avg): 2.9324 | Elapsed: 0.9 min
Epoch 1/1 | Step 75/1118 | Micro-batch loss (last 25 avg): 2.9138 | Elapsed: 1.4 min
Epoch 1/1 | Step 100/1118 | Micro-batch loss (last 25 avg): 2.9906 | Elapsed: 1.8 min
Epoch 1/1 | Step 125/1118 | Micro-batch loss (last 25 avg): 2.8250 | Elapsed: 2.3 min
Epoch 1/1 | Step 150/1118 | Micro-batch loss (last 25 avg): 3.0318 | Elapsed: 2.7 min
Epoch 1/1 | Step 175/1118 | Micro-batch loss (last 25 avg): 2.9107 | Elapsed: 3.2 min
Epoch 1/1 | Step 200/1118 | Micro-batch loss (last 25 avg): 2.9957 | Elapsed: 3.6 min
Epoch 1/1 | Step 225/1118 | Micro-batch loss (last 25 avg): 3.0150 | Elapsed: 4.1 min
Epoch 1/1 | Step 250/1118 | Micro-batch loss (last 25 avg): 2.7674 | Elapsed: 4.5 min
Checkpoint saved at step 250; next position: epoch 1, batch 1001.
Epoch 1/1 | St

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 896)
        (layers): ModuleList(
          (0-23): 24 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=896, out_features=896, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=896, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=896, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
      

Qwen + LoRA Trial 2 generation and evaluation

In [ ]:

import os
import json
import time
import gc
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bert_score


# Paths

QWEN_TRIAL2_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "PersonaChat_Qwen/trial_2_conversation_level"
)

QWEN_ADAPTER_DIR = os.path.join(
    QWEN_TRIAL2_DIR, "final_adapter"
)

QWEN_BASELINE_CSV = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "PersonaChat_Qwen/base_model_evaluation/"
    "qwen_base_validation_generations.csv"
)

QWEN_TRIAL2_GENERATIONS_CSV = os.path.join(
    QWEN_TRIAL2_DIR,
    "qwen_lora_trial2_validation_generations.csv"
)

QWEN_TRIAL2_METRICS_CSV = os.path.join(
    QWEN_TRIAL2_DIR,
    "qwen_lora_trial2_generation_metrics_per_example.csv"
)

QWEN_TRIAL2_SUMMARY_JSON = os.path.join(
    QWEN_TRIAL2_DIR,
    "qwen_lora_trial2_generation_metrics_summary.json"
)

assert os.path.isdir(QWEN_ADAPTER_DIR), (
    f"Trained adapter not found: {QWEN_ADAPTER_DIR}"
)
assert os.path.isfile(QWEN_BASELINE_CSV), (
    f"Baseline generations not found: {QWEN_BASELINE_CSV}"
)

# Reuse exactly the baseline prompts and references.

baseline_df = pd.read_csv(QWEN_BASELINE_CSV).fillna("")

required_columns = [
    "prompt", "reference_response", "generated_response"
]

assert all(column in baseline_df.columns for column in required_columns)
assert len(baseline_df) == 1000, (
    f"Expected 1000 validation examples, found {len(baseline_df)}"
)

prompts = baseline_df["prompt"].astype(str).tolist()
references = baseline_df["reference_response"].astype(str).tolist()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Evaluation device:", device)


# Generate Trial 2 responses, or reuse existing generations.

generation_time_minutes = 0.0

if os.path.isfile(QWEN_TRIAL2_GENERATIONS_CSV):
    print("Existing Trial 2 generations found. Reusing them.")

    trial2_df = pd.read_csv(
        QWEN_TRIAL2_GENERATIONS_CSV
    ).fillna("")

    assert len(trial2_df) == 1000
    assert all(
        column in trial2_df.columns
        for column in required_columns
    )

    assert trial2_df["prompt"].astype(str).tolist() == prompts
    assert (
        trial2_df["reference_response"].astype(str).tolist()
        == references
    )

    predictions = (
        trial2_df["generated_response"].astype(str).tolist()
    )

else:
    print("Loading trained Qwen LoRA Trial 2 adapter...")

    tokenizer = AutoTokenizer.from_pretrained(
        QWEN_ADAPTER_DIR
    )
    tokenizer.padding_side = "left"
    tokenizer.truncation_side = "left"

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        QWEN_MODEL_NAME
    )

    model = PeftModel.from_pretrained(
        base_model,
        QWEN_ADAPTER_DIR
    )

    model.config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = True
    model.to(device)
    model.eval()

    BATCH_SIZE = 8
    MAX_NEW_TOKENS = 40
    QWEN_EOS_TOKEN_ID = tokenizer.convert_tokens_to_ids(
        "<|im_end|>"
    )

    generation_start = time.time()
    predictions = []

    for start in range(0, len(prompts), BATCH_SIZE):
        batch_prompts = prompts[start:start + BATCH_SIZE]
        formatted_prompts = []

        for raw_prompt in batch_prompts:
            if "\n\nConversation:\n" not in raw_prompt:
                raise ValueError(
                    "Unexpected prompt format. Expected "
                    "'\\n\\nConversation:\\n' in each prompt."
                )

            persona_text, history = raw_prompt.split(
                "\n\nConversation:\n",
                1
            )

            messages = [
                {
                    "role": "system",
                    "content": (
                        "You are a helpful conversational partner. "
                        "Stay consistent with this persona:\n"
                        f"{persona_text}"
                    )
                },
                {
                    "role": "user",
                    "content": (
                        "Continue this conversation naturally. "
                        "Conversation history:\n"
                        f"{history}"
                    )
                }
            ]

            formatted_prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            formatted_prompts.append(formatted_prompt)

        inputs = tokenizer(
            formatted_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=280,
            add_special_tokens=False
        ).to(device)

        with torch.inference_mode():
            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                output_ids = model.generate(
                    **inputs,
                    do_sample=False,
                    max_new_tokens=MAX_NEW_TOKENS,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=QWEN_EOS_TOKEN_ID,
                    use_cache=True
                )

        # Exclude the padded input prompt from decoded responses.
        input_width = inputs["input_ids"].shape[1]
        generated_ids = output_ids[:, input_width:]

        batch_predictions = tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        predictions.extend(
            response.strip() for response in batch_predictions
        )

        completed = min(start + len(batch_prompts), len(prompts))

        if completed % 200 == 0 or completed == len(prompts):
            print(
                f"Generated {completed}/{len(prompts)} responses"
            )

    generation_time_minutes = (
        time.time() - generation_start
    ) / 60

    trial2_df = pd.DataFrame({
        "prompt": prompts,
        "reference_response": references,
        "generated_response": predictions
    })

    assert len(trial2_df) == 1000

    # Save separately; never modify baseline or Trial 1 files.
    trial2_df.to_csv(
        QWEN_TRIAL2_GENERATIONS_CSV,
        index=False
    )

    print("Generated responses saved:", QWEN_TRIAL2_GENERATIONS_CSV)

    del model, base_model, tokenizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# BLEU — corpus-level word overlap.

references_tokens = [
    text.lower().split() for text in references
]
predictions_tokens = [
    text.lower().split() for text in predictions
]

bleu = corpus_bleu(
    [[reference] for reference in references_tokens],
    predictions_tokens,
    smoothing_function=SmoothingFunction().method1
)


# ROUGE-L — per-example F1.

rouge = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True
)

rouge_l_scores = [
    rouge.score(reference, prediction)["rougeL"].fmeasure
    for reference, prediction in zip(references, predictions)
]

mean_rouge_l = sum(rouge_l_scores) / len(rouge_l_scores)


# Distinct-1 and Distinct-2.

all_unigrams = []
all_bigrams = []

for prediction in predictions_tokens:
    all_unigrams.extend(prediction)
    all_bigrams.extend(
        tuple(prediction[i:i + 2])
        for i in range(max(0, len(prediction) - 1))
    )

distinct_1 = (
    len(set(all_unigrams)) / len(all_unigrams)
    if all_unigrams else 0.0
)

distinct_2 = (
    len(set(all_bigrams)) / len(all_bigrams)
    if all_bigrams else 0.0
)


# Response lengths.

reply_lengths = [
    len(text.split()) for text in predictions
]

average_reply_length = sum(reply_lengths) / len(reply_lengths)
median_reply_length = float(
    pd.Series(reply_lengths).median()
)


# BERTScore — use the same encoder and settings as the baseline.

bert_start = time.time()

P, R, F1 = bert_score(
    cands=predictions,
    refs=references,
    model_type="distilbert-base-uncased",
    lang="en",
    batch_size=16,
    device=str(device),
    verbose=True,
    rescale_with_baseline=False
)

mean_bert_precision = P.mean().item()
mean_bert_recall = R.mean().item()
mean_bert_f1 = F1.mean().item()

bert_time_minutes = (time.time() - bert_start) / 60


# Save per-example metrics.

trial2_df["rougeL_f1"] = rouge_l_scores
trial2_df["bert_score_precision"] = P.tolist()
trial2_df["bert_score_recall"] = R.tolist()
trial2_df["bert_score_f1"] = F1.tolist()
trial2_df["generated_word_count"] = reply_lengths

trial2_df.to_csv(
    QWEN_TRIAL2_METRICS_CSV,
    index=False
)


# Save aggregate metrics.

metrics = {
    "model": QWEN_MODEL_NAME,
    "stage": "LoRA Trial 2",
    "number_of_validation_conversations": len(trial2_df),
    "training_examples": 17878,
    "validation_loss": 2.7346,
    "validation_perplexity": 15.4034,
    "corpus_bleu": bleu,
    "mean_rougeL_f1": mean_rouge_l,
    "mean_bert_score_precision": mean_bert_precision,
    "mean_bert_score_recall": mean_bert_recall,
    "mean_bert_score_f1": mean_bert_f1,
    "distinct_1": distinct_1,
    "distinct_2": distinct_2,
    "average_reply_length_words": average_reply_length,
    "median_reply_length_words": median_reply_length,
    "empty_reply_count": sum(
        not text.strip() for text in predictions
    ),
    "generation_time_minutes": generation_time_minutes,
    "bert_score_time_minutes": bert_time_minutes,
    "generation_method": "greedy decoding",
    "max_new_tokens": 40
}

with open(
    QWEN_TRIAL2_SUMMARY_JSON,
    "w",
    encoding="utf-8"
) as f:
    json.dump(metrics, f, indent=2)


# Display results.

print("\n===== QWEN LoRA TRIAL 2 GENERATION METRICS =====")

for name, value in metrics.items():
    if isinstance(value, float):
        print(f"{name}: {value:.4f}")
    else:
        print(f"{name}: {value}")

print("\nGeneration CSV:", QWEN_TRIAL2_GENERATIONS_CSV)
print("Per-example metrics CSV:", QWEN_TRIAL2_METRICS_CSV)
print("Summary JSON:", QWEN_TRIAL2_SUMMARY_JSON)
print("================================================")


Evaluation device: cuda
Loading trained Qwen LoRA Trial 2 adapter...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Generated 200/1000 responses
Generated 400/1000 responses
Generated 600/1000 responses
Generated 800/1000 responses
Generated 1000/1000 responses
Generated responses saved: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_Qwen/trial_2_conversation_level/qwen_lora_trial2_validation_generations.csv


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/108 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/63 [00:00<?, ?it/s]

done in 1.44 seconds, 696.09 sentences/sec

===== QWEN LoRA TRIAL 2 GENERATION METRICS =====
model: Qwen/Qwen2.5-0.5B-Instruct
stage: LoRA Trial 2
number_of_validation_conversations: 1000
training_examples: 17878
validation_loss: 2.7346
validation_perplexity: 15.4034
corpus_bleu: 0.0129
mean_rougeL_f1: 0.1077
mean_bert_score_precision: 0.7040
mean_bert_score_recall: 0.6918
mean_bert_score_f1: 0.6975
distinct_1: 0.1076
distinct_2: 0.3901
average_reply_length_words: 7.4450
median_reply_length_words: 7.0000
empty_reply_count: 0
generation_time_minutes: 1.8317
bert_score_time_minutes: 0.0577
generation_method: greedy decoding
max_new_tokens: 40

Generation CSV: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_Qwen/trial_2_conversation_level/qwen_lora_trial2_validation_generations.csv
Per-example metrics CSV: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_Qwen/trial_2_conversation_level/qwen_lora_trial2_generation_metrics_per_example.csv
Summary JSON: /content/drive/MyDrive/

Verifying Qwen Baseline vs LoRA Trial 2 and Comparing Generation Lengths

In [ ]:

import os
import json
import pandas as pd

QWEN_BASELINE_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "PersonaChat_Qwen/base_model_evaluation"
)

QWEN_TRIAL2_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "PersonaChat_Qwen/trial_2_conversation_level"
)

baseline_csv = os.path.join(
    QWEN_BASELINE_DIR,
    "qwen_base_validation_generations.csv"
)

trial2_csv = os.path.join(
    QWEN_TRIAL2_DIR,
    "qwen_lora_trial2_validation_generations.csv"
)

baseline_metrics_json = os.path.join(
    QWEN_BASELINE_DIR,
    "qwen_base_generation_metrics_summary.json"
)

trial2_metrics_json = os.path.join(
    QWEN_TRIAL2_DIR,
    "qwen_lora_trial2_generation_metrics_summary.json"
)

assert os.path.isfile(baseline_csv), "Base generation CSV is missing."
assert os.path.isfile(trial2_csv), "Trial 2 generation CSV is missing."

base_df = pd.read_csv(baseline_csv).fillna("")
trial2_df = pd.read_csv(trial2_csv).fillna("")

required_columns = [
    "prompt", "reference_response", "generated_response"
]

assert all(c in base_df.columns for c in required_columns)
assert all(c in trial2_df.columns for c in required_columns)
assert len(base_df) == 1000
assert len(trial2_df) == 1000

# Verify the exact prompts and reference responses, including row order.
prompts_match = (
    base_df["prompt"].astype(str).tolist()
    ==
    trial2_df["prompt"].astype(str).tolist()
)

references_match = (
    base_df["reference_response"].astype(str).tolist()
    ==
    trial2_df["reference_response"].astype(str).tolist()
)

print("===== BASE vs QWEN LoRA TRIAL 2 VERIFICATION =====")
print("Base validation rows:", len(base_df))
print("Trial 2 validation rows:", len(trial2_df))
print("Prompts match exactly and in order:", prompts_match)
print("Reference responses match exactly and in order:", references_match)
print(
    "Base and Trial 2 generated responses identical:",
    base_df["generated_response"].astype(str).tolist()
    ==
    trial2_df["generated_response"].astype(str).tolist()
)

assert prompts_match, "Prompt mismatch detected."
assert references_match, "Reference-response mismatch detected."

# Report settings recorded in the Trial 2 metrics summary.
if os.path.isfile(trial2_metrics_json):
    with open(trial2_metrics_json, "r", encoding="utf-8") as f:
        trial2_metrics = json.load(f)

    print("\nTrial 2 recorded generation settings:")
    for key in [
        "generation_method",
        "max_new_tokens",
        "generation_time_minutes"
    ]:
        print(f"{key}: {trial2_metrics.get(key, 'Not recorded')}")

if os.path.isfile(baseline_metrics_json):
    with open(baseline_metrics_json, "r", encoding="utf-8") as f:
        base_metrics = json.load(f)

    print("\nBaseline recorded generation settings:")
    for key in [
        "generation_method",
        "max_new_tokens",
        "generation_time_minutes"
    ]:
        print(f"{key}: {base_metrics.get(key, 'Not recorded')}")

print("\n===== GENERATION LENGTH DIAGNOSTICS =====")
for name, df in [
    ("Base Qwen", base_df),
    ("Qwen LoRA Trial 2", trial2_df)
]:
    lengths = df["generated_response"].astype(str).str.split().str.len()
    print(
        f"{name}: mean={lengths.mean():.2f} words, "
        f"median={lengths.median():.0f}, "
        f"empty={int(lengths.eq(0).sum())}"
    )

print("\nPrompt and reference verification passed.")
print("No files were modified.")


===== BASE vs QWEN LoRA TRIAL 2 VERIFICATION =====
Base validation rows: 1000
Trial 2 validation rows: 1000
Prompts match exactly and in order: True
Reference responses match exactly and in order: True
Base and Trial 2 generated responses identical: False

Trial 2 recorded generation settings:
generation_method: greedy decoding
max_new_tokens: 40
generation_time_minutes: 1.8316718618075052

Baseline recorded generation settings:
generation_method: Not recorded
max_new_tokens: Not recorded
generation_time_minutes: Not recorded

===== GENERATION LENGTH DIAGNOSTICS =====
Base Qwen: mean=29.46 words, median=32, empty=0
Qwen LoRA Trial 2: mean=7.45 words, median=7, empty=0

Prompt and reference verification passed.
No files were modified.


Configuring Qwen + LoRA Trail 3

Learning rate: 1e-4

Epoch: 2

Batch size: 4

Gradient Accumulation: 4

Effective Batch Size: 4X4 = 16

Maximum sequence length: 320

In [ ]:

import os
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from torch.optim import AdamW

QWEN_TRIAL_NAME = "qwen_personachat_conversation_trial3"
QWEN_LEARNING_RATE = 1e-4
QWEN_EPOCHS = 2
QWEN_BATCH_SIZE = 4
QWEN_GRAD_ACCUMULATION_STEPS = 4
QWEN_EFFECTIVE_BATCH_SIZE = 16
QWEN_MAX_GRAD_NORM = 1.0
QWEN_CHECKPOINT_EVERY = 250
QWEN_MAX_LENGTH = 320
QWEN_SEED = 42

QWEN_CONVERSATION_TRIAL3_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "PersonaChat_Qwen/trial_3_conversation_level"
)

os.makedirs(QWEN_CONVERSATION_TRIAL3_DIR, exist_ok=True)

torch.manual_seed(QWEN_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(QWEN_SEED)

qwen_conversation_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_NAME
)

qwen_conversation_model.config.pad_token_id = (
    qwen_tokenizer.pad_token_id
)
qwen_conversation_model.config.use_cache = False

qwen_lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

qwen_conversation_model = get_peft_model(
    qwen_conversation_model,
    qwen_lora_config
)

qwen_conversation_model.to(DEVICE)

qwen_conversation_model.print_trainable_parameters()

qwen_conversation_optimizer = AdamW(
    (
        parameter
        for parameter in qwen_conversation_model.parameters()
        if parameter.requires_grad
    ),
    lr=QWEN_LEARNING_RATE,
    weight_decay=0.01
)

print("\n========== QWEN CONVERSATION-LEVEL TRIAL 3 ==========")
print("Model:", QWEN_MODEL_NAME)
print("Training examples:", len(qwen_train_conversation_dataset))
print("Validation examples:", len(qwen_validation_conversation_dataset))
print("Learning rate:", QWEN_LEARNING_RATE)
print("Epochs:", QWEN_EPOCHS)
print("Micro-batch size:", QWEN_BATCH_SIZE)
print("Gradient accumulation steps:", QWEN_GRAD_ACCUMULATION_STEPS)
print("Effective batch size:", QWEN_EFFECTIVE_BATCH_SIZE)
print("Weight decay:", 0.01)
print("Maximum gradient norm:", QWEN_MAX_GRAD_NORM)
print("Maximum sequence length:", QWEN_MAX_LENGTH)
print("Checkpoint interval:", QWEN_CHECKPOINT_EVERY)
print("Output directory:", QWEN_CONVERSATION_TRIAL3_DIR)
print("=====================================================")


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184

========== QWEN CONVERSATION-LEVEL TRIAL 3 ==========
Model: Qwen/Qwen2.5-0.5B-Instruct
Training examples: 17878
Validation examples: 1000
Learning rate: 0.0001
Epochs: 2
Micro-batch size: 4
Gradient accumulation steps: 4
Effective batch size: 16
Weight decay: 0.01
Maximum gradient norm: 1.0
Maximum sequence length: 320
Checkpoint interval: 250
Output directory: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_Qwen/trial_3_conversation_level


Training Qwen + LoRA Trial 3

In [ ]:

import os
import time
import math
import torch
from torch.utils.data import DataLoader
from torch.nn.utils import clip_grad_norm_


# Deterministic DataLoader: recreates the same order on resume

def make_epoch_loader(epoch_index):
    generator = torch.Generator()
    generator.manual_seed(QWEN_SEED + epoch_index)

    return DataLoader(
        qwen_train_conversation_dataset,
        batch_size=QWEN_BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        collate_fn=qwen_collator,
    )


# Checkpoint paths and mixed precision

QWEN_CHECKPOINT_PATH = os.path.join(
    QWEN_CONVERSATION_TRIAL3_DIR,
    "training_checkpoint.pt",
)
QWEN_FINAL_ADAPTER_DIR = os.path.join(
    QWEN_CONVERSATION_TRIAL3_DIR,
    "final_adapter",
)

use_amp = DEVICE.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

global_step = 0
start_epoch = 0
start_batch = 0
loss_history = []
elapsed_before = 0.0


# Resume from Trial 3 checkpoint, if present

if os.path.isfile(QWEN_CHECKPOINT_PATH):
    checkpoint = torch.load(
        QWEN_CHECKPOINT_PATH,
        map_location=DEVICE,
        weights_only=False,
    )

    # Prevent accidentally resuming a checkpoint from another trial.
    assert checkpoint.get("trial_name") == QWEN_TRIAL_NAME, (
        "Checkpoint trial name mismatch. "
        "Check the Trial 3 checkpoint path before continuing."
    )

    qwen_conversation_model.load_state_dict(checkpoint["model_state"])
    qwen_conversation_optimizer.load_state_dict(
        checkpoint["optimizer_state"]
    )

    if "scaler_state" in checkpoint:
        scaler.load_state_dict(checkpoint["scaler_state"])

    global_step = checkpoint["global_step"]
    start_epoch = checkpoint["next_epoch"]
    start_batch = checkpoint["next_batch"]
    loss_history = checkpoint.get("loss_history", [])
    elapsed_before = checkpoint.get("elapsed_seconds", 0.0)

    torch.set_rng_state(checkpoint["torch_rng_state"].cpu())
    if (
        torch.cuda.is_available()
        and checkpoint.get("cuda_rng_state") is not None
    ):
        torch.cuda.set_rng_state_all([
            state.cpu().to(torch.uint8)
            for state in checkpoint["cuda_rng_state"]
        ])


    print(
        f"Resuming Trial 3 at epoch {start_epoch + 1}, "
        f"batch {start_batch + 1}, global step {global_step}."
    )
else:
    print("No existing Trial 3 training checkpoint. Starting a fresh run.")


# Atomic checkpoint saving

def save_checkpoint(next_epoch, next_batch, elapsed_seconds):
    checkpoint_data = {
        "model_state": qwen_conversation_model.state_dict(),
        "optimizer_state": qwen_conversation_optimizer.state_dict(),
        "scaler_state": scaler.state_dict(),
        "global_step": global_step,
        "next_epoch": next_epoch,
        "next_batch": next_batch,
        "loss_history": loss_history,
        "elapsed_seconds": elapsed_seconds,
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else None
        ),
        "trial_name": QWEN_TRIAL_NAME,
        "learning_rate": QWEN_LEARNING_RATE,
        "batch_size": QWEN_BATCH_SIZE,
        "gradient_accumulation_steps": QWEN_GRAD_ACCUMULATION_STEPS,
        "seed": QWEN_SEED,
    }

    temporary_path = QWEN_CHECKPOINT_PATH + ".tmp"
    torch.save(checkpoint_data, temporary_path)
    os.replace(temporary_path, QWEN_CHECKPOINT_PATH)


# Training

qwen_conversation_model.train()
run_start = time.time()

for epoch_index in range(start_epoch, QWEN_EPOCHS):
    train_loader = make_epoch_loader(epoch_index)
    batch_offset = start_batch if epoch_index == start_epoch else 0

    qwen_conversation_optimizer.zero_grad(set_to_none=True)
    accumulated_micro_batches = 0

    for batch_index, batch in enumerate(train_loader):
        if batch_index < batch_offset:
            continue

        batch = {
            key: value.to(DEVICE, non_blocking=True)
            for key, value in batch.items()
        }

        # Correctly scale the final, possibly incomplete accumulation group.
        group_start = (
            batch_index // QWEN_GRAD_ACCUMULATION_STEPS
        ) * QWEN_GRAD_ACCUMULATION_STEPS
        group_size = min(
            QWEN_GRAD_ACCUMULATION_STEPS,
            len(train_loader) - group_start,
        )

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=use_amp,
        ):
            outputs = qwen_conversation_model(**batch)
            loss = outputs.loss

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss at epoch {epoch_index + 1}, "
                f"batch {batch_index + 1}: {loss.item()}"
            )

        scaler.scale(loss / group_size).backward()
        accumulated_micro_batches += 1
        loss_history.append(float(loss.detach().cpu()))

        is_accumulation_boundary = (
            accumulated_micro_batches >= group_size
        )
        is_last_batch = batch_index + 1 >= len(train_loader)

        if is_accumulation_boundary or is_last_batch:
            scaler.unscale_(qwen_conversation_optimizer)
            clip_grad_norm_(
                qwen_conversation_model.parameters(),
                QWEN_MAX_GRAD_NORM,
            )
            scaler.step(qwen_conversation_optimizer)
            scaler.update()
            qwen_conversation_optimizer.zero_grad(set_to_none=True)

            global_step += 1
            accumulated_micro_batches = 0

            if global_step % 25 == 0:
                recent_losses = loss_history[-25:]
                recent_loss = sum(recent_losses) / len(recent_losses)
                elapsed = elapsed_before + time.time() - run_start

                print(
                    f"Epoch {epoch_index + 1}/{QWEN_EPOCHS} | "
                    f"Step {global_step}/{QWEN_TOTAL_STEPS} | "
                    f"Micro-batch loss (last 25 avg): "
                    f"{recent_loss:.4f} | "
                    f"Elapsed: {elapsed / 60:.1f} min"
                )

            next_epoch = epoch_index
            next_batch = batch_index + 1

            if next_batch >= len(train_loader):
                next_epoch = epoch_index + 1
                next_batch = 0

            if (
                global_step % QWEN_CHECKPOINT_EVERY == 0
                or next_epoch >= QWEN_EPOCHS
            ):
                elapsed = elapsed_before + time.time() - run_start
                save_checkpoint(next_epoch, next_batch, elapsed)

                print(
                    f"Checkpoint saved at step {global_step}; "
                    f"next position: epoch {next_epoch + 1}, "
                    f"batch {next_batch + 1}."
                )

    start_batch = 0

elapsed_total = elapsed_before + time.time() - run_start


# Save final LoRA adapter and tokenizer

os.makedirs(QWEN_FINAL_ADAPTER_DIR, exist_ok=True)
qwen_conversation_model.save_pretrained(QWEN_FINAL_ADAPTER_DIR)
qwen_tokenizer.save_pretrained(QWEN_FINAL_ADAPTER_DIR)



# Training summary — validation is handled in the next cell due to a memory error

print("\n========== QWEN CONVERSATION-LEVEL TRIAL 3 ==========")
print("Training examples:", len(qwen_train_conversation_dataset))
print("Configured epochs:", QWEN_EPOCHS)
print("Optimizer steps completed:", global_step)
print(
    "Mean training loss:",
    round(sum(loss_history) / len(loss_history), 4)
    if loss_history else "No new training batches"
)
print("Elapsed training time (minutes):", round(elapsed_total / 60, 2))
print("Final adapter:", QWEN_FINAL_ADAPTER_DIR)
print("Training checkpoint:", QWEN_CHECKPOINT_PATH)
print("Training and adapter saving completed.")
print("Validation will run in the separate validation cell.")
print("=====================================================")



Resuming Trial 3 at epoch 3, batch 1, global step 2236.

========== QWEN CONVERSATION-LEVEL TRIAL 3 ==========
Training examples: 17878
Configured epochs: 2
Optimizer steps completed: 2236
Mean training loss: 2.8127
Elapsed training time (minutes): 41.13
Final adapter: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_Qwen/trial_3_conversation_level/final_adapter
Training checkpoint: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_Qwen/trial_3_conversation_level/training_checkpoint.pt
Training and adapter saving completed.
Validation will run in the separate validation cell.


Qwen + LoRA training validation and perplexity loss

In [ ]:

import gc
import math
import torch
from torch.utils.data import DataLoader

# Release temporary tensors from the previous failed validation
for name in ["outputs", "batch", "validation_outputs"]:
    if name in globals():
        del globals()[name]

# Training is complete, so the optimizer is no longer needed.
if "qwen_conversation_optimizer" in globals():
    del qwen_conversation_optimizer

if "scaler" in globals():
    del scaler

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

assert os.path.isdir(QWEN_FINAL_ADAPTER_DIR), (
    f"Trial 3 adapter not found: {QWEN_FINAL_ADAPTER_DIR}"
)

qwen_conversation_model.eval()

validation_loss_sum = 0.0
validation_batches = 0

validation_loader = DataLoader(
    qwen_validation_conversation_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=qwen_collator,
)

with torch.inference_mode():
    for batch in validation_loader:
        batch = {
            key: value.to(DEVICE, non_blocking=True)
            for key, value in batch.items()
        }

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda"),
        ):
            outputs = qwen_conversation_model(**batch)

        if not torch.isfinite(outputs.loss):
            raise RuntimeError("Non-finite validation loss detected.")

        validation_loss_sum += outputs.loss.item()
        validation_batches += 1

        del outputs, batch

        if validation_batches % 100 == 0:
            print(
                f"Validated {validation_batches}/"
                f"{len(qwen_validation_conversation_dataset)} examples"
            )

validation_loss = validation_loss_sum / validation_batches
validation_perplexity = math.exp(validation_loss)

print("\n========== QWEN CONVERSATION-LEVEL TRIAL 3 ==========")
print("Training examples:", len(qwen_train_conversation_dataset))
print("Validation examples:", len(qwen_validation_conversation_dataset))
print("Optimizer steps completed:", global_step)
print("Validation loss:", round(validation_loss, 4))
print("Validation perplexity:", round(validation_perplexity, 4))
print("Final adapter:", QWEN_FINAL_ADAPTER_DIR)
print("=====================================================")


Validated 100/1000 examples
Validated 200/1000 examples
Validated 300/1000 examples
Validated 400/1000 examples
Validated 500/1000 examples
Validated 600/1000 examples
Validated 700/1000 examples
Validated 800/1000 examples
Validated 900/1000 examples
Validated 1000/1000 examples

========== QWEN CONVERSATION-LEVEL TRIAL 3 ==========
Training examples: 17878
Validation examples: 1000
Optimizer steps completed: 2236
Validation loss: 2.6743
Validation perplexity: 14.5022
Final adapter: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_Qwen/trial_3_conversation_level/final_adapter


Qwen + LoRA Trial 3 generation and evaluation

In [ ]:

import os
import json
import time
import gc
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bert_score


# Paths

QWEN_TRIAL3_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "PersonaChat_Qwen/trial_3_conversation_level"
)

QWEN_ADAPTER_DIR = os.path.join(
    QWEN_TRIAL3_DIR, "final_adapter"
)

QWEN_BASELINE_CSV = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "PersonaChat_Qwen/base_model_evaluation/"
    "qwen_base_validation_generations.csv"
)

QWEN_TRIAL3_GENERATIONS_CSV = os.path.join(
    QWEN_TRIAL3_DIR,
    "qwen_lora_trial3_validation_generations.csv"
)

QWEN_TRIAL3_METRICS_CSV = os.path.join(
    QWEN_TRIAL3_DIR,
    "qwen_lora_trial3_generation_metrics_per_example.csv"
)

QWEN_TRIAL3_SUMMARY_JSON = os.path.join(
    QWEN_TRIAL3_DIR,
    "qwen_lora_trial3_generation_metrics_summary.json"
)

assert os.path.isdir(QWEN_ADAPTER_DIR), (
    f"Trained adapter not found: {QWEN_ADAPTER_DIR}"
)
assert os.path.isfile(QWEN_BASELINE_CSV), (
    f"Baseline generations not found: {QWEN_BASELINE_CSV}"
)


# Reuse exactly the baseline prompts and references.

baseline_df = pd.read_csv(QWEN_BASELINE_CSV).fillna("")

required_columns = [
    "prompt", "reference_response", "generated_response"
]

assert all(column in baseline_df.columns for column in required_columns)
assert len(baseline_df) == 1000, (
    f"Expected 1000 validation examples, found {len(baseline_df)}"
)

prompts = baseline_df["prompt"].astype(str).tolist()
references = baseline_df["reference_response"].astype(str).tolist()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Evaluation device:", device)


# Generate Trial 3 responses, or reuse existing generations.

generation_time_minutes = 0.0

if os.path.isfile(QWEN_TRIAL3_GENERATIONS_CSV):
    print("Existing Trial 3 generations found. Reusing them.")

    trial3_df = pd.read_csv(
        QWEN_TRIAL3_GENERATIONS_CSV
    ).fillna("")

    assert len(trial3_df) == 1000
    assert all(
        column in trial3_df.columns
        for column in required_columns
    )

    assert trial3_df["prompt"].astype(str).tolist() == prompts
    assert (
        trial3_df["reference_response"].astype(str).tolist()
        == references
    )

    predictions = (
        trial3_df["generated_response"].astype(str).tolist()
    )

else:
    print("Loading trained Qwen LoRA Trial 3 adapter...")

    tokenizer = AutoTokenizer.from_pretrained(
        QWEN_ADAPTER_DIR
    )
    tokenizer.padding_side = "left"
    tokenizer.truncation_side = "left"

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        QWEN_MODEL_NAME
    )

    model = PeftModel.from_pretrained(
        base_model,
        QWEN_ADAPTER_DIR
    )

    model.config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = True
    model.to(device)
    model.eval()

    BATCH_SIZE = 8
    MAX_NEW_TOKENS = 40
    QWEN_EOS_TOKEN_ID = tokenizer.convert_tokens_to_ids(
        "<|im_end|>"
    )

    generation_start = time.time()
    predictions = []

    for start in range(0, len(prompts), BATCH_SIZE):
        batch_prompts = prompts[start:start + BATCH_SIZE]
        formatted_prompts = []

        for raw_prompt in batch_prompts:
            if "\n\nConversation:\n" not in raw_prompt:
                raise ValueError(
                    "Unexpected prompt format. Expected "
                    "'\\n\\nConversation:\\n' in each prompt."
                )

            persona_text, history = raw_prompt.split(
                "\n\nConversation:\n",
                1
            )

            messages = [
                {
                    "role": "system",
                    "content": (
                        "You are a helpful conversational partner. "
                        "Stay consistent with this persona:\n"
                        f"{persona_text}"
                    )
                },
                {
                    "role": "user",
                    "content": (
                        "Continue this conversation naturally. "
                        "Conversation history:\n"
                        f"{history}"
                    )
                }
            ]

            formatted_prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            formatted_prompts.append(formatted_prompt)

        inputs = tokenizer(
            formatted_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=280,
            add_special_tokens=False
        ).to(device)

        with torch.inference_mode():
            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                output_ids = model.generate(
                    **inputs,
                    do_sample=False,
                    max_new_tokens=MAX_NEW_TOKENS,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=QWEN_EOS_TOKEN_ID,
                    use_cache=True
                )

        # Exclude the padded input prompt from decoded responses.
        input_width = inputs["input_ids"].shape[1]
        generated_ids = output_ids[:, input_width:]

        batch_predictions = tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        predictions.extend(
            response.strip() for response in batch_predictions
        )

        completed = min(start + len(batch_prompts), len(prompts))

        if completed % 200 == 0 or completed == len(prompts):
            print(
                f"Generated {completed}/{len(prompts)} responses"
            )

    generation_time_minutes = (
        time.time() - generation_start
    ) / 60

    trial3_df = pd.DataFrame({
        "prompt": prompts,
        "reference_response": references,
        "generated_response": predictions
    })

    assert len(trial3_df) == 1000

    # Save separately; never modify baseline or Trial 1/2 files.
    trial3_df.to_csv(
        QWEN_TRIAL3_GENERATIONS_CSV,
        index=False
    )

    print("Generated responses saved:", QWEN_TRIAL3_GENERATIONS_CSV)

    del model, base_model, tokenizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# BLEU — corpus-level word overlap.

references_tokens = [
    text.lower().split() for text in references
]
predictions_tokens = [
    text.lower().split() for text in predictions
]

bleu = corpus_bleu(
    [[reference] for reference in references_tokens],
    predictions_tokens,
    smoothing_function=SmoothingFunction().method1
)


# ROUGE-L — per-example F1.

rouge = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True
)

rouge_l_scores = [
    rouge.score(reference, prediction)["rougeL"].fmeasure
    for reference, prediction in zip(references, predictions)
]

mean_rouge_l = sum(rouge_l_scores) / len(rouge_l_scores)


# Distinct-1 and Distinct-2.

all_unigrams = []
all_bigrams = []

for prediction in predictions_tokens:
    all_unigrams.extend(prediction)
    all_bigrams.extend(
        tuple(prediction[i:i + 2])
        for i in range(max(0, len(prediction) - 1))
    )

distinct_1 = (
    len(set(all_unigrams)) / len(all_unigrams)
    if all_unigrams else 0.0
)

distinct_2 = (
    len(set(all_bigrams)) / len(all_bigrams)
    if all_bigrams else 0.0
)


# Response lengths.

reply_lengths = [
    len(text.split()) for text in predictions
]

average_reply_length = sum(reply_lengths) / len(reply_lengths)
median_reply_length = float(
    pd.Series(reply_lengths).median()
)


# BERTScore — use the same encoder and settings as the baseline.

bert_start = time.time()

P, R, F1 = bert_score(
    cands=predictions,
    refs=references,
    model_type="distilbert-base-uncased",
    lang="en",
    batch_size=16,
    device=str(device),
    verbose=True,
    rescale_with_baseline=False
)

mean_bert_precision = P.mean().item()
mean_bert_recall = R.mean().item()
mean_bert_f1 = F1.mean().item()

bert_time_minutes = (time.time() - bert_start) / 60


# Save per-example metrics.

trial3_df["rougeL_f1"] = rouge_l_scores
trial3_df["bert_score_precision"] = P.tolist()
trial3_df["bert_score_recall"] = R.tolist()
trial3_df["bert_score_f1"] = F1.tolist()
trial3_df["generated_word_count"] = reply_lengths

trial3_df.to_csv(
    QWEN_TRIAL3_METRICS_CSV,
    index=False
)


# Save aggregate metrics.

metrics = {
    "model": QWEN_MODEL_NAME,
    "stage": "LoRA Trial 3",
    "number_of_validation_conversations": len(trial3_df),
    "training_examples": 17878,
    "validation_loss": 2.6743,
    "validation_perplexity": 14.5022,
    "corpus_bleu": bleu,
    "mean_rougeL_f1": mean_rouge_l,
    "mean_bert_score_precision": mean_bert_precision,
    "mean_bert_score_recall": mean_bert_recall,
    "mean_bert_score_f1": mean_bert_f1,
    "distinct_1": distinct_1,
    "distinct_2": distinct_2,
    "average_reply_length_words": average_reply_length,
    "median_reply_length_words": median_reply_length,
    "empty_reply_count": sum(
        not text.strip() for text in predictions
    ),
    "generation_time_minutes": generation_time_minutes,
    "bert_score_time_minutes": bert_time_minutes,
    "generation_method": "greedy decoding",
    "max_new_tokens": 40
}

with open(
    QWEN_TRIAL3_SUMMARY_JSON,
    "w",
    encoding="utf-8"
) as f:
    json.dump(metrics, f, indent=2)


# Display results.

print("\n===== QWEN LoRA TRIAL 3 GENERATION METRICS =====")

for name, value in metrics.items():
    if isinstance(value, float):
        print(f"{name}: {value:.4f}")
    else:
        print(f"{name}: {value}")

print("\nGeneration CSV:", QWEN_TRIAL3_GENERATIONS_CSV)
print("Per-example metrics CSV:", QWEN_TRIAL3_METRICS_CSV)
print("Summary JSON:", QWEN_TRIAL3_SUMMARY_JSON)
print("================================================")


Evaluation device: cuda
Loading trained Qwen LoRA Trial 3 adapter...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Generated 200/1000 responses
Generated 400/1000 responses
Generated 600/1000 responses
Generated 800/1000 responses
Generated 1000/1000 responses
Generated responses saved: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_Qwen/trial_3_conversation_level/qwen_lora_trial3_validation_generations.csv


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/111 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/63 [00:00<?, ?it/s]

done in 1.56 seconds, 642.36 sentences/sec

===== QWEN LoRA TRIAL 3 GENERATION METRICS =====
model: Qwen/Qwen2.5-0.5B-Instruct
stage: LoRA Trial 3
number_of_validation_conversations: 1000
training_examples: 17878
validation_loss: 2.6743
validation_perplexity: 14.5022
corpus_bleu: 0.0150
mean_rougeL_f1: 0.1208
mean_bert_score_precision: 0.7159
mean_bert_score_recall: 0.7028
mean_bert_score_f1: 0.7089
distinct_1: 0.1088
distinct_2: 0.3933
average_reply_length_words: 8.0870
median_reply_length_words: 7.0000
empty_reply_count: 0
generation_time_minutes: 2.0010
bert_score_time_minutes: 0.0582
generation_method: greedy decoding
max_new_tokens: 40

Generation CSV: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_Qwen/trial_3_conversation_level/qwen_lora_trial3_validation_generations.csv
Per-example metrics CSV: /content/drive/MyDrive/COMP6012_Assignment3/PersonaChat_Qwen/trial_3_conversation_level/qwen_lora_trial3_generation_metrics_per_example.csv
Summary JSON: /content/drive/MyDrive/

Verifying Qwen Baseline vs LoRA Trial 3 and Comparing Generation Lengths

In [ ]:

import os
import json
import pandas as pd

QWEN_BASELINE_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "PersonaChat_Qwen/base_model_evaluation"
)

QWEN_TRIAL3_DIR = (
    "/content/drive/MyDrive/COMP6012_Assignment3/"
    "PersonaChat_Qwen/trial_3_conversation_level"
)

baseline_csv = os.path.join(
    QWEN_BASELINE_DIR,
    "qwen_base_validation_generations.csv"
)

trial3_csv = os.path.join(
    QWEN_TRIAL3_DIR,
    "qwen_lora_trial3_validation_generations.csv"
)

baseline_metrics_json = os.path.join(
    QWEN_BASELINE_DIR,
    "qwen_base_generation_metrics_summary.json"
)

trial3_metrics_json = os.path.join(
    QWEN_TRIAL3_DIR,
    "qwen_lora_trial3_generation_metrics_summary.json"
)

assert os.path.isfile(baseline_csv), "Base generation CSV is missing."
assert os.path.isfile(trial3_csv), "Trial 3 generation CSV is missing."

base_df = pd.read_csv(baseline_csv).fillna("")
trial3_df = pd.read_csv(trial3_csv).fillna("")

required_columns = [
    "prompt", "reference_response", "generated_response"
]

assert all(c in base_df.columns for c in required_columns)
assert all(c in trial3_df.columns for c in required_columns)
assert len(base_df) == 1000
assert len(trial3_df) == 1000

# Verify the exact prompts and reference responses, including row order.
prompts_match = (
    base_df["prompt"].astype(str).tolist()
    ==
    trial3_df["prompt"].astype(str).tolist()
)

references_match = (
    base_df["reference_response"].astype(str).tolist()
    ==
    trial3_df["reference_response"].astype(str).tolist()
)

print("===== BASE vs QWEN LoRA TRIAL 3 VERIFICATION =====")
print("Base validation rows:", len(base_df))
print("Trial 3 validation rows:", len(trial3_df))
print("Prompts match exactly and in order:", prompts_match)
print("Reference responses match exactly and in order:", references_match)
print(
    "Base and Trial 3 generated responses identical:",
    base_df["generated_response"].astype(str).tolist()
    ==
    trial3_df["generated_response"].astype(str).tolist()
)

assert prompts_match, "Prompt mismatch detected."
assert references_match, "Reference-response mismatch detected."

# Report settings recorded in the Trial 3 metrics summary.
if os.path.isfile(trial3_metrics_json):
    with open(trial3_metrics_json, "r", encoding="utf-8") as f:
        trial3_metrics = json.load(f)

    print("\nTrial 3 recorded generation settings:")
    for key in [
        "generation_method",
        "max_new_tokens",
        "generation_time_minutes"
    ]:
        print(f"{key}: {trial3_metrics.get(key, 'Not recorded')}")

if os.path.isfile(baseline_metrics_json):
    with open(baseline_metrics_json, "r", encoding="utf-8") as f:
        base_metrics = json.load(f)

    print("\nBaseline recorded generation settings:")
    for key in [
        "generation_method",
        "max_new_tokens",
        "generation_time_minutes"
    ]:
        print(f"{key}: {base_metrics.get(key, 'Not recorded')}")

print("\n===== GENERATION LENGTH DIAGNOSTICS =====")
for name, df in [
    ("Base Qwen", base_df),
    ("Qwen LoRA Trial 3", trial3_df)
]:
    lengths = df["generated_response"].astype(str).str.split().str.len()
    print(
        f"{name}: mean={lengths.mean():.2f} words, "
        f"median={lengths.median():.0f}, "
        f"empty={int(lengths.eq(0).sum())}"
    )

print("\nPrompt and reference verification passed.")
print("No files were modified.")


===== BASE vs QWEN LoRA TRIAL 3 VERIFICATION =====
Base validation rows: 1000
Trial 3 validation rows: 1000
Prompts match exactly and in order: True
Reference responses match exactly and in order: True
Base and Trial 3 generated responses identical: False

Trial 3 recorded generation settings:
generation_method: greedy decoding
max_new_tokens: 40
generation_time_minutes: 2.0010351220766704

Baseline recorded generation settings:
generation_method: Not recorded
max_new_tokens: Not recorded
generation_time_minutes: Not recorded

===== GENERATION LENGTH DIAGNOSTICS =====
Base Qwen: mean=29.46 words, median=32, empty=0
Qwen LoRA Trial 3: mean=8.09 words, median=7, empty=0

Prompt and reference verification passed.
No files were modified.
